# 03 · Tumbabiro — Año sintético 2050 y exportación SAM

**Objetivo.** Consumir los artefactos congelados de los Notebooks 1 y 2 para construir el año sintético 2050.

Entradas obligatorias:

- `consol_tumbabiro_2026_total.csv`
- `physical_2050_base.csv`
- `PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras`
- `scaler_hist_v23.joblib`
- `scaler_fut_v23.joblib`
- `model_metadata_v23.json`

Este notebook **no reentrena el modelo**, **no vuelve a ajustar scalers** y **no redefine de forma independiente las 34/17 features**.

Se exportan dos recursos:

1. `synthetic_2050_raw_v23.csv`: salida neuronal reconstruida físicamente antes del postproceso climatológico.
2. `synthetic_2050_SAM_master.csv`: recurso final postprocesado para SAM.

Esta separación permite discutir transparentemente cuánto aporta la red y cuánto aporta el postprocesamiento.

In [1]:
# ============================================================
# 1. PI-LSTM V2.4 V1 RAW — PIPELINE SINTÉTICO 2050 PARA SAM
#    CONFIGURACIÓN GENERAL
# ============================================================
#
# OBJETIVO
# --------
#
# Construir un año sintético 2050 de irradiancia solar para
# Tumbabiro/Urcuquí, Ecuador, utilizando:
#
#   1. base física 2050 generada en el Notebook 1
#   2. modelo canónico PI-LSTM V2.4 V1 RAW del Notebook 2
#   3. climatología meteorológica histórica 1998–2024
#   4. inferencia secuencial sobre el calendario completo 2050
#   5. reconstrucción física:
#
#          kDNI = salida 1
#
#          qDHI = salida 2
#
#          kDHI = exp(qDHI) - 1
#
#          DNI = kDNI * Clearsky DNI
#
#          DHI = kDHI * Clearsky DHI
#
#          GHI = DNI*cos(z) + DHI
#
#
# IMPORTANTE
# ----------
#
# El modelo usado es:
#
#       PI-LSTM V2.4 V1 RAW
#
# NO se utiliza la calibración isotónica estudiada y
# rechazada en las Celdas 15–16 del Notebook 2.
#
#
# El año 2050 generado NO debe interpretarse como una
# proyección climática futura.
#
# Es un:
#
#   "synthetic 2050 solar resource year conditioned on the
#    historical NSRDB climatological regime and the solar
#    geometry of calendar year 2050."
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path

import json
import hashlib
import warnings
import random

import numpy as np
import pandas as pd

import tensorflow as tf
import joblib


# ============================================================
# 2. SEMILLAS Y REPRODUCIBILIDAD
# ============================================================

SEED_2050 = 42

random.seed(SEED_2050)
np.random.seed(SEED_2050)
tf.random.set_seed(SEED_2050)


# ============================================================
# 3. RUTAS DE ENTRADA
#
# Ajustar únicamente si los archivos se encuentran en otra
# ubicación de Colab.
# ============================================================

BASE_DIR_2050 = Path("/content")


# Histórico NSRDB 1998–2024
HISTORICAL_DATA_PATH_2050 = (BASE_DIR_2050/"consol_tumbabiro_2026_total.csv")

# Base física 2050 construida en Notebook 1
PHYSICAL_2050_PATH = (BASE_DIR_2050/"physical_2050_base.csv")

# Modelo canónico V2.4 V1 RAW
MODEL_V24V1_PATH = (BASE_DIR_2050/"PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras")

# Scalers canónicos
SCALER_HIST_V24V1_PATH = (BASE_DIR_2050/"scaler_hist_v24_v1.joblib")

SCALER_FUT_V24V1_PATH = (BASE_DIR_2050/"scaler_fut_v24_v1.joblib")

# Metadata del modelo
MODEL_METADATA_V24V1_PATH = (BASE_DIR_2050/"model_metadata_v24_v1.json")

# Selección canónica registrada en Notebook 2
MODEL_SELECTION_V24V1_PATH = (BASE_DIR_2050/"final_model_selection_v24_v1.json")


# ============================================================
# 4. CARPETA DE SALIDA
# ============================================================

OUTPUT_DIR_2050_V24V1 = Path("/content/model_v24_v1_2050_outputs")

OUTPUT_DIR_2050_V24V1.mkdir(parents=True,
                            exist_ok=True)


# ============================================================
# 5. CONFIGURACIÓN CANÓNICA V2.4 V1
# ============================================================

class Config2050V24V1:

    # --------------------------------------------------------
    # Ventanas temporales
    # --------------------------------------------------------

    L = 72
    H = 24
    stride = 2

    # --------------------------------------------------------
    # Resolución temporal
    # --------------------------------------------------------

    freq = "30min"
    dt_hours = 0.5

    # --------------------------------------------------------
    # Umbrales físicos
    # --------------------------------------------------------

    thr_cs = 20.0
    thr_day_cosz = 0.05


    # --------------------------------------------------------
    # Dominios de salida
    # --------------------------------------------------------

    kdni_max = 1.20
    kdhi_max = 8.00


    # --------------------------------------------------------
    # Transformación difusa V2.4
    # --------------------------------------------------------

    qdhi_max = float(
        np.log1p(
            kdhi_max
        )
    )


    # --------------------------------------------------------
    # Split histórico usado durante entrenamiento
    # --------------------------------------------------------

    train_end = "2019-12-31 23:30:00"
    val_end = "2022-12-31 23:30:00"
    test_end = "2024-12-31 23:30:00"

    # --------------------------------------------------------
    # Sitio
    # --------------------------------------------------------

    latitude = 0.45
    longitude = -78.18
    elevation_m = 2050.0
    timezone = "Etc/GMT+5"

cfg2050 = Config2050V24V1()


# ============================================================
# 6. COLUMNAS FÍSICAS CANÓNICAS
# ============================================================

COL_GHI = "GHI"
COL_DNI = "DNI"
COL_DHI = "DHI"
COL_CSGHI = "Clearsky GHI"
COL_CSDNI = "Clearsky DNI"
COL_CSDHI = "Clearsky DHI"
COL_ZENITH = "Solar Zenith Angle"

# ============================================================
# 7. FEATURES EXACTAS DEL MODELO V2.4 V1
# ============================================================

HIST_FEATURES_V24V1 = [
    "DNI_lag1",
    "DNI_roll3",

    "DHI_lag1",
    "DHI_roll3",

    "GHI_lag1",

    "dGHI",
    "dDNI",
    "dDHI",

    "kGHI",
    "kDNI",
    "kDHI",

    "kGHI_lag1",
    "kGHI_roll3",

    "kDNI_lag1",
    "kDNI_roll3",

    "qDHI_lag1",
    "qDHI_roll3",

    "diffuse_ratio_lag1",
    "diffuse_ratio_roll3",

    "beam_ratio_lag1",
    "beam_ratio_roll3",

    "Temperature",
    "Dew Point",
    "Relative Humidity",
    "Pressure",
    "Aerosol Optical Depth",
    "Precipitable Water",
    "Wind Speed",

    "sin_wdir",
    "cos_wdir",

    "cosz_plus",

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


FUTURE_FEATURES_SCALED_V24V1 = [

    "cosz_plus",

    "sin_hour",
    "cos_hour",

    "sin_doy",
    "cos_doy",

    "Temperature",
    "Dew Point",
    "Relative Humidity",
    "Pressure",
    "Aerosol Optical Depth",
    "Precipitable Water",
    "Wind Speed",

    "sin_wdir",
    "cos_wdir",
]


FUTURE_FEATURES_RAW_V24V1 = [

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


FUTURE_FEATURES_V24V1 = (

    FUTURE_FEATURES_SCALED_V24V1

    +

    FUTURE_FEATURES_RAW_V24V1
)


# ============================================================
# 8. AUDITORÍA DE DIMENSIONES
# ============================================================

assert (
    len(
        HIST_FEATURES_V24V1
    )
    ==
    34
)


assert (
    len(
        FUTURE_FEATURES_SCALED_V24V1
    )
    ==
    14
)


assert (
    len(
        FUTURE_FEATURES_RAW_V24V1
    )
    ==
    3
)


assert (
    len(
        FUTURE_FEATURES_V24V1
    )
    ==
    17
)


# ============================================================
# 9. FUNCIÓN SHA256
# ============================================================

def sha256_file_2050(
    path,
    chunk_size=1024 * 1024
):

    sha = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            sha.update(
                chunk
            )

    return sha.hexdigest()


# ============================================================
# 10. VERIFICACIÓN DE ARCHIVOS DE ENTRADA
# ============================================================

required_paths_2050 = {

    "Historical NSRDB":
        HISTORICAL_DATA_PATH_2050,

    "Physical 2050":
        PHYSICAL_2050_PATH,

    "PI-LSTM V2.4 V1 RAW":
        MODEL_V24V1_PATH,

    "Historical scaler":
        SCALER_HIST_V24V1_PATH,

    "Future scaler":
        SCALER_FUT_V24V1_PATH,

    "Model metadata":
        MODEL_METADATA_V24V1_PATH,

    "Model selection":
        MODEL_SELECTION_V24V1_PATH,
}


print(
    "============================================================"
)

print(
    " PI-LSTM V2.4 V1 RAW — PIPELINE SINTÉTICO 2050"
)

print(
    "============================================================"
)


print(
    "\n=== ARCHIVOS DE ENTRADA ==="
)


for name, path in required_paths_2050.items():

    exists = path.exists()

    print(
        f"{name}:"
    )

    print(
        " ",
        path
    )

    print(
        "  exists =",
        exists
    )

    assert exists, (
        f"No se encontró el archivo requerido:\n{path}"
    )


# ============================================================
# 11. AUDITORÍA DEL MODELO SELECCIONADO
# ============================================================

with open(

    MODEL_SELECTION_V24V1_PATH,

    "r",

    encoding="utf-8"

) as f:

    selection_info_2050 = json.load(
        f
    )


print(
    "\n=== SELECCIÓN CANÓNICA NOTEBOOK 2 ==="
)


print(
    "selected_model:",
    selection_info_2050.get(
        "selected_model"
    )
)


print(
    "selection_status:",
    selection_info_2050.get(
        "selection_status"
    )
)


print(
    "calibration_status:",
    selection_info_2050.get(
        "calibration_status"
    )
)


print(
    "canonical_postprocessing:",
    selection_info_2050.get(
        "canonical_postprocessing"
    )
)


assert (
    selection_info_2050.get(
        "selected_model"
    )
    ==
    "PI-LSTM V2.4 V1 RAW"
)


assert (
    selection_info_2050.get(
        "calibration_status"
    )
    ==
    "REJECTED"
)


assert (
    selection_info_2050.get(
        "canonical_postprocessing"
    )
    ==
    "NONE"
)


# ============================================================
# 12. VERIFICACIÓN SHA256 DEL MODELO
# ============================================================

model_sha256_current_2050 = (

    sha256_file_2050(
        MODEL_V24V1_PATH
    )
)


model_sha256_expected_2050 = (

    selection_info_2050.get(
        "selected_model_sha256"
    )
)


print(
    "\n=== SHA256 MODELO ==="
)


print(
    "Actual:"
)

print(
    model_sha256_current_2050
)


print(
    "\nRegistrado en Notebook 2:"
)

print(
    model_sha256_expected_2050
)


assert (
    model_sha256_current_2050
    ==
    model_sha256_expected_2050
), (
    "El modelo cargado no coincide con el modelo "
    "canónico seleccionado en Notebook 2."
)


# ============================================================
# 13. RESUMEN DE CONFIGURACIÓN
# ============================================================

print("\n=== CONFIGURACIÓN V2.4 V1 RAW ===")


print("L:", cfg2050.L)

print(
    "H:",
    cfg2050.H
)

print(
    "stride:",
    cfg2050.stride
)

print(
    "freq:",
    cfg2050.freq
)

print(
    "kDNI max:",
    cfg2050.kdni_max
)

print(
    "kDHI safety max:",
    cfg2050.kdhi_max
)

print(
    "qDHI max:",
    cfg2050.qdhi_max
)


print(
    "\nHistorical features:",
    len(
        HIST_FEATURES_V24V1
    )
)


print(
    "Future features:",
    len(
        FUTURE_FEATURES_V24V1
    )
)


print(
    "\nSitio:"
)

print(
    "Lat:",
    cfg2050.latitude
)

print(
    "Lon:",
    cfg2050.longitude
)

print(
    "Elevation:",
    cfg2050.elevation_m,
    "m"
)


# ============================================================
# 14. DECLARACIÓN METODOLÓGICA
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    " DECLARACIÓN METODOLÓGICA"
)

print(
    "============================================================"
)


print(
    """
El recurso 2050 generado en este notebook NO constituye una
proyección climática futura.

Se construirá como un año sintético condicionado por:

  - la climatología histórica NSRDB 1998–2024,
  - la geometría solar correspondiente al calendario 2050,
  - la dinámica aprendida por el modelo PI-LSTM V2.4 V1 RAW.

No se utilizará la calibración isotónica rechazada en el
Notebook 2.
"""
)


print(
    "✓ Celda 1 completada correctamente."
)

 PI-LSTM V2.4 V1 RAW — PIPELINE SINTÉTICO 2050

=== ARCHIVOS DE ENTRADA ===
Historical NSRDB:
  /content/consol_tumbabiro_2026_total.csv
  exists = True
Physical 2050:
  /content/physical_2050_base.csv
  exists = True
PI-LSTM V2.4 V1 RAW:
  /content/PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras
  exists = True
Historical scaler:
  /content/scaler_hist_v24_v1.joblib
  exists = True
Future scaler:
  /content/scaler_fut_v24_v1.joblib
  exists = True
Model metadata:
  /content/model_metadata_v24_v1.json
  exists = True
Model selection:
  /content/final_model_selection_v24_v1.json
  exists = True

=== SELECCIÓN CANÓNICA NOTEBOOK 2 ===
selected_model: PI-LSTM V2.4 V1 RAW
selection_status: CANONICAL
calibration_status: REJECTED
canonical_postprocessing: NONE

=== SHA256 MODELO ===
Actual:
5aef4783a3dc0b37b98711ebb489d20458f6c5efd2308ca579909e4cc8724a4e

Registrado en Notebook 2:
5aef4783a3dc0b37b98711ebb489d20458f6c5efd2308ca579909e4cc8724a4e

=== CONFIGURACIÓN V2.4 V1 RAW ===
L: 72
H: 24
st

In [2]:
# ============================================================
# 2. PI-LSTM V2.4 V1 RAW — CARGA Y AUDITORÍA DE INSUMOS
#    HISTÓRICO + BASE FÍSICA 2050 + SCALERS/METADATA
# ============================================================
#
# OBJETIVO
# --------
#
# Cargar y verificar:
#
#   1. histórico NSRDB 1998–2024
#   2. physical_2050_base.csv del Notebook 1
#   3. scaler histórico V2.4 V1
#   4. scaler futuro V2.4 V1
#   5. metadata del modelo
#
#
# En esta celda NO se realiza:
#
#   - climatología meteorológica 2050
#   - construcción de features
#   - inferencia PI-LSTM
#   - postprocesamiento
#
# ============================================================


# ============================================================
# 1. FUNCIÓN ROBUSTA PARA CONSTRUIR TIMESTAMP
# ============================================================

def build_timestamp_2050_audit(df):

    df = df.copy()

    print(
        "\nColumnas disponibles para timestamp:"
    )

    print(
        list(df.columns)
    )


    # --------------------------------------------------------
    # Mapa normalizado de nombres
    # --------------------------------------------------------

    normalized_map = {

        str(col).strip().lower():
            col

        for col in df.columns
    }


    # ========================================================
    # CASO 1:
    # timestamp / datetime ya existente
    #
    # Aquí NO usamos dayfirst=True como primera opción,
    # porque physical_2050_base.csv utiliza formato ISO:
    #
    #       YYYY-MM-DD HH:MM:SS
    #
    # ========================================================

    timestamp_candidates = [

        "timestamp",
        "datetime",
        "date_time",
        "date time",
    ]


    for candidate in timestamp_candidates:

        if candidate in normalized_map:

            original_col = normalized_map[
                candidate
            ]


            raw_ts = df[
                original_col
            ]


            print(
                "\nColumna temporal directa detectada:",
                original_col
            )


            print(
                "Primeros valores:"
            )

            print(
                raw_ts.head(
                    3
                ).to_string(
                    index=False
                )
            )


            # ------------------------------------------------
            # Intento 1:
            # parser flexible moderno para ISO/mixed
            # ------------------------------------------------

            try:

                ts = pd.to_datetime(

                    raw_ts,

                    format="mixed",

                    errors="coerce",

                    dayfirst=False
                )

            except TypeError:

                # Compatibilidad con pandas antiguos
                ts = pd.to_datetime(

                    raw_ts,

                    errors="coerce",

                    dayfirst=False
                )


            n_bad = int(
                ts.isna().sum()
            )


            # ------------------------------------------------
            # Intento 2:
            # fallback con dayfirst=True si hiciera falta
            # ------------------------------------------------

            if n_bad > 0:

                try:

                    ts_alt = pd.to_datetime(

                        raw_ts,

                        format="mixed",

                        errors="coerce",

                        dayfirst=True
                    )

                except TypeError:

                    ts_alt = pd.to_datetime(

                        raw_ts,

                        errors="coerce",

                        dayfirst=True
                    )


                # Usar fallback solo si mejora
                if (
                    ts_alt.isna().sum()
                    <
                    ts.isna().sum()
                ):

                    ts = ts_alt


            n_bad_final = int(
                ts.isna().sum()
            )


            print(
                "Timestamps inválidos tras parseo:",
                n_bad_final
            )


            if n_bad_final == 0:

                # --------------------------------------------
                # Si hubiera timezone, convertir a naive
                # conservando la hora local representada.
                # --------------------------------------------

                try:

                    if ts.dt.tz is not None:

                        ts = ts.dt.tz_localize(
                            None
                        )

                except AttributeError:

                    pass


                df[
                    "timestamp"
                ] = ts


                print(
                    "Timestamp construido correctamente "
                    "desde columna:",
                    original_col
                )


                return df


            else:

                print(
                    "No fue posible interpretar completamente "
                    "la columna temporal directa."
                )


    # ========================================================
    # CASO 2:
    # Date + Hour + Minute
    #
    # Formato real del consolidado histórico 1998–2024.
    #
    # Aquí SÍ utilizamos dayfirst=True.
    # ========================================================

    if all(

        c in normalized_map

        for c in [

            "date",
            "hour",
            "minute",

        ]

    ):


        date_col = normalized_map[
            "date"
        ]

        hour_col = normalized_map[
            "hour"
        ]

        minute_col = normalized_map[
            "minute"
        ]


        print(
            "\nFormato temporal detectado:"
        )

        print(
            f"  Date   -> {date_col}"
        )

        print(
            f"  Hour   -> {hour_col}"
        )

        print(
            f"  Minute -> {minute_col}"
        )


        # ----------------------------------------------------
        # Parsear Date
        # ----------------------------------------------------

        try:

            base_date = pd.to_datetime(

                df[
                    date_col
                ],

                format="mixed",

                errors="coerce",

                dayfirst=True
            )

        except TypeError:

            base_date = pd.to_datetime(

                df[
                    date_col
                ],

                errors="coerce",

                dayfirst=True
            )


        hour_values = pd.to_numeric(

            df[
                hour_col
            ],

            errors="coerce"
        )


        minute_values = pd.to_numeric(

            df[
                minute_col
            ],

            errors="coerce"
        )


        # ----------------------------------------------------
        # Auditoría
        # ----------------------------------------------------

        n_bad_date = int(
            base_date.isna().sum()
        )

        n_bad_hour = int(
            hour_values.isna().sum()
        )

        n_bad_minute = int(
            minute_values.isna().sum()
        )


        if n_bad_date > 0:

            raise ValueError(
                f"Hay {n_bad_date:,} valores Date inválidos."
            )


        if n_bad_hour > 0:

            raise ValueError(
                f"Hay {n_bad_hour:,} valores Hour inválidos."
            )


        if n_bad_minute > 0:

            raise ValueError(
                f"Hay {n_bad_minute:,} valores Minute inválidos."
            )


        assert (

            (
                hour_values
                >=
                0
            )

            &

            (
                hour_values
                <=
                23
            )

        ).all(), (
            "Existen valores Hour fuera de [0,23]."
        )


        assert (

            (
                minute_values
                >=
                0
            )

            &

            (
                minute_values
                <=
                59
            )

        ).all(), (
            "Existen valores Minute fuera de [0,59]."
        )


        # ----------------------------------------------------
        # Timestamp final
        # ----------------------------------------------------

        df[
            "timestamp"
        ] = (

            base_date.dt.normalize()

            +

            pd.to_timedelta(
                hour_values,
                unit="h"
            )

            +

            pd.to_timedelta(
                minute_values,
                unit="m"
            )
        )


        assert not df[
            "timestamp"
        ].isna().any()


        print(
            "\nTimestamp construido correctamente "
            "desde Date + Hour + Minute."
        )


        return df


    # ========================================================
    # CASO 3:
    # Year + Month + Day + Hour + Minute
    # ========================================================

    required_parts = [

        "year",
        "month",
        "day",
        "hour",
        "minute",
    ]


    if all(

        part in normalized_map

        for part in required_parts

    ):


        original_cols = {

            part:
                normalized_map[
                    part
                ]

            for part in required_parts
        }


        print(
            "\nColumnas temporales detectadas:"
        )


        for part, original_col in original_cols.items():

            print(
                f"  {part}: {original_col}"
            )


        df[
            "timestamp"
        ] = pd.to_datetime(
            {
                "year":
                    pd.to_numeric(
                        df[
                            original_cols[
                                "year"
                            ]
                        ],
                        errors="coerce"
                    ),

                "month":
                    pd.to_numeric(
                        df[
                            original_cols[
                                "month"
                            ]
                        ],
                        errors="coerce"
                    ),

                "day":
                    pd.to_numeric(
                        df[
                            original_cols[
                                "day"
                            ]
                        ],
                        errors="coerce"
                    ),

                "hour":
                    pd.to_numeric(
                        df[
                            original_cols[
                                "hour"
                            ]
                        ],
                        errors="coerce"
                    ),

                "minute":
                    pd.to_numeric(
                        df[
                            original_cols[
                                "minute"
                            ]
                        ],
                        errors="coerce"
                    ),
            },

            errors="coerce"
        )


        n_bad = int(

            df[
                "timestamp"
            ]
            .isna()
            .sum()
        )


        if n_bad > 0:

            raise ValueError(
                f"Se construyó timestamp, pero existen "
                f"{n_bad:,} valores inválidos."
            )


        print(
            "\nTimestamp construido correctamente "
            "desde Year/Month/Day/Hour/Minute."
        )


        return df


    # ========================================================
    # CASO 4:
    # estructura no reconocida
    # ========================================================

    temporal_candidates_found = [

        col

        for col in df.columns

        if any(

            token in str(
                col
            ).lower()

            for token in [

                "year",
                "month",
                "day",
                "hour",
                "minute",
                "date",
                "time",

            ]
        )
    ]


    raise ValueError(
        "\nNo fue posible construir timestamp.\n"
        f"Columnas temporalmente relacionadas encontradas: "
        f"{temporal_candidates_found}\n"
        f"Primeras columnas del archivo: "
        f"{list(df.columns[:20])}"
    )


# ============================================================
# 2. CARGAR HISTÓRICO NSRDB 1998–2024
# ============================================================

print(
    "============================================================"
)

print(
    " CARGA Y AUDITORÍA DE INSUMOS"
)

print(
    "============================================================"
)


print(
    "\n=== HISTÓRICO NSRDB 1998–2024 ==="
)


hist_raw_2050 = pd.read_csv(
    HISTORICAL_DATA_PATH_2050
)


print(
    "Shape original:",
    hist_raw_2050.shape
)


hist_2050 = build_timestamp_2050_audit(
    hist_raw_2050
)


assert not hist_2050[
    "timestamp"
].isna().any(), (
    "Hay timestamps históricos inválidos."
)


hist_2050 = (

    hist_2050
    .sort_values(
        "timestamp"
    )
    .reset_index(
        drop=True
    )
)


print(
    "Inicio:",
    hist_2050[
        "timestamp"
    ].min()
)


print(
    "Fin:",
    hist_2050[
        "timestamp"
    ].max()
)


print(
    "Filas:",
    f"{len(hist_2050):,}"
)


print(
    "Columnas:",
    len(
        hist_2050.columns
    )
)


# ============================================================
# 3. AUDITORÍA TEMPORAL HISTÓRICA
# ============================================================

expected_hist_rows_2050 = 473376


assert (
    len(hist_2050)
    ==
    expected_hist_rows_2050
), (
    f"Se esperaban {expected_hist_rows_2050:,} filas "
    f"históricas y se encontraron {len(hist_2050):,}."
)


hist_duplicates_2050 = int(

    hist_2050[
        "timestamp"
    ]
    .duplicated()
    .sum()
)


print(
    "Duplicados timestamp:",
    hist_duplicates_2050
)


assert (
    hist_duplicates_2050
    ==
    0
)


hist_dt_minutes_2050 = (

    hist_2050[
        "timestamp"
    ]
    .diff()
    .dropna()
    .dt.total_seconds()

    /

    60.0
)


hist_bad_steps_2050 = int(

    (
        hist_dt_minutes_2050
        !=
        30.0
    )
    .sum()
)


print(
    "Saltos distintos de 30 min:",
    hist_bad_steps_2050
)


assert (
    hist_bad_steps_2050
    ==
    0
), (
    "El histórico no presenta una secuencia continua "
    "de 30 minutos."
)


assert (
    hist_2050[
        "timestamp"
    ].dt.year.min()
    ==
    1998
)


assert (
    hist_2050[
        "timestamp"
    ].dt.year.max()
    ==
    2024
)


# ============================================================
# 4. COLUMNAS HISTÓRICAS ESENCIALES
# ============================================================

historical_required_cols_2050 = [

    COL_GHI,
    COL_DNI,
    COL_DHI,

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,

    COL_ZENITH,

    "Temperature",
    "Dew Point",
    "Relative Humidity",
    "Pressure",
    "Aerosol Optical Depth",
    "Precipitable Water",
    "Wind Speed",
    "Wind Direction",
]


missing_hist_cols_2050 = [

    c
    for c in historical_required_cols_2050
    if c not in hist_2050.columns
]


assert not missing_hist_cols_2050, (
    "Faltan columnas históricas requeridas: "
    f"{missing_hist_cols_2050}"
)


print(
    "Columnas físicas/meteo requeridas:",
    "OK"
)


# ============================================================
# 5. NaN HISTÓRICOS EN VARIABLES ESENCIALES
# ============================================================

hist_nan_summary_2050 = (

    hist_2050[
        historical_required_cols_2050
    ]
    .isna()
    .sum()
)


print(
    "\nNaN históricos en columnas esenciales:"
)


print(
    hist_nan_summary_2050.to_string()
)


assert (
    hist_nan_summary_2050.sum()
    ==
    0
), (
    "Existen NaN históricos en variables esenciales."
)


# ============================================================
# 6. CARGAR BASE FÍSICA 2050
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    " BASE FÍSICA 2050 — NOTEBOOK 1"
)

print(
    "============================================================"
)


physical_2050 = pd.read_csv(
    PHYSICAL_2050_PATH
)


print(
    "Shape original:",
    physical_2050.shape
)


physical_2050 = build_timestamp_2050_audit(
    physical_2050
)


physical_2050 = (

    physical_2050
    .sort_values(
        "timestamp"
    )
    .reset_index(
        drop=True
    )
)


print(
    "Inicio:",
    physical_2050[
        "timestamp"
    ].min()
)


print(
    "Fin:",
    physical_2050[
        "timestamp"
    ].max()
)


print(
    "Filas:",
    f"{len(physical_2050):,}"
)


# ============================================================
# 7. AUDITORÍA TEMPORAL 2050
# ============================================================

assert (
    len(
        physical_2050
    )
    ==
    17520
), (
    "El calendario 2050 debe contener exactamente "
    "17,520 registros de 30 min."
)


assert not physical_2050[
    "timestamp"
].duplicated().any()


assert (
    physical_2050[
        "timestamp"
    ].dt.year.nunique()
    ==
    1
)


assert (
    physical_2050[
        "timestamp"
    ].dt.year.iloc[0]
    ==
    2050
)


physical_dt_minutes_2050 = (

    physical_2050[
        "timestamp"
    ]
    .diff()
    .dropna()
    .dt.total_seconds()

    /

    60.0
)


physical_bad_steps_2050 = int(

    (
        physical_dt_minutes_2050
        !=
        30.0
    )
    .sum()
)


print(
    "Duplicados:",
    int(
        physical_2050[
            "timestamp"
        ]
        .duplicated()
        .sum()
    )
)


print(
    "Saltos distintos de 30 min:",
    physical_bad_steps_2050
)


assert (
    physical_bad_steps_2050
    ==
    0
)


# ============================================================
# 8. COLUMNAS FÍSICAS 2050 REQUERIDAS
# ============================================================

physical_required_cols_2050 = [

    COL_ZENITH,
    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


missing_physical_cols_2050 = [

    c
    for c in physical_required_cols_2050
    if c not in physical_2050.columns
]


assert not missing_physical_cols_2050, (
    "Faltan columnas en physical_2050_base.csv: "
    f"{missing_physical_cols_2050}"
)


# ============================================================
# 9. VALIDACIÓN DE VALORES FÍSICOS
# ============================================================

physical_nan_summary_2050 = (

    physical_2050[
        physical_required_cols_2050
    ]
    .isna()
    .sum()
)


print(
    "\nNaN base física 2050:"
)


print(
    physical_nan_summary_2050.to_string()
)


assert (
    physical_nan_summary_2050.sum()
    ==
    0
)


for col in [

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,

]:

    n_negative = int(

        (
            physical_2050[
                col
            ]
            <
            0.0
        )
        .sum()
    )


    print(
        f"{col} negativos:",
        n_negative
    )


    assert (
        n_negative
        ==
        0
    )


zenith_min_2050 = float(

    physical_2050[
        COL_ZENITH
    ].min()
)


zenith_max_2050 = float(

    physical_2050[
        COL_ZENITH
    ].max()
)


print(
    "\nSolar Zenith Angle:"
)

print(
    "min:",
    zenith_min_2050
)

print(
    "max:",
    zenith_max_2050
)


assert (
    zenith_min_2050
    >=
    0.0
)


assert (
    zenith_max_2050
    <=
    180.0
)


# ============================================================
# 10. RECONSTRUIR cos(z)
# ============================================================

physical_2050[
    "cosz"
] = np.cos(

    np.deg2rad(

        physical_2050[
            COL_ZENITH
        ]
        .to_numpy(
            dtype=np.float64
        )
    )
)


physical_2050[
    "cosz_plus"
] = np.maximum(

    physical_2050[
        "cosz"
    ],

    0.0
)


assert not physical_2050[
    [
        "cosz",
        "cosz_plus"
    ]
].isna().any().any()


print(
    "\ncosz_plus:"
)

print(
    "min:",
    float(
        physical_2050[
            "cosz_plus"
        ].min()
    )
)

print(
    "max:",
    float(
        physical_2050[
            "cosz_plus"
        ].max()
    )
)


# ============================================================
# 11. AUDITORÍA DE CIERRE CLEAR-SKY
#
# CS_GHI ≈ CS_DNI*cos(z) + CS_DHI
# ============================================================

clear_sky_closure_2050 = (

    physical_2050[
        COL_CSGHI
    ].to_numpy(
        dtype=np.float64
    )

    -

    (
        physical_2050[
            COL_CSDNI
        ].to_numpy(
            dtype=np.float64
        )

        *

        physical_2050[
            "cosz_plus"
        ].to_numpy(
            dtype=np.float64
        )

        +

        physical_2050[
            COL_CSDHI
        ].to_numpy(
            dtype=np.float64
        )
    )
)


clear_sky_closure_mae_2050 = float(

    np.mean(

        np.abs(
            clear_sky_closure_2050
        )
    )
)


clear_sky_closure_rmse_2050 = float(

    np.sqrt(

        np.mean(

            clear_sky_closure_2050 ** 2
        )
    )
)


clear_sky_closure_max_2050 = float(

    np.max(

        np.abs(
            clear_sky_closure_2050
        )
    )
)


print(
    "\n=== CIERRE CLEAR-SKY 2050 ==="
)


print(
    "MAE:",
    clear_sky_closure_mae_2050,
    "W/m²"
)


print(
    "RMSE:",
    clear_sky_closure_rmse_2050,
    "W/m²"
)


print(
    "Max abs:",
    clear_sky_closure_max_2050,
    "W/m²"
)


# ============================================================
# 12. SHA256 BASE FÍSICA
#
# Este es el hash canónico obtenido en Notebook 1 después
# de corregir la longitud a -78.18°.
# ============================================================

PHYSICAL_2050_EXPECTED_SHA256 = (
    "230cf4a767e52917b803f4c1a45add56"
    "c48a89aa88962fe3962188e265b9fd1f"
)


physical_2050_sha256 = sha256_file_2050(
    PHYSICAL_2050_PATH
)


print(
    "\n=== SHA256 BASE FÍSICA 2050 ==="
)


print(
    "Actual:"
)

print(
    physical_2050_sha256
)


print(
    "\nEsperado Notebook 1:"
)

print(
    PHYSICAL_2050_EXPECTED_SHA256
)


assert (
    physical_2050_sha256
    ==
    PHYSICAL_2050_EXPECTED_SHA256
), (
    "physical_2050_base.csv no coincide con el artefacto "
    "canónico generado en Notebook 1."
)


# ============================================================
# 13. CARGAR SCALERS V2.4 V1
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    " SCALERS CANÓNICOS V2.4 V1"
)

print(
    "============================================================"
)


scaler_hist_v24v1_2050 = joblib.load(

    SCALER_HIST_V24V1_PATH
)


scaler_fut_v24v1_2050 = joblib.load(

    SCALER_FUT_V24V1_PATH
)


print(
    "Historical scaler:",
    type(
        scaler_hist_v24v1_2050
    )
)


print(
    "Future scaler:",
    type(
        scaler_fut_v24v1_2050
    )
)


# ============================================================
# 14. VALIDAR DIMENSIONES DE SCALERS
# ============================================================

hist_scaler_n_features_2050 = getattr(

    scaler_hist_v24v1_2050,

    "n_features_in_",

    None
)


fut_scaler_n_features_2050 = getattr(

    scaler_fut_v24v1_2050,

    "n_features_in_",

    None
)


print(
    "\nHistorical scaler n_features:",
    hist_scaler_n_features_2050
)


print(
    "Future scaler n_features:",
    fut_scaler_n_features_2050
)


if hist_scaler_n_features_2050 is not None:

    assert (
        hist_scaler_n_features_2050
        ==
        34
    )


if fut_scaler_n_features_2050 is not None:

    assert (
        fut_scaler_n_features_2050
        ==
        14
    )


# ============================================================
# 15. CARGAR METADATA DEL MODELO
# ============================================================

with open(

    MODEL_METADATA_V24V1_PATH,

    "r",

    encoding="utf-8"

) as f:

    model_metadata_2050 = json.load(
        f
    )


print(
    "\n"
    "============================================================"
)

print(
    " METADATA V2.4 V1"
)

print(
    "============================================================"
)


print(
    "Keys:"
)


for key in sorted(
    model_metadata_2050.keys()
):

    print(
        " -",
        key
    )


# ============================================================
# 16. AUDITAR FEATURE LISTS SI CONSTAN EN METADATA
# ============================================================

metadata_hist_features_2050 = (

    model_metadata_2050.get(
        "hist_features"
    )

    or

    model_metadata_2050.get(
        "historical_features"
    )
)


metadata_future_features_2050 = (

    model_metadata_2050.get(
        "future_features"
    )
)


if metadata_hist_features_2050 is not None:

    print(
        "\nFeatures históricas metadata:",
        len(
            metadata_hist_features_2050
        )
    )


    assert (
        list(
            metadata_hist_features_2050
        )
        ==
        HIST_FEATURES_V24V1
    ), (
        "La lista de features históricas del metadata "
        "no coincide con la configuración canónica."
    )


else:

    print(
        "\nMetadata no contiene lista explícita "
        "de features históricas."
    )


if metadata_future_features_2050 is not None:

    print(
        "Features futuras metadata:",
        len(
            metadata_future_features_2050
        )
    )


    assert (
        list(
            metadata_future_features_2050
        )
        ==
        FUTURE_FEATURES_V24V1
    ), (
        "La lista de features futuras del metadata "
        "no coincide con la configuración canónica."
    )


else:

    print(
        "Metadata no contiene lista explícita "
        "de features futuras."
    )


# ============================================================
# 17. AUDITORÍA DE ESTADÍSTICAS CLEAR-SKY 2050
# ============================================================

print(
    "\n=== ESTADÍSTICAS BASE FÍSICA 2050 ==="
)


physical_summary_2050 = (

    physical_2050[
        [
            COL_ZENITH,
            "cosz_plus",

            COL_CSGHI,
            COL_CSDNI,
            COL_CSDHI,
        ]
    ]
    .describe()
    .T
)


print(
    physical_summary_2050[
        [
            "min",
            "mean",
            "50%",
            "max",
        ]
    ].to_string()
)


# ============================================================
# 18. RESUMEN DE AUDITORÍA
# ============================================================

audit_summary_cell2_2050 = {

    "historical_rows":
        int(
            len(
                hist_2050
            )
        ),

    "historical_start":
        str(
            hist_2050[
                "timestamp"
            ].min()
        ),

    "historical_end":
        str(
            hist_2050[
                "timestamp"
            ].max()
        ),

    "historical_duplicate_timestamps":
        hist_duplicates_2050,

    "historical_bad_30min_steps":
        hist_bad_steps_2050,

    "physical_2050_rows":
        int(
            len(
                physical_2050
            )
        ),

    "physical_2050_start":
        str(
            physical_2050[
                "timestamp"
            ].min()
        ),

    "physical_2050_end":
        str(
            physical_2050[
                "timestamp"
            ].max()
        ),

    "physical_2050_sha256":
        physical_2050_sha256,

    "clear_sky_closure_mae_Wm2":
        clear_sky_closure_mae_2050,

    "clear_sky_closure_rmse_Wm2":
        clear_sky_closure_rmse_2050,

    "clear_sky_closure_max_abs_Wm2":
        clear_sky_closure_max_2050,

    "hist_scaler_features":
        (
            int(
                hist_scaler_n_features_2050
            )

            if hist_scaler_n_features_2050
            is not None

            else None
        ),

    "future_scaler_features":
        (
            int(
                fut_scaler_n_features_2050
            )

            if fut_scaler_n_features_2050
            is not None

            else None
        ),
}


print(
    "\n"
    "============================================================"
)

print(
    " RESUMEN CELDA 2"
)

print(
    "============================================================"
)


print(
    json.dumps(
        audit_summary_cell2_2050,
        indent=2,
        ensure_ascii=False
    )
)


print(
    "\n✓ Histórico NSRDB 1998–2024 cargado y validado."
)

print(
    "✓ Continuidad temporal histórica verificada."
)

print(
    "✓ Base física 2050 cargada y validada."
)

print(
    "✓ SHA256 de physical_2050_base.csv verificado."
)

print(
    "✓ Geometría solar 2050 reconstruida."
)

print(
    "✓ Cierre clear-sky auditado."
)

print(
    "✓ Scalers V2.4 V1 cargados."
)

print(
    "✓ Dimensiones de scalers verificadas."
)

print(
    "✓ Metadata V2.4 V1 cargada."
)

print(
    "✓ Celda 2 completada correctamente."
)

 CARGA Y AUDITORÍA DE INSUMOS

=== HISTÓRICO NSRDB 1998–2024 ===
Shape original: (473376, 28)

Columnas disponibles para timestamp:
['Date', 'Hour', 'Minute', 'Temperature', 'Alpha', 'Aerosol Optical Depth', 'Asymmetry', 'Clearsky DHI', 'Clearsky DNI', 'Clearsky GHI', 'Cloud Fill Flag', 'Cloud Type', 'Dew Point', 'DHI', 'DNI', 'Fill Flag', 'GHI', 'Ozone', 'Relative Humidity', 'Solar Zenith Angle', 'SSA', 'Surface Albedo', 'Pressure', 'Precipitable Water', 'Wind Direction', 'Wind Speed', 'Global Horizontal UV Irradiance (280-400nm)', 'Global Horizontal UV Irradiance (295-385nm)']

Formato temporal detectado:
  Date   -> Date
  Hour   -> Hour
  Minute -> Minute

Timestamp construido correctamente desde Date + Hour + Minute.
Inicio: 1998-01-01 00:00:00
Fin: 2024-12-31 23:30:00
Filas: 473,376
Columnas: 29
Duplicados timestamp: 0
Saltos distintos de 30 min: 0
Columnas físicas/meteo requeridas: OK

NaN históricos en columnas esenciales:
GHI                      0
DNI                      0
D

In [3]:
# ============================================================
# 3. CLIMATOLOGÍA METEOROLÓGICA 2050
#    Histórico 1998–2024 -> calendario físico 2050
# ============================================================

METEO_COLS_2050 = [
    "Temperature", "Dew Point", "Relative Humidity", "Pressure",
    "Aerosol Optical Depth", "Precipitable Water",
    "Wind Speed", "Wind Direction"
]

# ------------------------------------------------------------
# 1. Variables temporales históricas
# ------------------------------------------------------------

hist_clim_2050 = hist_2050.copy()
hist_clim_2050["month"] = hist_clim_2050["timestamp"].dt.month
hist_clim_2050["day"] = hist_clim_2050["timestamp"].dt.day
hist_clim_2050["hour"] = hist_clim_2050["timestamp"].dt.hour
hist_clim_2050["minute"] = hist_clim_2050["timestamp"].dt.minute

# Día del año climatológico sin depender del año real.
# 2050 no es bisiesto, por lo que eliminamos 29-Feb del histórico.
hist_clim_2050 = hist_clim_2050[
    ~((hist_clim_2050["month"] == 2) & (hist_clim_2050["day"] == 29))
].copy()

hist_clim_2050["doy_clim"] = pd.to_datetime(
    dict(
        year=np.full(len(hist_clim_2050), 2001),
        month=hist_clim_2050["month"],
        day=hist_clim_2050["day"]
    )
).dt.dayofyear

hist_clim_2050["slot"] = hist_clim_2050["hour"] * 2 + hist_clim_2050["minute"] // 30

assert hist_clim_2050["slot"].between(0, 47).all()

# ------------------------------------------------------------
# 2. Wind Direction -> componentes circulares
# ------------------------------------------------------------

wd_rad = np.deg2rad(hist_clim_2050["Wind Direction"].to_numpy(dtype=float))
hist_clim_2050["sin_wdir"] = np.sin(wd_rad)
hist_clim_2050["cos_wdir"] = np.cos(wd_rad)

CLIM_COLS_2050 = [
    "Temperature", "Dew Point", "Relative Humidity", "Pressure",
    "Aerosol Optical Depth", "Precipitable Water", "Wind Speed",
    "sin_wdir", "cos_wdir"
]

# ------------------------------------------------------------
# 3. Climatología mediana por día del año y slot de 30 min
# ------------------------------------------------------------

climatology_2050 = (
    hist_clim_2050
    .groupby(["doy_clim", "slot"], as_index=False)[CLIM_COLS_2050]
    .median()
)

expected_clim_rows = 365 * 48

print("=== CLIMATOLOGÍA HISTÓRICA ===")
print("Filas esperadas:", expected_clim_rows)
print("Filas obtenidas :", len(climatology_2050))

assert len(climatology_2050) == expected_clim_rows
assert not climatology_2050[CLIM_COLS_2050].isna().any().any()

# ------------------------------------------------------------
# 4. Claves temporales del calendario 2050
# ------------------------------------------------------------

df_2050_base = physical_2050.copy()
df_2050_base["doy_clim"] = df_2050_base["timestamp"].dt.dayofyear
df_2050_base["slot"] = (
    df_2050_base["timestamp"].dt.hour * 2
    + df_2050_base["timestamp"].dt.minute // 30
)

# ------------------------------------------------------------
# 5. Merge climatología -> 2050
# ------------------------------------------------------------

df_2050_base = df_2050_base.merge(
    climatology_2050,
    on=["doy_clim", "slot"],
    how="left",
    validate="many_to_one"
)

assert len(df_2050_base) == 17520
assert not df_2050_base[CLIM_COLS_2050].isna().any().any()

# ------------------------------------------------------------
# 6. Reconstruir Wind Direction climatológica
# ------------------------------------------------------------

df_2050_base["Wind Direction"] = (
    np.rad2deg(
        np.arctan2(df_2050_base["sin_wdir"], df_2050_base["cos_wdir"])
    ) % 360.0
)

# ------------------------------------------------------------
# 7. Features temporales futuras
# ------------------------------------------------------------

hour_decimal = (
    df_2050_base["timestamp"].dt.hour
    + df_2050_base["timestamp"].dt.minute / 60.0
)

doy = df_2050_base["timestamp"].dt.dayofyear.astype(float)

df_2050_base["sin_hour"] = np.sin(2*np.pi*hour_decimal/24.0)
df_2050_base["cos_hour"] = np.cos(2*np.pi*hour_decimal/24.0)
df_2050_base["sin_doy"] = np.sin(2*np.pi*doy/365.25)
df_2050_base["cos_doy"] = np.cos(2*np.pi*doy/365.25)

# ------------------------------------------------------------
# 8. Auditoría final
# ------------------------------------------------------------

required_future_2050 = FUTURE_FEATURES_SCALED_V24V1 + FUTURE_FEATURES_RAW_V24V1

missing_future_2050 = [
    c for c in required_future_2050 if c not in df_2050_base.columns
]

assert not missing_future_2050, f"Faltan features futuras: {missing_future_2050}"
assert not df_2050_base[required_future_2050].isna().any().any()

print("\n=== BASE 2050 CON METEOROLOGÍA ===")
print("Shape:", df_2050_base.shape)
print("Inicio:", df_2050_base["timestamp"].min())
print("Fin   :", df_2050_base["timestamp"].max())

print("\nRangos meteorológicos 2050:")
print(
    df_2050_base[
        [
            "Temperature", "Dew Point", "Relative Humidity", "Pressure",
            "Aerosol Optical Depth", "Precipitable Water", "Wind Speed"
        ]
    ].agg(["min", "mean", "max"]).T
)

print("\nFeatures futuras:", len(required_future_2050))
print("NaN futuras:", int(df_2050_base[required_future_2050].isna().sum().sum()))
print("✓ Celda 3 completada correctamente.")

=== CLIMATOLOGÍA HISTÓRICA ===
Filas esperadas: 17520
Filas obtenidas : 17520

=== BASE 2050 CON METEOROLOGÍA ===
Shape: (17520, 23)
Inicio: 2050-01-01 00:00:00
Fin   : 2050-12-31 23:30:00

Rangos meteorológicos 2050:
                           min        mean      max
Temperature             11.300   17.257494   24.300
Dew Point                7.500   11.508037   14.100
Relative Humidity       40.140   70.803648   85.530
Pressure               801.000  803.217751  805.000
Aerosol Optical Depth    0.027    0.100802    0.306
Precipitable Water       2.200    2.870953    3.400
Wind Speed               0.400    1.599880    5.400

Features futuras: 17
NaN futuras: 0
✓ Celda 3 completada correctamente.


In [4]:
# ============================================================
# 4. CARGA Y AUDITORÍA DEL MODELO CANÓNICO V2.4 V1 RAW
# ============================================================
#
# OBJETIVO
# --------
# Antes de construir las features históricas y realizar
# inferencia 2050, verificamos el contrato exacto heredado
# del Notebook 2:
#
#   - 34 features históricas
#   - 17 features futuras:
#         14 escaladas + 3 clear-sky sin escalar
#   - entrada histórica: (L=72, 34)
#   - entrada futura:    (H=24, 17)
#   - salida:            (H=24, 2)
#
# Las dos salidas son:
#
#       0 -> kDNI
#       1 -> qDHI = log(1 + kDHI)
#
# No se utiliza ningún calibrador isotónico.
# ============================================================

# ------------------------------------------------------------
# 1. Recuperar listas exactas del metadata
# ------------------------------------------------------------

hist_cols_meta_2050 = model_metadata_2050.get("hist_cols")
future_cols_meta_2050 = model_metadata_2050.get("future_cols")

assert hist_cols_meta_2050 is not None, "Metadata no contiene 'hist_cols'."
assert future_cols_meta_2050 is not None, "Metadata no contiene 'future_cols'."

hist_cols_meta_2050 = list(hist_cols_meta_2050)
future_cols_meta_2050 = list(future_cols_meta_2050)

print("=== CONTRATO DE FEATURES DEL METADATA ===")
print("Históricas:", len(hist_cols_meta_2050))
print("Futuras   :", len(future_cols_meta_2050))

assert len(hist_cols_meta_2050) == 34
assert len(future_cols_meta_2050) == 17


# ------------------------------------------------------------
# 2. Comparar metadata con la configuración declarada
# ------------------------------------------------------------

hist_match_2050 = hist_cols_meta_2050 == HIST_FEATURES_V24V1
future_match_2050 = future_cols_meta_2050 == FUTURE_FEATURES_V24V1

print("\nCoincidencia lista histórica:", hist_match_2050)
print("Coincidencia lista futura   :", future_match_2050)

if not hist_match_2050:
    print("\nDiferencias HIST:")
    for i, (meta, cfg_name) in enumerate(
        zip(hist_cols_meta_2050, HIST_FEATURES_V24V1)
    ):
        if meta != cfg_name:
            print(f"  [{i:02d}] metadata={meta!r} | configuración={cfg_name!r}")

if not future_match_2050:
    print("\nDiferencias FUTURE:")
    for i, (meta, cfg_name) in enumerate(
        zip(future_cols_meta_2050, FUTURE_FEATURES_V24V1)
    ):
        if meta != cfg_name:
            print(f"  [{i:02d}] metadata={meta!r} | configuración={cfg_name!r}")

assert hist_match_2050, "Las features históricas no coinciden con el modelo entrenado."
assert future_match_2050, "Las features futuras no coinciden con el modelo entrenado."


# ------------------------------------------------------------
# 3. Mostrar orden exacto de features
#
# El orden es crítico porque los scalers fueron ajustados
# respetando precisamente estas posiciones.
# ------------------------------------------------------------

print("\n=== FEATURES HISTÓRICAS EXACTAS ===")
for i, col in enumerate(hist_cols_meta_2050, start=1):
    print(f"{i:02d}. {col}")

print("\n=== FEATURES FUTURAS EXACTAS ===")
for i, col in enumerate(future_cols_meta_2050, start=1):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 4. Auditar parámetros centrales del metadata
# ------------------------------------------------------------

metadata_contract_2050 = {
    "L": model_metadata_2050.get("L"),
    "H": model_metadata_2050.get("H"),
    "stride": model_metadata_2050.get("stride"),
    "n_hist_features": model_metadata_2050.get("n_hist_features"),
    "n_future_features": model_metadata_2050.get("n_future_features"),
    "target_direct": model_metadata_2050.get("target_direct"),
    "target_diffuse": model_metadata_2050.get("target_diffuse"),
    "kdni_max": model_metadata_2050.get("kdni_max"),
    "kdhi_max": model_metadata_2050.get("kdhi_max"),
    "qdhi_max": model_metadata_2050.get("qdhi_max"),
}

print("\n=== PARÁMETROS DEL MODELO ===")
for k, v in metadata_contract_2050.items():
    print(f"{k:18s}: {v}")

assert int(metadata_contract_2050["L"]) == cfg2050.L
assert int(metadata_contract_2050["H"]) == cfg2050.H
assert int(metadata_contract_2050["stride"]) == cfg2050.stride
assert int(metadata_contract_2050["n_hist_features"]) == 34
assert int(metadata_contract_2050["n_future_features"]) == 17

assert np.isclose(float(metadata_contract_2050["kdni_max"]), cfg2050.kdni_max)
assert np.isclose(float(metadata_contract_2050["kdhi_max"]), cfg2050.kdhi_max)
assert np.isclose(float(metadata_contract_2050["qdhi_max"]), cfg2050.qdhi_max)


# ------------------------------------------------------------
# 5. Definir capa custom necesaria para cargar el backbone
#
# V2.4 transforma qDHI -> kDHI mediante expm1 en componentes
# físicos. El backbone guardado puede contener la capa custom
# usada durante la construcción del modelo.
# ------------------------------------------------------------

@tf.keras.utils.register_keras_serializable(package="PI_LSTM")
class Expm1Layer(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.math.expm1(inputs)

    def get_config(self):
        return super().get_config()


# ------------------------------------------------------------
# 6. Cargar modelo canónico
#
# compile=False porque en Notebook 3 solo necesitamos
# inferencia; no se reentrena ni se evalúa la loss.
# ------------------------------------------------------------

model_v24v1_2050 = tf.keras.models.load_model(
    MODEL_V24V1_PATH,
    custom_objects={"Expm1Layer": Expm1Layer},
    compile=False
)

print("\n=== MODELO CARGADO ===")
print("Nombre     :", model_v24v1_2050.name)
print("Parámetros :", f"{model_v24v1_2050.count_params():,}")
print("Entradas   :", model_v24v1_2050.input_shape)
print("Salida     :", model_v24v1_2050.output_shape)

assert model_v24v1_2050.count_params() == 110146
assert len(model_v24v1_2050.inputs) == 2


# ------------------------------------------------------------
# 7. Validar shapes exactos
# ------------------------------------------------------------

input_shapes_2050 = model_v24v1_2050.input_shape

assert tuple(input_shapes_2050[0][1:]) == (cfg2050.L, 34)
assert tuple(input_shapes_2050[1][1:]) == (cfg2050.H, 17)
assert tuple(model_v24v1_2050.output_shape[1:]) == (cfg2050.H, 2)

print("\nShapes validados:")
print(f"  X_hist : (batch, {cfg2050.L}, 34)")
print(f"  X_fut  : (batch, {cfg2050.H}, 17)")
print(f"  Y_pred : (batch, {cfg2050.H}, 2)")


# ------------------------------------------------------------
# 8. Smoke test de inferencia
#
# No busca interpretar físicamente el resultado: únicamente
# verifica que el modelo ejecuta inferencia y respeta los
# dominios matemáticos de las dos salidas.
# ------------------------------------------------------------

dummy_hist_2050 = np.zeros((1, cfg2050.L, 34), dtype=np.float32)
dummy_fut_2050 = np.zeros((1, cfg2050.H, 17), dtype=np.float32)

dummy_pred_2050 = model_v24v1_2050(
    [dummy_hist_2050, dummy_fut_2050],
    training=False
).numpy()

assert dummy_pred_2050.shape == (1, cfg2050.H, 2)
assert np.isfinite(dummy_pred_2050).all()

dummy_kdni_2050 = dummy_pred_2050[..., 0]
dummy_qdhi_2050 = dummy_pred_2050[..., 1]
dummy_kdhi_2050 = np.expm1(dummy_qdhi_2050)

print("\n=== SMOKE TEST ===")
print("Shape salida:", dummy_pred_2050.shape)
print(
    "kDNI :",
    float(dummy_kdni_2050.min()),
    "→",
    float(dummy_kdni_2050.max())
)
print(
    "qDHI :",
    float(dummy_qdhi_2050.min()),
    "→",
    float(dummy_qdhi_2050.max())
)
print(
    "kDHI reconstruido:",
    float(dummy_kdhi_2050.min()),
    "→",
    float(dummy_kdhi_2050.max())
)

assert dummy_kdni_2050.min() >= -1e-6
assert dummy_kdni_2050.max() <= cfg2050.kdni_max + 1e-6
assert dummy_qdhi_2050.min() >= -1e-6
assert dummy_qdhi_2050.max() <= cfg2050.qdhi_max + 1e-6


# ------------------------------------------------------------
# 9. Verificar una vez más que no existe calibración en flujo
# ------------------------------------------------------------

assert selection_info_2050["calibration_status"] == "REJECTED"
assert selection_info_2050["canonical_postprocessing"] == "NONE"

print("\n=== ESTADO CANÓNICO ===")
print("Modelo          : PI-LSTM V2.4 V1 RAW")
print("Post-calibración: NONE")
print("Isotonic        : REJECTED")
print("✓ Celda 4 completada correctamente.")

=== CONTRATO DE FEATURES DEL METADATA ===
Históricas: 34
Futuras   : 17

Coincidencia lista histórica: True
Coincidencia lista futura   : True

=== FEATURES HISTÓRICAS EXACTAS ===
01. DNI_lag1
02. DNI_roll3
03. DHI_lag1
04. DHI_roll3
05. GHI_lag1
06. dGHI
07. dDNI
08. dDHI
09. kGHI
10. kDNI
11. kDHI
12. kGHI_lag1
13. kGHI_roll3
14. kDNI_lag1
15. kDNI_roll3
16. qDHI_lag1
17. qDHI_roll3
18. diffuse_ratio_lag1
19. diffuse_ratio_roll3
20. beam_ratio_lag1
21. beam_ratio_roll3
22. Temperature
23. Dew Point
24. Relative Humidity
25. Pressure
26. Aerosol Optical Depth
27. Precipitable Water
28. Wind Speed
29. sin_wdir
30. cos_wdir
31. cosz_plus
32. Clearsky GHI
33. Clearsky DNI
34. Clearsky DHI

=== FEATURES FUTURAS EXACTAS ===
01. cosz_plus
02. sin_hour
03. cos_hour
04. sin_doy
05. cos_doy
06. Temperature
07. Dew Point
08. Relative Humidity
09. Pressure
10. Aerosol Optical Depth
11. Precipitable Water
12. Wind Speed
13. sin_wdir
14. cos_wdir
15. Clearsky GHI
16. Clearsky DNI
17. Clearsky DHI


In [5]:
# ============================================================
# 5. FEATURES HISTÓRICAS V2.4 V1 Y SEMILLA PARA 2050
# ============================================================

# ------------------------------------------------------------
# 1. Base histórica, geometría y dirección del viento
# ------------------------------------------------------------

hist_feat_2050 = hist_2050.copy()

theta = np.deg2rad(hist_feat_2050[COL_ZENITH].to_numpy(dtype=np.float64))
hist_feat_2050["cosz"] = np.cos(theta)
hist_feat_2050["cosz_plus"] = np.maximum(hist_feat_2050["cosz"], 0.0)

wd = np.deg2rad(hist_feat_2050["Wind Direction"].to_numpy(dtype=np.float64))
hist_feat_2050["sin_wdir"] = np.sin(wd)
hist_feat_2050["cos_wdir"] = np.cos(wd)


# ------------------------------------------------------------
# 2. Índices normalizados y representación difusa
# ------------------------------------------------------------

eps = 1e-6

ghi = hist_feat_2050[COL_GHI].to_numpy(dtype=np.float64)
dni = hist_feat_2050[COL_DNI].to_numpy(dtype=np.float64)
dhi = hist_feat_2050[COL_DHI].to_numpy(dtype=np.float64)

csghi = hist_feat_2050[COL_CSGHI].to_numpy(dtype=np.float64)
csdni = hist_feat_2050[COL_CSDNI].to_numpy(dtype=np.float64)
csdhi = hist_feat_2050[COL_CSDHI].to_numpy(dtype=np.float64)

hist_feat_2050["kGHI"] = np.clip(np.where(csghi > cfg2050.thr_cs, ghi / (csghi + eps), 0.0), 0.0, 2.0)
hist_feat_2050["kDNI"] = np.clip(np.where(csdni > cfg2050.thr_cs, dni / (csdni + eps), 0.0), 0.0, cfg2050.kdni_max)
hist_feat_2050["kDHI"] = np.clip(np.where(csdhi > cfg2050.thr_cs, dhi / (csdhi + eps), 0.0), 0.0, cfg2050.kdhi_max)
hist_feat_2050["qDHI"] = np.log1p(hist_feat_2050["kDHI"])


# ------------------------------------------------------------
# 3. Razones de partición radiativa
# ------------------------------------------------------------

day_ghi = ghi > cfg2050.thr_cs

hist_feat_2050["diffuse_ratio"] = np.clip(
    np.where(day_ghi, dhi / (ghi + eps), 0.0), 0.0, 1.2
)

hist_feat_2050["beam_ratio"] = np.clip(
    np.where(
        day_ghi,
        dni * hist_feat_2050["cosz_plus"].to_numpy(dtype=np.float64) / (ghi + eps),
        0.0
    ),
    0.0, 1.5
)


# ------------------------------------------------------------
# 4. Diferencias, retardos y medias móviles causales
# ------------------------------------------------------------

hist_feat_2050["dGHI"] = hist_feat_2050[COL_GHI].diff()
hist_feat_2050["dDNI"] = hist_feat_2050[COL_DNI].diff()
hist_feat_2050["dDHI"] = hist_feat_2050[COL_DHI].diff()

lag_map = {
    "GHI_lag1": COL_GHI,
    "DNI_lag1": COL_DNI,
    "DHI_lag1": COL_DHI,
    "kGHI_lag1": "kGHI",
    "kDNI_lag1": "kDNI",
    "qDHI_lag1": "qDHI",
    "diffuse_ratio_lag1": "diffuse_ratio",
    "beam_ratio_lag1": "beam_ratio",
}

roll_map = {
    "DNI_roll3": COL_DNI,
    "DHI_roll3": COL_DHI,
    "kGHI_roll3": "kGHI",
    "kDNI_roll3": "kDNI",
    "qDHI_roll3": "qDHI",
    "diffuse_ratio_roll3": "diffuse_ratio",
    "beam_ratio_roll3": "beam_ratio",
}

for dst, src in lag_map.items():
    hist_feat_2050[dst] = hist_feat_2050[src].shift(1)

for dst, src in roll_map.items():
    hist_feat_2050[dst] = hist_feat_2050[src].shift(1).rolling(3, min_periods=1).mean()


# ------------------------------------------------------------
# 5. Auditoría de las 34 features históricas
# ------------------------------------------------------------

missing_hist_features_2050 = [c for c in hist_cols_meta_2050 if c not in hist_feat_2050.columns]
assert not missing_hist_features_2050, f"Faltan features históricas: {missing_hist_features_2050}"

print("=== FEATURES HISTÓRICAS RECONSTRUIDAS ===")
print("Total requeridas:", len(hist_cols_meta_2050))
print("Faltantes:", missing_hist_features_2050)


# ------------------------------------------------------------
# 6. Zona histórica utilizable
# ------------------------------------------------------------

hist_valid_2050 = hist_feat_2050.dropna(subset=hist_cols_meta_2050).copy()

print("\nFilas históricas originales:", len(hist_feat_2050))
print("Filas utilizables          :", len(hist_valid_2050))
print("Filas eliminadas por lags  :", len(hist_feat_2050) - len(hist_valid_2050))

assert len(hist_valid_2050) > cfg2050.L
assert not hist_valid_2050[hist_cols_meta_2050].isna().any().any()


# ------------------------------------------------------------
# 7. Auditoría de rangos principales
# ------------------------------------------------------------

print("\n=== RANGOS DE VARIABLES CLAVE ===")
for col in ["kGHI", "kDNI", "kDHI", "qDHI", "diffuse_ratio", "beam_ratio"]:
    s = hist_valid_2050[col]
    print(f"{col:16s} min={s.min():.6f} mean={s.mean():.6f} max={s.max():.6f}")

assert hist_valid_2050["kDNI"].between(0.0, cfg2050.kdni_max).all()
assert hist_valid_2050["kDHI"].between(0.0, cfg2050.kdhi_max).all()
assert hist_valid_2050["qDHI"].between(0.0, cfg2050.qdhi_max + 1e-9).all()
assert hist_valid_2050["diffuse_ratio"].between(0.0, 1.2).all()
assert hist_valid_2050["beam_ratio"].between(0.0, 1.5).all()


# ------------------------------------------------------------
# 8. Escalado histórico canónico
# ------------------------------------------------------------

X_hist_df_2050 = hist_valid_2050[hist_cols_meta_2050].astype(np.float32)
X_hist_scaled_all_2050 = scaler_hist_v24v1_2050.transform(X_hist_df_2050).astype(np.float32)

assert X_hist_scaled_all_2050.shape == (len(hist_valid_2050), 34)
assert np.isfinite(X_hist_scaled_all_2050).all()


# ------------------------------------------------------------
# 9. Semilla histórica final L=72
# ------------------------------------------------------------

seed_hist_raw_2050 = hist_valid_2050.iloc[-cfg2050.L:].copy()
seed_hist_scaled_2050 = X_hist_scaled_all_2050[-cfg2050.L:].copy()

assert seed_hist_raw_2050.shape[0] == cfg2050.L
assert seed_hist_scaled_2050.shape == (cfg2050.L, 34)

print("\n=== SEMILLA HISTÓRICA ===")
print("Filas :", len(seed_hist_raw_2050))
print("Inicio:", seed_hist_raw_2050["timestamp"].iloc[0])
print("Fin   :", seed_hist_raw_2050["timestamp"].iloc[-1])
print("Shape escalado:", seed_hist_scaled_2050.shape)


# ------------------------------------------------------------
# 10. Continuidad temporal de la semilla
# ------------------------------------------------------------

seed_dt_2050 = seed_hist_raw_2050["timestamp"].diff().dropna().dt.total_seconds() / 60.0

assert (seed_dt_2050 == 30.0).all()
assert seed_hist_raw_2050["timestamp"].iloc[-1] == pd.Timestamp("2024-12-31 23:30:00")

print("Continuidad 30 min: OK")


# ------------------------------------------------------------
# 11. Persistencia de la semilla auditada
# ------------------------------------------------------------

SEED_RAW_PATH_2050 = OUTPUT_DIR_2050_V24V1 / "historical_seed_L72_v24_v1_raw.csv"
SEED_SCALED_PATH_2050 = OUTPUT_DIR_2050_V24V1 / "historical_seed_L72_v24_v1_scaled.npy"

seed_hist_raw_2050[["timestamp"] + hist_cols_meta_2050].to_csv(SEED_RAW_PATH_2050, index=False)
np.save(SEED_SCALED_PATH_2050, seed_hist_scaled_2050)


# ------------------------------------------------------------
# 12. Resumen
# ------------------------------------------------------------

print("\n=== ARCHIVOS DE SEMILLA ===")
print(SEED_RAW_PATH_2050.name)
print(SEED_SCALED_PATH_2050.name)

print("\n✓ 34 features históricas reconstruidas con contrato V2.4 V1.")
print("✓ Ratios de partición con umbral GHI > 20 W/m².")
print("✓ Rolling3 causal: shift(1) + ventana retrospectiva de 3 registros.")
print("✓ Scaler histórico canónico aplicado.")
print("✓ Semilla final L=72 extraída.")
print("✓ Semilla termina exactamente en 2024-12-31 23:30.")
print("✓ Celda 5 completada correctamente.")

=== FEATURES HISTÓRICAS RECONSTRUIDAS ===
Total requeridas: 34
Faltantes: []

Filas históricas originales: 473376
Filas utilizables          : 473375
Filas eliminadas por lags  : 1

=== RANGOS DE VARIABLES CLAVE ===
kGHI             min=0.000000 mean=0.365479 max=1.000000
kDNI             min=0.000000 mean=0.249161 max=1.000000
kDHI             min=0.000000 mean=0.963338 max=7.918033
qDHI             min=0.000000 mean=0.488541 max=2.188075
diffuse_ratio    min=0.000000 mean=0.267814 max=1.011494
beam_ratio       min=0.000000 mean=0.210457 max=0.963574

=== SEMILLA HISTÓRICA ===
Filas : 72
Inicio: 2024-12-30 12:00:00
Fin   : 2024-12-31 23:30:00
Shape escalado: (72, 34)
Continuidad 30 min: OK

=== ARCHIVOS DE SEMILLA ===
historical_seed_L72_v24_v1_raw.csv
historical_seed_L72_v24_v1_scaled.npy

✓ 34 features históricas reconstruidas con contrato V2.4 V1.
✓ Ratios de partición con umbral GHI > 20 W/m².
✓ Rolling3 causal: shift(1) + ventana retrospectiva de 3 registros.
✓ Scaler histórico c

## Meteorología exógena 2050

Para mantener el escenario como un **año sintético condicionado al régimen climatológico histórico**, las variables meteorológicas exógenas de 2050 se construyen a partir de la **mediana histórica 1998–2024 para cada día climatológico del año y cada intervalo de 30 minutos** (`doy_clim`, `slot`).

De este modo se conserva el ciclo estacional y diurno característico del sitio, sin introducir una tendencia climática futura explícita.

Por tanto, el año 2050 generado **no constituye una proyección climática** ni representa un escenario CMIP6/SSP. Debe interpretarse como un **año sintético 2050 condicionado por la climatología histórica de NSRDB y por la geometría solar correspondiente al calendario 2050**.

In [6]:
# ============================================================
# 6. FEATURES FUTURAS V2.4 V1 PARA INFERENCIA 2050
# ============================================================
#
# OBJETIVO
# --------
# Construir la matriz futura completa del calendario 2050 con
# el contrato exacto utilizado durante entrenamiento:
#
#   X_future = [14 features escaladas + 3 clear-sky RAW]
#
# Dimensión esperada:
#
#       (17520, 17)
#
# Las primeras 14 variables pasan por scaler_fut.
# Clearsky GHI/DNI/DHI se incorporan SIN escalar.
# ============================================================


# ------------------------------------------------------------
# 1. Confirmar orden exacto del contrato futuro
# ------------------------------------------------------------

future_scaled_cols_2050 = FUTURE_FEATURES_SCALED_V24V1.copy()
future_raw_cols_2050 = FUTURE_FEATURES_RAW_V24V1.copy()

assert len(future_scaled_cols_2050) == 14
assert len(future_raw_cols_2050) == 3
assert future_scaled_cols_2050 + future_raw_cols_2050 == future_cols_meta_2050

print("=== CONTRATO FUTURO ===")
print("Escaladas :", len(future_scaled_cols_2050))
print("RAW       :", len(future_raw_cols_2050))
print("Total     :", len(future_cols_meta_2050))


# ------------------------------------------------------------
# 2. Auditoría de disponibilidad
# ------------------------------------------------------------

missing_future_cols_2050 = [
    c for c in future_cols_meta_2050
    if c not in df_2050_base.columns
]

assert not missing_future_cols_2050, (
    f"Faltan variables futuras en df_2050_base: "
    f"{missing_future_cols_2050}"
)

assert not df_2050_base[
    future_cols_meta_2050
].isna().any().any()


# ------------------------------------------------------------
# 3. Escalar únicamente las primeras 14 variables
#
# Conservamos DataFrame + nombres para respetar exactamente
# el contrato del StandardScaler y evitar warnings sklearn.
# ------------------------------------------------------------

X_future_scaled_df_2050 = (
    df_2050_base[future_scaled_cols_2050]
    .astype(np.float32)
)

X_future_scaled_part_2050 = (
    scaler_fut_v24v1_2050
    .transform(X_future_scaled_df_2050)
    .astype(np.float32)
)

assert X_future_scaled_part_2050.shape == (17520, 14)
assert np.isfinite(X_future_scaled_part_2050).all()


# ------------------------------------------------------------
# 4. Clear-sky RAW
# ------------------------------------------------------------

X_future_raw_part_2050 = (
    df_2050_base[future_raw_cols_2050]
    .to_numpy(dtype=np.float32)
)

assert X_future_raw_part_2050.shape == (17520, 3)
assert np.isfinite(X_future_raw_part_2050).all()
assert (X_future_raw_part_2050 >= 0.0).all()


# ------------------------------------------------------------
# 5. Concatenar en el orden exacto del modelo
# ------------------------------------------------------------

X_future_all_2050 = np.concatenate(
    [
        X_future_scaled_part_2050,
        X_future_raw_part_2050
    ],
    axis=1
).astype(np.float32)

assert X_future_all_2050.shape == (17520, 17)
assert np.isfinite(X_future_all_2050).all()


# ------------------------------------------------------------
# 6. Verificar explícitamente que las columnas RAW finales
#    sean exactamente CS_GHI, CS_DNI y CS_DHI
# ------------------------------------------------------------

np.testing.assert_allclose(
    X_future_all_2050[:, 14:17],
    df_2050_base[
        [COL_CSGHI, COL_CSDNI, COL_CSDHI]
    ].to_numpy(dtype=np.float32),
    rtol=0.0,
    atol=0.0
)

print("\nClear-sky RAW preservado exactamente: OK")


# ------------------------------------------------------------
# 7. Auditoría de distribución de features escaladas
#
# Al ser climatología 2050, no esperamos necesariamente
# media 0/desv. 1 exactas. Solo verificamos que los valores
# sean finitos y que no aparezcan extrapolaciones extremas.
# ------------------------------------------------------------

scaled_audit_2050 = pd.DataFrame({
    "feature": future_scaled_cols_2050,
    "min_scaled": X_future_scaled_part_2050.min(axis=0),
    "mean_scaled": X_future_scaled_part_2050.mean(axis=0),
    "max_scaled": X_future_scaled_part_2050.max(axis=0)
})

print("\n=== RANGOS ESCALADOS 2050 ===")
print(scaled_audit_2050.to_string(index=False))


# ------------------------------------------------------------
# 8. Auditoría clear-sky RAW
# ------------------------------------------------------------

raw_audit_2050 = pd.DataFrame({
    "feature": future_raw_cols_2050,
    "min": X_future_raw_part_2050.min(axis=0),
    "mean": X_future_raw_part_2050.mean(axis=0),
    "max": X_future_raw_part_2050.max(axis=0)
})

print("\n=== CLEAR-SKY RAW 2050 ===")
print(raw_audit_2050.to_string(index=False))


# ------------------------------------------------------------
# 9. Divisibilidad por horizonte H
#
# 17520 / 24 = 730 bloques exactos.
#
# Esto permite recorrer TODO el año mediante bloques
# consecutivos de H=24 sin dejar residuo temporal.
# ------------------------------------------------------------

n_steps_2050 = len(df_2050_base)
n_blocks_2050 = n_steps_2050 // cfg2050.H
remainder_2050 = n_steps_2050 % cfg2050.H

print("\n=== BLOQUES DE INFERENCIA ===")
print("Registros 2050 :", n_steps_2050)
print("Horizonte H    :", cfg2050.H)
print("Bloques        :", n_blocks_2050)
print("Residuo        :", remainder_2050)

assert n_blocks_2050 == 730
assert remainder_2050 == 0


# ------------------------------------------------------------
# 10. Primer bloque futuro H=24
#
# Se deja preparado para el smoke test de inferencia real
# que haremos antes del rolling completo.
# ------------------------------------------------------------

first_future_block_2050 = X_future_all_2050[:cfg2050.H].copy()

assert first_future_block_2050.shape == (24, 17)

print("\n=== PRIMER BLOQUE 2050 ===")
print("Inicio:", df_2050_base["timestamp"].iloc[0])
print("Fin   :", df_2050_base["timestamp"].iloc[cfg2050.H - 1])
print("Shape :", first_future_block_2050.shape)


# ------------------------------------------------------------
# 11. Guardar matriz futura completa
# ------------------------------------------------------------

FUTURE_MATRIX_PATH_2050 = (
    OUTPUT_DIR_2050_V24V1 /
    "future_features_2050_v24_v1.npy"
)

np.save(
    FUTURE_MATRIX_PATH_2050,
    X_future_all_2050
)

print("\nArchivo:")
print(FUTURE_MATRIX_PATH_2050.name)


# ------------------------------------------------------------
# 12. Resumen
# ------------------------------------------------------------

print("\n✓ 14 features futuras escaladas correctamente.")
print("✓ 3 componentes clear-sky conservadas RAW.")
print("✓ Matriz futura final: (17520, 17).")
print("✓ Año 2050 divisible exactamente en 730 bloques H=24.")
print("✓ Primer bloque futuro preparado.")
print("✓ Celda 6 completada correctamente.")

=== CONTRATO FUTURO ===
Escaladas : 14
RAW       : 3
Total     : 17

Clear-sky RAW preservado exactamente: OK

=== RANGOS ESCALADOS 2050 ===
              feature  min_scaled   mean_scaled  max_scaled
            cosz_plus   -0.825888 -6.702816e-04    1.875621
             sin_hour   -1.414214  5.443346e-11    1.414214
             cos_hour   -1.414214 -4.626845e-09    1.414214
              sin_doy   -1.414174 -1.143190e-05    1.414144
              cos_doy   -1.414140 -8.805924e-04    1.414333
          Temperature   -1.630144  7.501849e-03    1.943404
            Dew Point   -2.345973  9.073205e-02    1.666529
    Relative Humidity   -2.083905  3.800629e-02    1.057063
             Pressure   -1.536272 -4.531858e-02    1.152855
Aerosol Optical Depth   -0.885084 -1.837863e-01    1.766098
   Precipitable Water   -1.561267  9.184827e-02    1.395330
           Wind Speed   -1.032853 -8.732881e-02    2.907223
             sin_wdir   -2.098077  2.560945e-01    0.732187
             cos_wd

In [7]:
# ============================================================
# 7. SMOKE TEST REAL — PRIMER BLOQUE SINTÉTICO 2050
# ============================================================
#
# OBJETIVO
# --------
# Ejecutar una inferencia REAL con:
#
#   X_hist = últimas L=72 observaciones de 2024
#   X_fut  = primeras H=24 posiciones del calendario 2050
#
# y reconstruir:
#
#       kDNI
#       qDHI
#       kDHI = exp(qDHI)-1
#       DNI  = kDNI * CS_DNI
#       DHI  = kDHI * CS_DHI
#       GHI  = DNI*cos(z) + DHI
#
# Esta celda NO actualiza todavía el estado autoregresivo.
# Solo verifica el contrato completo sobre un bloque real.
# ============================================================


# ------------------------------------------------------------
# 1. Preparar entradas con dimensión batch
# ------------------------------------------------------------

X_hist_block1_2050 = seed_hist_scaled_2050[np.newaxis, ...].astype(np.float32)
X_fut_block1_2050 = first_future_block_2050[np.newaxis, ...].astype(np.float32)

assert X_hist_block1_2050.shape == (1, cfg2050.L, 34)
assert X_fut_block1_2050.shape == (1, cfg2050.H, 17)

print("=== ENTRADAS PRIMER BLOQUE ===")
print("X_hist:", X_hist_block1_2050.shape)
print("X_fut :", X_fut_block1_2050.shape)


# ------------------------------------------------------------
# 2. Inferencia
# ------------------------------------------------------------

pred_block1_2050 = model_v24v1_2050(
    [X_hist_block1_2050, X_fut_block1_2050],
    training=False
).numpy()[0]

assert pred_block1_2050.shape == (cfg2050.H, 2)
assert np.isfinite(pred_block1_2050).all()


# ------------------------------------------------------------
# 3. Recuperar outputs del modelo
# ------------------------------------------------------------

kDNI_pred_block1_2050 = np.clip(
    pred_block1_2050[:, 0],
    0.0,
    cfg2050.kdni_max
)

qDHI_pred_block1_2050 = np.clip(
    pred_block1_2050[:, 1],
    0.0,
    cfg2050.qdhi_max
)

kDHI_pred_block1_2050 = np.clip(
    np.expm1(qDHI_pred_block1_2050),
    0.0,
    cfg2050.kdhi_max
)


# ------------------------------------------------------------
# 4. Variables físicas del primer bloque
# ------------------------------------------------------------

block1_base_2050 = df_2050_base.iloc[:cfg2050.H].copy()

csdni_block1 = block1_base_2050[COL_CSDNI].to_numpy(dtype=np.float64)
csdhi_block1 = block1_base_2050[COL_CSDHI].to_numpy(dtype=np.float64)
cosz_block1 = block1_base_2050["cosz_plus"].to_numpy(dtype=np.float64)

DNI_pred_block1_2050 = kDNI_pred_block1_2050 * csdni_block1
DHI_pred_block1_2050 = kDHI_pred_block1_2050 * csdhi_block1
GHI_pred_block1_2050 = DNI_pred_block1_2050 * cosz_block1 + DHI_pred_block1_2050


# ------------------------------------------------------------
# 5. Condición nocturna
#
# Durante la noche imponemos exactamente cero para evitar
# leakage radiativo numérico.
# ------------------------------------------------------------

night_mask_block1 = cosz_block1 <= 0.0

DNI_pred_block1_2050[night_mask_block1] = 0.0
DHI_pred_block1_2050[night_mask_block1] = 0.0
GHI_pred_block1_2050[night_mask_block1] = 0.0


# ------------------------------------------------------------
# 6. Construir tabla diagnóstica
# ------------------------------------------------------------

first_block_result_2050 = pd.DataFrame({
    "timestamp": block1_base_2050["timestamp"].to_numpy(),
    "cosz_plus": cosz_block1,
    "CS_GHI": block1_base_2050[COL_CSGHI].to_numpy(dtype=float),
    "CS_DNI": csdni_block1,
    "CS_DHI": csdhi_block1,
    "kDNI_pred": kDNI_pred_block1_2050,
    "qDHI_pred": qDHI_pred_block1_2050,
    "kDHI_pred": kDHI_pred_block1_2050,
    "DNI_pred": DNI_pred_block1_2050,
    "DHI_pred": DHI_pred_block1_2050,
    "GHI_pred": GHI_pred_block1_2050
})


# ------------------------------------------------------------
# 7. Auditoría física
# ------------------------------------------------------------

closure_block1_2050 = (
    first_block_result_2050["GHI_pred"].to_numpy()
    -
    (
        first_block_result_2050["DNI_pred"].to_numpy()
        * first_block_result_2050["cosz_plus"].to_numpy()
        +
        first_block_result_2050["DHI_pred"].to_numpy()
    )
)

closure_mae_block1_2050 = float(np.mean(np.abs(closure_block1_2050)))
closure_max_block1_2050 = float(np.max(np.abs(closure_block1_2050)))

night_leakage_block1_2050 = int(
    (
        first_block_result_2050.loc[
            first_block_result_2050["cosz_plus"] <= 0.0,
            ["GHI_pred", "DNI_pred", "DHI_pred"]
        ].to_numpy() != 0.0
    ).sum()
)

assert np.isfinite(
    first_block_result_2050[
        ["kDNI_pred", "qDHI_pred", "kDHI_pred",
         "DNI_pred", "DHI_pred", "GHI_pred"]
    ].to_numpy()
).all()

assert (first_block_result_2050[["DNI_pred", "DHI_pred", "GHI_pred"]] >= 0).all().all()
assert closure_max_block1_2050 < 1e-6
assert night_leakage_block1_2050 == 0


# ------------------------------------------------------------
# 8. Resumen numérico
# ------------------------------------------------------------

print("\n=== RANGOS DE SALIDA DEL MODELO ===")
print(
    f"kDNI : {kDNI_pred_block1_2050.min():.6f} "
    f"→ {kDNI_pred_block1_2050.max():.6f}"
)
print(
    f"qDHI : {qDHI_pred_block1_2050.min():.6f} "
    f"→ {qDHI_pred_block1_2050.max():.6f}"
)
print(
    f"kDHI : {kDHI_pred_block1_2050.min():.6f} "
    f"→ {kDHI_pred_block1_2050.max():.6f}"
)

print("\n=== IRRADIANCIA RECONSTRUIDA ===")
for col in ["GHI_pred", "DNI_pred", "DHI_pred"]:
    s = first_block_result_2050[col]
    print(f"{col:10s} min={s.min():.3f} mean={s.mean():.3f} max={s.max():.3f}")

print("\n=== AUDITORÍA FÍSICA ===")
print("Closure MAE     :", closure_mae_block1_2050)
print("Closure max abs :", closure_max_block1_2050)
print("Night leakage   :", night_leakage_block1_2050)


# ------------------------------------------------------------
# 9. Mostrar las 24 posiciones
# ------------------------------------------------------------

print("\n=== PRIMER BLOQUE SINTÉTICO 2050 ===")
print(
    first_block_result_2050[
        [
            "timestamp", "cosz_plus",
            "kDNI_pred", "kDHI_pred",
            "DNI_pred", "DHI_pred", "GHI_pred"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 10. Guardar diagnóstico
# ------------------------------------------------------------

FIRST_BLOCK_PATH_2050 = (
    OUTPUT_DIR_2050_V24V1 /
    "first_block_2050_smoke_test_v24_v1.csv"
)

first_block_result_2050.to_csv(
    FIRST_BLOCK_PATH_2050,
    index=False
)

print("\nArchivo:")
print(FIRST_BLOCK_PATH_2050.name)

print("\n✓ Inferencia real del primer bloque completada.")
print("✓ qDHI reconstruido correctamente a kDHI.")
print("✓ DNI/DHI/GHI físicamente reconstruidos.")
print("✓ Cierre radiativo exacto verificado.")
print("✓ Leakage nocturno = 0.")
print("✓ Celda 7 completada correctamente.")

=== ENTRADAS PRIMER BLOQUE ===
X_hist: (1, 72, 34)
X_fut : (1, 24, 17)

=== RANGOS DE SALIDA DEL MODELO ===
kDNI : 0.000001 → 0.211490
qDHI : 0.000008 → 1.456845
kDHI : 0.000008 → 3.292395

=== IRRADIANCIA RECONSTRUIDA ===
GHI_pred   min=0.000 mean=143.149 max=588.731
DNI_pred   min=0.000 mean=41.186 max=199.647
DHI_pred   min=0.000 mean=111.837 max=414.842

=== AUDITORÍA FÍSICA ===
Closure MAE     : 0.0
Closure max abs : 0.0
Night leakage   : 0

=== PRIMER BLOQUE SINTÉTICO 2050 ===
          timestamp  cosz_plus  kDNI_pred  kDHI_pred   DNI_pred   DHI_pred   GHI_pred
2050-01-01 00:00:00   0.000000   0.000019   0.000271   0.000000   0.000000   0.000000
2050-01-01 00:30:00   0.000000   0.000019   0.000129   0.000000   0.000000   0.000000
2050-01-01 01:00:00   0.000000   0.000020   0.000087   0.000000   0.000000   0.000000
2050-01-01 01:30:00   0.000000   0.000016   0.000057   0.000000   0.000000   0.000000
2050-01-01 02:00:00   0.000000   0.000010   0.000035   0.000000   0.000000   0.000

In [8]:
# ============================================================
# 8. ACTUALIZACIÓN AUTOREGRESIVA DEL ESTADO HISTÓRICO
#    PRUEBA CON EL PRIMER BLOQUE H=24
# ============================================================
#
# OBJETIVO
# --------
# Convertir las predicciones del primer bloque 2050 en nuevas
# observaciones sintéticas compatibles con las 34 features
# históricas utilizadas por PI-LSTM V2.4 V1.
#
# Después de añadir las 24 predicciones:
#
#   historial previo: 72 registros
#   bloque generado  : 24 registros
#
# se conservan los últimos L=72 registros para alimentar el
# segundo bloque de inferencia.
#
# IMPORTANTE
# ----------
# Las features derivadas se recalculan secuencialmente a partir
# de irradiancias ya generadas. No se reutilizan valores futuros
# observados ni se introduce información externa de irradiancia.
# ============================================================


# ------------------------------------------------------------
# 1. Columnas mínimas necesarias para reconstruir el estado
# ------------------------------------------------------------

STATE_BASE_COLS_2050 = [
    "timestamp",
    COL_GHI, COL_DNI, COL_DHI,
    COL_CSGHI, COL_CSDNI, COL_CSDHI,
    COL_ZENITH, "cosz", "cosz_plus",
    "Temperature", "Dew Point", "Relative Humidity", "Pressure",
    "Aerosol Optical Depth", "Precipitable Water",
    "Wind Speed", "Wind Direction",
    "sin_wdir", "cos_wdir"
]


# ------------------------------------------------------------
# 2. Preparar la semilla RAW histórica
#
# seed_hist_raw_2050 ya contiene las 34 features, pero para
# reconstruir lags/rolling necesitamos también las variables
# físicas base correspondientes a esos mismos timestamps.
# ------------------------------------------------------------

seed_base_2050 = hist_feat_2050[
    hist_feat_2050["timestamp"].isin(seed_hist_raw_2050["timestamp"])
][STATE_BASE_COLS_2050].copy()

seed_base_2050 = (
    seed_base_2050
    .sort_values("timestamp")
    .reset_index(drop=True)
)

assert len(seed_base_2050) == cfg2050.L
assert seed_base_2050["timestamp"].iloc[-1] == pd.Timestamp("2024-12-31 23:30:00")


# ------------------------------------------------------------
# 3. Construir las 24 filas sintéticas del primer bloque
#
# Se incorporan:
#
#   - irradiancias predichas RAW
#   - clear-sky y geometría física del Notebook 1
#   - meteorología climatológica 2050
# ------------------------------------------------------------

synthetic_block1_state_2050 = pd.DataFrame({
    "timestamp": block1_base_2050["timestamp"].to_numpy(),

    COL_GHI: GHI_pred_block1_2050,
    COL_DNI: DNI_pred_block1_2050,
    COL_DHI: DHI_pred_block1_2050,

    COL_CSGHI: block1_base_2050[COL_CSGHI].to_numpy(dtype=float),
    COL_CSDNI: block1_base_2050[COL_CSDNI].to_numpy(dtype=float),
    COL_CSDHI: block1_base_2050[COL_CSDHI].to_numpy(dtype=float),

    COL_ZENITH: block1_base_2050[COL_ZENITH].to_numpy(dtype=float),
    "cosz": block1_base_2050["cosz"].to_numpy(dtype=float),
    "cosz_plus": block1_base_2050["cosz_plus"].to_numpy(dtype=float),

    "Temperature": block1_base_2050["Temperature"].to_numpy(dtype=float),
    "Dew Point": block1_base_2050["Dew Point"].to_numpy(dtype=float),
    "Relative Humidity": block1_base_2050["Relative Humidity"].to_numpy(dtype=float),
    "Pressure": block1_base_2050["Pressure"].to_numpy(dtype=float),
    "Aerosol Optical Depth": block1_base_2050["Aerosol Optical Depth"].to_numpy(dtype=float),
    "Precipitable Water": block1_base_2050["Precipitable Water"].to_numpy(dtype=float),
    "Wind Speed": block1_base_2050["Wind Speed"].to_numpy(dtype=float),
    "Wind Direction": block1_base_2050["Wind Direction"].to_numpy(dtype=float),
    "sin_wdir": block1_base_2050["sin_wdir"].to_numpy(dtype=float),
    "cos_wdir": block1_base_2050["cos_wdir"].to_numpy(dtype=float),
})

assert len(synthetic_block1_state_2050) == cfg2050.H
assert not synthetic_block1_state_2050.isna().any().any()


# ------------------------------------------------------------
# 4. Función canónica para reconstruir features históricas
#
# Esta función será reutilizada en el rolling anual.
#
# Recibe una secuencia física ordenada temporalmente y genera
# exactamente las 34 variables históricas del modelo.
# ------------------------------------------------------------

def build_historical_features_v24v1(df):
    out = df.copy().sort_values("timestamp").reset_index(drop=True)
    eps_local = 1e-6

    # --------------------------------------------------------
    # Índices radiativos
    # --------------------------------------------------------
    csghi = out[COL_CSGHI].to_numpy(dtype=float)
    csdni = out[COL_CSDNI].to_numpy(dtype=float)
    csdhi = out[COL_CSDHI].to_numpy(dtype=float)

    ghi = out[COL_GHI].to_numpy(dtype=float)
    dni = out[COL_DNI].to_numpy(dtype=float)
    dhi = out[COL_DHI].to_numpy(dtype=float)

    kg = np.zeros(len(out), dtype=float)
    kn = np.zeros(len(out), dtype=float)
    kd = np.zeros(len(out), dtype=float)

    m = csghi > cfg2050.thr_cs
    kg[m] = ghi[m] / np.maximum(csghi[m], eps_local)

    m = csdni > cfg2050.thr_cs
    kn[m] = dni[m] / np.maximum(csdni[m], eps_local)

    m = csdhi > cfg2050.thr_cs
    kd[m] = dhi[m] / np.maximum(csdhi[m], eps_local)

    out["kGHI"] = np.clip(kg, 0.0, 2.0)
    out["kDNI"] = np.clip(kn, 0.0, cfg2050.kdni_max)
    out["kDHI"] = np.clip(kd, 0.0, cfg2050.kdhi_max)
    out["qDHI"] = np.log1p(out["kDHI"])

    # --------------------------------------------------------
    # Ratios radiativos
    # --------------------------------------------------------
    ghi_safe = np.maximum(ghi, eps_local)

    out["diffuse_ratio"] = np.clip(
        dhi / ghi_safe, 0.0, 1.2
    )

    out["beam_ratio"] = np.clip(
        dni * out["cosz_plus"].to_numpy(dtype=float) / ghi_safe,
        0.0, 1.2
    )

    # --------------------------------------------------------
    # Diferencias
    # --------------------------------------------------------
    out["dGHI"] = out[COL_GHI].diff()
    out["dDNI"] = out[COL_DNI].diff()
    out["dDHI"] = out[COL_DHI].diff()

    # --------------------------------------------------------
    # Irradiancia: lag + rolling
    # --------------------------------------------------------
    out["GHI_lag1"] = out[COL_GHI].shift(1)

    out["DNI_lag1"] = out[COL_DNI].shift(1)
    out["DNI_roll3"] = out[COL_DNI].rolling(3).mean()

    out["DHI_lag1"] = out[COL_DHI].shift(1)
    out["DHI_roll3"] = out[COL_DHI].rolling(3).mean()

    # --------------------------------------------------------
    # kGHI
    # --------------------------------------------------------
    out["kGHI_lag1"] = out["kGHI"].shift(1)
    out["kGHI_roll3"] = out["kGHI"].rolling(3).mean()

    # --------------------------------------------------------
    # kDNI
    # --------------------------------------------------------
    out["kDNI_lag1"] = out["kDNI"].shift(1)
    out["kDNI_roll3"] = out["kDNI"].rolling(3).mean()

    # --------------------------------------------------------
    # qDHI — representación difusa V2.4
    # --------------------------------------------------------
    out["qDHI_lag1"] = out["qDHI"].shift(1)
    out["qDHI_roll3"] = out["qDHI"].rolling(3).mean()

    # --------------------------------------------------------
    # Ratios
    # --------------------------------------------------------
    out["diffuse_ratio_lag1"] = out["diffuse_ratio"].shift(1)
    out["diffuse_ratio_roll3"] = out["diffuse_ratio"].rolling(3).mean()

    out["beam_ratio_lag1"] = out["beam_ratio"].shift(1)
    out["beam_ratio_roll3"] = out["beam_ratio"].rolling(3).mean()

    return out


# ------------------------------------------------------------
# 5. Concatenar semilla histórica + primer bloque sintético
#
# Aquí sí existe deliberadamente un salto de calendario:
#
#   2024-12-31 23:30 -> 2050-01-01 00:00
#
# Esto NO representa 25 años de dinámica temporal simulada.
# La semilla únicamente inicializa el estado del modelo para
# el comienzo del año sintético 2050.
#
# Las features temporales futuras describen explícitamente el
# calendario 2050.
# ------------------------------------------------------------

state_after_block1_base_2050 = pd.concat(
    [seed_base_2050, synthetic_block1_state_2050],
    ignore_index=True
)

assert len(state_after_block1_base_2050) == cfg2050.L + cfg2050.H

state_after_block1_feat_2050 = build_historical_features_v24v1(
    state_after_block1_base_2050
)


# ------------------------------------------------------------
# 6. Extraer solamente las nuevas 24 filas
#
# Como existen 72 registros anteriores, todos los lags y
# rolling del bloque sintético deben estar completamente
# definidos.
# ------------------------------------------------------------

new_features_block1_2050 = (
    state_after_block1_feat_2050
    .iloc[-cfg2050.H:]
    .copy()
)

assert not new_features_block1_2050[
    hist_cols_meta_2050
].isna().any().any()

assert np.isfinite(
    new_features_block1_2050[
        hist_cols_meta_2050
    ].to_numpy(dtype=float)
).all()


# ------------------------------------------------------------
# 7. Validar rangos de las variables de salida reconstruidas
# ------------------------------------------------------------

assert new_features_block1_2050["kDNI"].between(
    0.0, cfg2050.kdni_max
).all()

assert new_features_block1_2050["kDHI"].between(
    0.0, cfg2050.kdhi_max
).all()

assert new_features_block1_2050["qDHI"].between(
    0.0, cfg2050.qdhi_max + 1e-9
).all()


# ------------------------------------------------------------
# 8. Nuevo estado RAW: conservar los últimos L=72 registros
# ------------------------------------------------------------

state_after_block1_raw_2050 = (
    state_after_block1_feat_2050
    .iloc[-cfg2050.L:]
    .copy()
    .reset_index(drop=True)
)

assert len(state_after_block1_raw_2050) == cfg2050.L
assert not state_after_block1_raw_2050[
    hist_cols_meta_2050
].isna().any().any()


# ------------------------------------------------------------
# 9. Escalar el nuevo estado histórico
#
# Usamos DataFrame con nombres, igual que durante fit del
# StandardScaler.
# ------------------------------------------------------------

state_after_block1_input_df = (
    state_after_block1_raw_2050[
        hist_cols_meta_2050
    ]
    .astype(np.float32)
)

state_after_block1_scaled_2050 = (
    scaler_hist_v24v1_2050
    .transform(state_after_block1_input_df)
    .astype(np.float32)
)

assert state_after_block1_scaled_2050.shape == (cfg2050.L, 34)
assert np.isfinite(state_after_block1_scaled_2050).all()


# ------------------------------------------------------------
# 10. Auditoría temporal del nuevo estado
#
# Los 72 registros ahora contienen:
#
#   48 registros finales de la semilla histórica
#   24 registros sintéticos del primer bloque 2050
#
# Por eso no exigimos continuidad cronológica global de 30 min
# a través del salto 2024 -> 2050.
#
# Sí exigimos continuidad dentro del tramo sintético.
# ------------------------------------------------------------

synthetic_tail_2050 = state_after_block1_raw_2050[
    state_after_block1_raw_2050["timestamp"].dt.year == 2050
]

synthetic_dt_2050 = (
    synthetic_tail_2050["timestamp"]
    .diff()
    .dropna()
    .dt.total_seconds()
    / 60.0
)

assert len(synthetic_tail_2050) == cfg2050.H
assert (synthetic_dt_2050 == 30.0).all()

print("=== NUEVO ESTADO AUTOREGRESIVO ===")
print("Filas RAW            :", len(state_after_block1_raw_2050))
print("Filas sintéticas 2050:", len(synthetic_tail_2050))
print("Inicio estado        :", state_after_block1_raw_2050["timestamp"].iloc[0])
print("Fin estado           :", state_after_block1_raw_2050["timestamp"].iloc[-1])
print("Shape escalado       :", state_after_block1_scaled_2050.shape)


# ------------------------------------------------------------
# 11. Segundo bloque futuro
#
# Esto permitirá verificar que ya tenemos ambas entradas para
# la siguiente llamada al modelo:
#
#       X_hist -> estado actualizado
#       X_fut  -> 2050-01-01 12:00 ... 23:30
# ------------------------------------------------------------

second_block_start = cfg2050.H
second_block_end = 2 * cfg2050.H

second_future_block_2050 = (
    X_future_all_2050[
        second_block_start:second_block_end
    ]
    .copy()
)

assert second_future_block_2050.shape == (cfg2050.H, 17)

print("\n=== SEGUNDO BLOQUE PREPARADO ===")
print(
    "Inicio:",
    df_2050_base["timestamp"].iloc[second_block_start]
)
print(
    "Fin   :",
    df_2050_base["timestamp"].iloc[second_block_end - 1]
)
print("X_hist:", state_after_block1_scaled_2050.shape)
print("X_fut :", second_future_block_2050.shape)


# ------------------------------------------------------------
# 12. Auditoría de las nuevas features sintéticas
# ------------------------------------------------------------

print("\n=== FEATURES DEL PRIMER BLOQUE GENERADO ===")

diagnostic_cols_block1 = [
    "timestamp",
    COL_GHI, COL_DNI, COL_DHI,
    "kGHI", "kDNI", "kDHI", "qDHI",
    "diffuse_ratio", "beam_ratio"
]

print(
    new_features_block1_2050[
        diagnostic_cols_block1
    ].tail(8).to_string(index=False)
)


# ------------------------------------------------------------
# 13. Guardar estado diagnóstico
# ------------------------------------------------------------

STATE_BLOCK1_PATH_2050 = (
    OUTPUT_DIR_2050_V24V1 /
    "autoregressive_state_after_block1_v24_v1.csv"
)

state_after_block1_raw_2050[
    ["timestamp"] + hist_cols_meta_2050
].to_csv(
    STATE_BLOCK1_PATH_2050,
    index=False
)

print("\nArchivo:")
print(STATE_BLOCK1_PATH_2050.name)


# ------------------------------------------------------------
# 14. Resumen
# ------------------------------------------------------------

print("\n✓ Primer bloque convertido a estado histórico sintético.")
print("✓ 34 features autoregresivas reconstruidas.")
print("✓ Nuevo historial L=72 construido.")
print("✓ Scaler histórico aplicado al estado actualizado.")
print("✓ Segundo bloque futuro preparado.")
print("✓ Celda 8 completada correctamente.")

=== NUEVO ESTADO AUTOREGRESIVO ===
Filas RAW            : 72
Filas sintéticas 2050: 24
Inicio estado        : 2024-12-31 00:00:00
Fin estado           : 2050-01-01 11:30:00
Shape escalado       : (72, 34)

=== SEGUNDO BLOQUE PREPARADO ===
Inicio: 2050-01-01 12:00:00
Fin   : 2050-01-01 23:30:00
X_hist: (72, 34)
X_fut : (24, 17)

=== FEATURES DEL PRIMER BLOQUE GENERADO ===
          timestamp        GHI        DNI        DHI     kGHI     kDNI     kDHI     qDHI  diffuse_ratio  beam_ratio
2050-01-01 08:00:00 185.222896  35.051796 171.224563 0.488715 0.047113 2.113883 1.135871       0.924424    0.075576
2050-01-01 08:30:00 242.259276  47.085678 218.528146 0.482588 0.057987 2.401408 1.224190       0.902042    0.097958
2050-01-01 09:00:00 303.670032  82.409263 254.227053 0.490582 0.095161 2.704543 1.309560       0.837182    0.162818
2050-01-01 09:30:00 370.385929 104.934288 298.439901 0.513000 0.116335 2.925881 1.367591       0.805754    0.194246
2050-01-01 10:00:00 443.890620 127.752553 346.

## Postproceso y auditoría posterior

La salida **raw** del modelo se genera y conserva íntegramente antes de aplicar cualquier ajuste posterior.

Una vez completada la simulación anual 2050, se evaluarán de forma independiente:

1. la consistencia radiativa y energética;
2. la distribución mensual de GHI, DNI y DHI;
3. la fracción difusa y su comportamiento por régimen solar;
4. la posible deriva del proceso autoregresivo a lo largo del año.

Solo si estas auditorías muestran sesgos sistemáticos relevantes se considerará un postproceso adicional.

Cualquier corrección posterior deberá quedar explícitamente separada de la salida del PI-LSTM y no podrá interpretarse como un parámetro aprendido por la red. Si se introduce algún factor de ajuste o calibración, deberá documentarse como un **hiperparámetro post hoc**, justificarse físicamente y someterse a análisis de sensibilidad.

In [9]:
# ============================================================
# 9. ROLLING AUTOREGRESIVO COMPLETO — AÑO SINTÉTICO 2050 RAW
# ============================================================

# Estado inicial: se parte nuevamente de la semilla original
rolling_state_base_2050 = seed_base_2050.copy().reset_index(drop=True)
rolling_state_scaled_2050 = seed_hist_scaled_2050.copy()

assert len(rolling_state_base_2050) == cfg2050.L
assert rolling_state_scaled_2050.shape == (cfg2050.L, 34)

generated_blocks_2050 = []

print("============================================================")
print(" ROLLING AUTOREGRESIVO 2050 — V2.4 V1 RAW")
print("============================================================")
print("Bloques totales:", n_blocks_2050)
print("H por bloque    :", cfg2050.H)
print("Registros       :", n_steps_2050)

for block_id in range(n_blocks_2050):

    i0 = block_id * cfg2050.H
    i1 = i0 + cfg2050.H

    # --------------------------------------------------------
    # 1. Entradas e inferencia
    # --------------------------------------------------------

    Xh = rolling_state_scaled_2050[np.newaxis, ...].astype(np.float32)
    Xf = X_future_all_2050[i0:i1][np.newaxis, ...].astype(np.float32)

    assert Xh.shape == (1, cfg2050.L, 34)
    assert Xf.shape == (1, cfg2050.H, 17)

    pred = model_v24v1_2050([Xh, Xf], training=False).numpy()[0]

    if pred.shape != (cfg2050.H, 2) or not np.isfinite(pred).all():
        raise RuntimeError(f"Bloque {block_id}: salida del modelo inválida.")

    # --------------------------------------------------------
    # 2. Salidas normalizadas y reconstrucción física
    # --------------------------------------------------------

    kdni_pred = np.clip(pred[:, 0], 0.0, cfg2050.kdni_max)
    qdhi_pred = np.clip(pred[:, 1], 0.0, cfg2050.qdhi_max)
    kdhi_pred = np.clip(np.expm1(qdhi_pred), 0.0, cfg2050.kdhi_max)

    base_block = df_2050_base.iloc[i0:i1].copy()

    csdni_b = base_block[COL_CSDNI].to_numpy(dtype=np.float64)
    csdhi_b = base_block[COL_CSDHI].to_numpy(dtype=np.float64)
    cosz_b = base_block["cosz_plus"].to_numpy(dtype=np.float64)

    dni_pred = kdni_pred.astype(np.float64) * csdni_b
    dhi_pred = kdhi_pred.astype(np.float64) * csdhi_b
    ghi_pred = dni_pred * cosz_b + dhi_pred

    night_b = cosz_b <= 0.0
    dni_pred[night_b] = 0.0
    dhi_pred[night_b] = 0.0
    ghi_pred[night_b] = 0.0

    if (
        not np.isfinite(dni_pred).all()
        or not np.isfinite(dhi_pred).all()
        or not np.isfinite(ghi_pred).all()
    ):
        raise RuntimeError(f"Bloque {block_id}: irradiancia contiene NaN/Inf.")

    if (dni_pred < 0).any() or (dhi_pred < 0).any() or (ghi_pred < 0).any():
        raise RuntimeError(f"Bloque {block_id}: irradiancia negativa.")

    closure_b = ghi_pred - (dni_pred * cosz_b + dhi_pred)

    if np.max(np.abs(closure_b)) > 1e-6:
        raise RuntimeError(f"Bloque {block_id}: fallo de cierre radiativo.")

    # --------------------------------------------------------
    # 3. Guardar salida RAW del bloque
    # --------------------------------------------------------

    block_output = pd.DataFrame({
        "timestamp": base_block["timestamp"].to_numpy(),
        "block_id": block_id,
        "horizon_step": np.arange(1, cfg2050.H + 1),
        "Solar Zenith Angle": base_block[COL_ZENITH].to_numpy(dtype=float),
        "cosz_plus": cosz_b,
        "Clearsky GHI": base_block[COL_CSGHI].to_numpy(dtype=float),
        "Clearsky DNI": csdni_b,
        "Clearsky DHI": csdhi_b,
        "kDNI_raw": kdni_pred,
        "qDHI_raw": qdhi_pred,
        "kDHI_raw": kdhi_pred,
        "DNI_raw": dni_pred,
        "DHI_raw": dhi_pred,
        "GHI_raw": ghi_pred,
    })

    generated_blocks_2050.append(block_output)

    # --------------------------------------------------------
    # 4. Construir bloque físico para retroalimentación
    # --------------------------------------------------------

    synthetic_state_block = pd.DataFrame({
        "timestamp": base_block["timestamp"].to_numpy(),
        COL_GHI: ghi_pred,
        COL_DNI: dni_pred,
        COL_DHI: dhi_pred,
        COL_CSGHI: base_block[COL_CSGHI].to_numpy(dtype=float),
        COL_CSDNI: csdni_b,
        COL_CSDHI: csdhi_b,
        COL_ZENITH: base_block[COL_ZENITH].to_numpy(dtype=float),
        "cosz": base_block["cosz"].to_numpy(dtype=float),
        "cosz_plus": cosz_b,
        "Temperature": base_block["Temperature"].to_numpy(dtype=float),
        "Dew Point": base_block["Dew Point"].to_numpy(dtype=float),
        "Relative Humidity": base_block["Relative Humidity"].to_numpy(dtype=float),
        "Pressure": base_block["Pressure"].to_numpy(dtype=float),
        "Aerosol Optical Depth": base_block["Aerosol Optical Depth"].to_numpy(dtype=float),
        "Precipitable Water": base_block["Precipitable Water"].to_numpy(dtype=float),
        "Wind Speed": base_block["Wind Speed"].to_numpy(dtype=float),
        "Wind Direction": base_block["Wind Direction"].to_numpy(dtype=float),
        "sin_wdir": base_block["sin_wdir"].to_numpy(dtype=float),
        "cos_wdir": base_block["cos_wdir"].to_numpy(dtype=float),
    })

    # --------------------------------------------------------
    # 5. Actualizar estado autoregresivo
    # --------------------------------------------------------

    combined_state = pd.concat(
        [rolling_state_base_2050, synthetic_state_block],
        ignore_index=True
    )

    combined_features = build_historical_features_v24v1(combined_state)

    new_state_features = (
        combined_features
        .iloc[-cfg2050.L:]
        .copy()
        .reset_index(drop=True)
    )

    if new_state_features[hist_cols_meta_2050].isna().any().any():
        raise RuntimeError(f"Bloque {block_id}: NaN en estado autoregresivo.")

    if not np.isfinite(
        new_state_features[hist_cols_meta_2050].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(f"Bloque {block_id}: Inf en estado autoregresivo.")

    assert new_state_features["kDNI"].between(0.0, cfg2050.kdni_max).all()
    assert new_state_features["kDHI"].between(0.0, cfg2050.kdhi_max).all()
    assert new_state_features["qDHI"].between(0.0, cfg2050.qdhi_max + 1e-9).all()
    assert new_state_features["diffuse_ratio"].between(0.0, 1.2).all()
    assert new_state_features["beam_ratio"].between(0.0, 1.5).all()

    rolling_state_scaled_2050 = (
        scaler_hist_v24v1_2050
        .transform(new_state_features[hist_cols_meta_2050].astype(np.float32))
        .astype(np.float32)
    )

    rolling_state_base_2050 = (
        combined_state
        .iloc[-cfg2050.L:]
        .copy()
        .reset_index(drop=True)
    )

    assert rolling_state_scaled_2050.shape == (cfg2050.L, 34)

    if block_id == 0 or (block_id + 1) % 50 == 0 or block_id == n_blocks_2050 - 1:
        print(
            f"Bloque {block_id + 1:3d}/{n_blocks_2050} "
            f"| {base_block['timestamp'].iloc[-1]}"
        )

# ============================================================
# 6. CONCATENACIÓN DEL AÑO RAW
# ============================================================

df_2050_raw = pd.concat(generated_blocks_2050, ignore_index=True)

assert len(df_2050_raw) == 17_520
assert df_2050_raw["timestamp"].iloc[0] == pd.Timestamp("2050-01-01 00:00:00")
assert df_2050_raw["timestamp"].iloc[-1] == pd.Timestamp("2050-12-31 23:30:00")

n_duplicates_raw_2050 = int(df_2050_raw["timestamp"].duplicated().sum())
dt_raw_2050 = df_2050_raw["timestamp"].diff().dropna().dt.total_seconds() / 60.0
n_bad_steps_raw_2050 = int((dt_raw_2050 != 30.0).sum())

assert n_duplicates_raw_2050 == 0
assert n_bad_steps_raw_2050 == 0

# ============================================================
# 7. AUDITORÍA FÍSICA ANUAL
# ============================================================

closure_raw_2050 = (
    df_2050_raw["GHI_raw"].to_numpy(dtype=float)
    - (
        df_2050_raw["DNI_raw"].to_numpy(dtype=float)
        * df_2050_raw["cosz_plus"].to_numpy(dtype=float)
        + df_2050_raw["DHI_raw"].to_numpy(dtype=float)
    )
)

closure_mae_raw_2050 = float(np.mean(np.abs(closure_raw_2050)))
closure_rmse_raw_2050 = float(np.sqrt(np.mean(closure_raw_2050**2)))
closure_max_raw_2050 = float(np.max(np.abs(closure_raw_2050)))

night_mask_raw_2050 = df_2050_raw["cosz_plus"].to_numpy(dtype=float) <= 0.0
night_leakage_raw_2050 = int(
    (
        df_2050_raw.loc[
            night_mask_raw_2050,
            ["GHI_raw", "DNI_raw", "DHI_raw"]
        ].to_numpy() != 0.0
    ).sum()
)

assert closure_max_raw_2050 < 1e-6
assert night_leakage_raw_2050 == 0
assert not df_2050_raw[["GHI_raw", "DNI_raw", "DHI_raw"]].isna().any().any()
assert np.isfinite(df_2050_raw[["GHI_raw", "DNI_raw", "DHI_raw"]].to_numpy()).all()

# ============================================================
# 8. DOMINIOS Y SATURACIÓN
# ============================================================

assert df_2050_raw["kDNI_raw"].between(0.0, cfg2050.kdni_max + 1e-7).all()
assert df_2050_raw["qDHI_raw"].between(0.0, cfg2050.qdhi_max + 1e-7).all()
assert df_2050_raw["kDHI_raw"].between(0.0, cfg2050.kdhi_max + 1e-6).all()

day_mask_raw_2050 = (
    df_2050_raw["cosz_plus"].to_numpy(dtype=float)
    > cfg2050.thr_day_cosz
)

n_day_raw_2050 = int(day_mask_raw_2050.sum())

kdni_sat_raw_2050 = int(
    (
        df_2050_raw.loc[day_mask_raw_2050, "kDNI_raw"]
        >= 0.99 * cfg2050.kdni_max
    ).sum()
)

kdhi_sat_raw_2050 = int(
    (
        df_2050_raw.loc[day_mask_raw_2050, "kDHI_raw"]
        >= 0.99 * cfg2050.kdhi_max
    ).sum()
)

kdni_sat_pct_raw_2050 = 100.0 * kdni_sat_raw_2050 / max(n_day_raw_2050, 1)
kdhi_sat_pct_raw_2050 = 100.0 * kdhi_sat_raw_2050 / max(n_day_raw_2050, 1)

print("\n=== RANGOS ANUALES DE SALIDA ===")
for col in ["kDNI_raw", "qDHI_raw", "kDHI_raw"]:
    s = df_2050_raw[col]
    print(f"{col:10s} min={s.min():.6f} mean={s.mean():.6f} max={s.max():.6f}")

print("\n=== IRRADIANCIA RAW 2050 ===")
for col in ["GHI_raw", "DNI_raw", "DHI_raw"]:
    s = df_2050_raw[col]
    print(f"{col:8s} min={s.min():8.3f} mean={s.mean():8.3f} max={s.max():8.3f}")

print("\n=== AUDITORÍA FÍSICA ANUAL ===")
print("Duplicados timestamp :", n_duplicates_raw_2050)
print("Saltos != 30 min     :", n_bad_steps_raw_2050)
print("Closure MAE          :", closure_mae_raw_2050)
print("Closure RMSE         :", closure_rmse_raw_2050)
print("Closure max abs      :", closure_max_raw_2050)
print("Night leakage        :", night_leakage_raw_2050)

print("\n=== SATURACIÓN DAYTIME ===")
print(f"kDNI >= 99% cap : {kdni_sat_raw_2050} ({kdni_sat_pct_raw_2050:.3f} %)")
print(f"kDHI >= 99% cap : {kdhi_sat_raw_2050} ({kdhi_sat_pct_raw_2050:.3f} %)")

# ============================================================
# 9. REPRODUCIBILIDAD DEL PRIMER BLOQUE
# ============================================================

for raw_col, smoke_col in [
    ("GHI_raw", "GHI_pred"),
    ("DNI_raw", "DNI_pred"),
    ("DHI_raw", "DHI_pred"),
]:
    np.testing.assert_allclose(
        df_2050_raw.iloc[:cfg2050.H][raw_col].to_numpy(),
        first_block_result_2050[smoke_col].to_numpy(),
        rtol=1e-6,
        atol=1e-6
    )

print("\nPrimer bloque reproduce Celda 7: OK")

# ============================================================
# 10. PERSISTENCIA RAW Y RESUMEN DE AUDITORÍA
# ============================================================

RAW_2050_PATH = OUTPUT_DIR_2050_V24V1 / "PI_LSTM_V24V1_2050_RAW.csv"
RAW_AUDIT_PATH_2050 = OUTPUT_DIR_2050_V24V1 / "PI_LSTM_V24V1_2050_RAW_audit.json"

df_2050_raw.to_csv(RAW_2050_PATH, index=False)

raw_audit_summary_2050 = {
    "model": "PI-LSTM V2.4 V1 RAW",
    "rows": int(len(df_2050_raw)),
    "blocks": int(n_blocks_2050),
    "L": int(cfg2050.L),
    "H": int(cfg2050.H),
    "timestamp_start": str(df_2050_raw["timestamp"].min()),
    "timestamp_end": str(df_2050_raw["timestamp"].max()),
    "duplicate_timestamps": n_duplicates_raw_2050,
    "bad_30min_steps": n_bad_steps_raw_2050,
    "closure_mae_Wm2": closure_mae_raw_2050,
    "closure_rmse_Wm2": closure_rmse_raw_2050,
    "closure_max_abs_Wm2": closure_max_raw_2050,
    "night_leakage_count": night_leakage_raw_2050,
    "daytime_records": n_day_raw_2050,
    "kdni_saturation_count": kdni_sat_raw_2050,
    "kdni_saturation_pct": kdni_sat_pct_raw_2050,
    "kdhi_saturation_count": kdhi_sat_raw_2050,
    "kdhi_saturation_pct": kdhi_sat_pct_raw_2050,
}

with open(RAW_AUDIT_PATH_2050, "w", encoding="utf-8") as f:
    json.dump(raw_audit_summary_2050, f, indent=2)

print("\n=== ARCHIVOS GENERADOS ===")
print(RAW_2050_PATH.name)
print(RAW_AUDIT_PATH_2050.name)

print("\n✓ 730 bloques procesados.")
print("✓ 17,520 registros sintéticos generados.")
print("✓ Rolling autoregresivo completo.")
print("✓ Features históricas recalculadas con contrato V2.4 V1.")
print("✓ Cierre físico anual verificado.")
print("✓ Leakage nocturno = 0.")
print("✓ V2.4 V1 RAW preservado sin calibración.")
print("✓ Celda 9 completada correctamente.")

 ROLLING AUTOREGRESIVO 2050 — V2.4 V1 RAW
Bloques totales: 730
H por bloque    : 24
Registros       : 17520
Bloque   1/730 | 2050-01-01 11:30:00
Bloque  50/730 | 2050-01-25 23:30:00
Bloque 100/730 | 2050-02-19 23:30:00
Bloque 150/730 | 2050-03-16 23:30:00
Bloque 200/730 | 2050-04-10 23:30:00
Bloque 250/730 | 2050-05-05 23:30:00
Bloque 300/730 | 2050-05-30 23:30:00
Bloque 350/730 | 2050-06-24 23:30:00
Bloque 400/730 | 2050-07-19 23:30:00
Bloque 450/730 | 2050-08-13 23:30:00
Bloque 500/730 | 2050-09-07 23:30:00
Bloque 550/730 | 2050-10-02 23:30:00
Bloque 600/730 | 2050-10-27 23:30:00
Bloque 650/730 | 2050-11-21 23:30:00
Bloque 700/730 | 2050-12-16 23:30:00
Bloque 730/730 | 2050-12-31 23:30:00

=== RANGOS ANUALES DE SALIDA ===
kDNI_raw   min=0.000000 mean=0.226671 max=0.944871
qDHI_raw   min=0.000000 mean=0.568963 max=1.848372
kDHI_raw   min=0.000000 mean=1.162960 max=5.349476

=== IRRADIANCIA RAW 2050 ===
GHI_raw  min=   0.000 mean= 248.409 max=1022.138
DNI_raw  min=   0.000 mean= 186.89

In [10]:
# ============================================================
# 10. AUDITORÍA ENERGÉTICA Y CLIMATOLÓGICA — 2050 RAW
# ============================================================
#
# OBJETIVO
# --------
# Evaluar el año sintético RAW antes de decidir si requiere
# cualquier tipo de postproceso.
#
# Se comparará con la climatología NSRDB 1998–2024 mediante:
#
#   1. energía anual GHI/DNI/DHI;
#   2. climatología energética mensual;
#   3. diffuse ratio puntual y energético;
#   4. comportamiento por régimen de cos(z);
#   5. evolución mensual de kDNI/kDHI;
#   6. posible deriva autoregresiva durante 2050.
#
# IMPORTANTE
# ----------
# Esta celda es exclusivamente diagnóstica.
# NO modifica df_2050_raw.
# ============================================================


# ------------------------------------------------------------
# 1. Preparar histórico para referencia energética
# ------------------------------------------------------------

DT_HOURS = 0.5

hist_audit_2050 = hist_feat_2050.copy()
hist_audit_2050["year"] = hist_audit_2050["timestamp"].dt.year
hist_audit_2050["month"] = hist_audit_2050["timestamp"].dt.month

raw_audit_2050 = df_2050_raw.copy()
raw_audit_2050["month"] = raw_audit_2050["timestamp"].dt.month


# ------------------------------------------------------------
# 2. Energía anual histórica por año
#
# E = sum(P * Δt) / 1000
#
# con Δt = 0.5 h y resultado en kWh/m².
# ------------------------------------------------------------

hist_annual_rows = []

for year, g in hist_audit_2050.groupby("year"):
    hist_annual_rows.append({
        "year": int(year),
        "GHI_energy": g[COL_GHI].sum() * DT_HOURS / 1000.0,
        "DNI_energy": g[COL_DNI].sum() * DT_HOURS / 1000.0,
        "DHI_energy": g[COL_DHI].sum() * DT_HOURS / 1000.0,
    })

hist_annual_energy_2050 = pd.DataFrame(hist_annual_rows)

raw_annual_energy_2050 = {
    "GHI": raw_audit_2050["GHI_raw"].sum() * DT_HOURS / 1000.0,
    "DNI": raw_audit_2050["DNI_raw"].sum() * DT_HOURS / 1000.0,
    "DHI": raw_audit_2050["DHI_raw"].sum() * DT_HOURS / 1000.0,
}


# ------------------------------------------------------------
# 3. Comparación anual contra distribución histórica
#
# Además de media y mediana mostramos P05–P95. Esto permite
# saber si 2050 cae dentro del rango climatológico observado
# sin convertir la climatología en una corrección.
# ------------------------------------------------------------

annual_compare_rows = []

for var in ["GHI", "DNI", "DHI"]:
    h = hist_annual_energy_2050[f"{var}_energy"]
    raw_e = raw_annual_energy_2050[var]

    annual_compare_rows.append({
        "variable": var,
        "hist_mean_kWh_m2": h.mean(),
        "hist_median_kWh_m2": h.median(),
        "hist_p05_kWh_m2": h.quantile(0.05),
        "hist_p95_kWh_m2": h.quantile(0.95),
        "raw_2050_kWh_m2": raw_e,
        "error_vs_hist_mean_pct": 100.0 * (raw_e / h.mean() - 1.0),
        "error_vs_hist_median_pct": 100.0 * (raw_e / h.median() - 1.0),
    })

annual_compare_2050 = pd.DataFrame(annual_compare_rows)

print("=== ENERGÍA ANUAL: HISTÓRICO vs 2050 RAW ===")
print(annual_compare_2050.to_string(index=False))


# ------------------------------------------------------------
# 4. Climatología energética mensual histórica
#
# Primero calculamos cada mes de cada año y después obtenemos
# la distribución histórica de ese mes.
# ------------------------------------------------------------

hist_month_year_energy = (
    hist_audit_2050
    .groupby(["year", "month"])
    .agg(
        GHI_energy=(COL_GHI, lambda x: x.sum() * DT_HOURS / 1000.0),
        DNI_energy=(COL_DNI, lambda x: x.sum() * DT_HOURS / 1000.0),
        DHI_energy=(COL_DHI, lambda x: x.sum() * DT_HOURS / 1000.0),
    )
    .reset_index()
)

hist_month_clim = (
    hist_month_year_energy
    .groupby("month")
    .agg(
        GHI_hist_mean=("GHI_energy", "mean"),
        GHI_hist_median=("GHI_energy", "median"),
        GHI_hist_p05=("GHI_energy", lambda x: x.quantile(0.05)),
        GHI_hist_p95=("GHI_energy", lambda x: x.quantile(0.95)),

        DNI_hist_mean=("DNI_energy", "mean"),
        DNI_hist_median=("DNI_energy", "median"),
        DNI_hist_p05=("DNI_energy", lambda x: x.quantile(0.05)),
        DNI_hist_p95=("DNI_energy", lambda x: x.quantile(0.95)),

        DHI_hist_mean=("DHI_energy", "mean"),
        DHI_hist_median=("DHI_energy", "median"),
        DHI_hist_p05=("DHI_energy", lambda x: x.quantile(0.05)),
        DHI_hist_p95=("DHI_energy", lambda x: x.quantile(0.95)),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Energía mensual 2050 RAW
# ------------------------------------------------------------

raw_month_energy = (
    raw_audit_2050
    .groupby("month")
    .agg(
        GHI_2050=("GHI_raw", lambda x: x.sum() * DT_HOURS / 1000.0),
        DNI_2050=("DNI_raw", lambda x: x.sum() * DT_HOURS / 1000.0),
        DHI_2050=("DHI_raw", lambda x: x.sum() * DT_HOURS / 1000.0),
    )
    .reset_index()
)

monthly_compare_2050 = raw_month_energy.merge(
    hist_month_clim,
    on="month",
    validate="one_to_one"
)

for var in ["GHI", "DNI", "DHI"]:
    monthly_compare_2050[f"{var}_error_mean_pct"] = (
        100.0 *
        (
            monthly_compare_2050[f"{var}_2050"]
            / monthly_compare_2050[f"{var}_hist_mean"]
            - 1.0
        )
    )

    monthly_compare_2050[f"{var}_inside_P05_P95"] = (
        (monthly_compare_2050[f"{var}_2050"] >=
         monthly_compare_2050[f"{var}_hist_p05"])
        &
        (monthly_compare_2050[f"{var}_2050"] <=
         monthly_compare_2050[f"{var}_hist_p95"])
    )


print("\n=== COMPARACIÓN ENERGÉTICA MENSUAL ===")

cols_show = [
    "month",
    "GHI_2050", "GHI_hist_mean", "GHI_error_mean_pct",
    "DNI_2050", "DNI_hist_mean", "DNI_error_mean_pct",
    "DHI_2050", "DHI_hist_mean", "DHI_error_mean_pct",
]

print(
    monthly_compare_2050[cols_show]
    .round(3)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Resumen de error mensual
# ------------------------------------------------------------

print("\n=== RESUMEN DEL ERROR MENSUAL ABSOLUTO ===")

monthly_error_summary = []

for var in ["GHI", "DNI", "DHI"]:
    err = monthly_compare_2050[f"{var}_error_mean_pct"]

    monthly_error_summary.append({
        "variable": var,
        "mean_bias_pct": err.mean(),
        "mean_abs_error_pct": err.abs().mean(),
        "max_abs_error_pct": err.abs().max(),
        "months_inside_P05_P95": int(
            monthly_compare_2050[f"{var}_inside_P05_P95"].sum()
        ),
    })

monthly_error_summary_2050 = pd.DataFrame(monthly_error_summary)

print(monthly_error_summary_2050.round(3).to_string(index=False))


# ------------------------------------------------------------
# 7. Diffuse ratio
#
# Se calculan dos conceptos distintos:
#
# a) ratio puntual daytime:
#
#       rd(t) = DHI(t) / GHI(t)
#
# b) ratio energético:
#
#       rd,E = E_DHI / E_GHI
#
# No deben confundirse.
# ------------------------------------------------------------

hist_day = (
    (hist_audit_2050["cosz_plus"] > cfg2050.thr_day_cosz)
    &
    (hist_audit_2050[COL_GHI] > cfg2050.thr_cs)
)

raw_day = (
    (raw_audit_2050["cosz_plus"] > cfg2050.thr_day_cosz)
    &
    (raw_audit_2050["GHI_raw"] > cfg2050.thr_cs)
)

hist_rd_point = np.clip(
    hist_audit_2050.loc[hist_day, COL_DHI]
    /
    hist_audit_2050.loc[hist_day, COL_GHI],
    0.0, 1.2
)

raw_rd_point = np.clip(
    raw_audit_2050.loc[raw_day, "DHI_raw"]
    /
    raw_audit_2050.loc[raw_day, "GHI_raw"],
    0.0, 1.2
)

hist_rd_energy_year = (
    hist_annual_energy_2050["DHI_energy"]
    /
    hist_annual_energy_2050["GHI_energy"]
)

raw_rd_energy = (
    raw_annual_energy_2050["DHI"]
    /
    raw_annual_energy_2050["GHI"]
)

ratio_summary_2050 = pd.DataFrame([{
    "hist_point_mean": hist_rd_point.mean(),
    "hist_point_median": hist_rd_point.median(),
    "raw2050_point_mean": raw_rd_point.mean(),
    "raw2050_point_median": raw_rd_point.median(),

    "hist_energy_mean": hist_rd_energy_year.mean(),
    "hist_energy_median": hist_rd_energy_year.median(),
    "hist_energy_p05": hist_rd_energy_year.quantile(0.05),
    "hist_energy_p95": hist_rd_energy_year.quantile(0.95),

    "raw2050_energy_ratio": raw_rd_energy,
}])

print("\n=== DIFFUSE RATIO ===")
print(ratio_summary_2050.T.to_string(header=False))


# ------------------------------------------------------------
# 8. Comportamiento por régimen de cos(z)
#
# Permite verificar si la partición se deteriora especialmente
# con Sol bajo o con alta elevación solar.
# ------------------------------------------------------------

cosz_bins = [0.05, 0.20, 0.40, 0.60, 0.80, 1.01]
cosz_labels = [
    "0.05-0.20",
    "0.20-0.40",
    "0.40-0.60",
    "0.60-0.80",
    "0.80-1.00"
]

raw_day_regime = raw_audit_2050[
    raw_audit_2050["cosz_plus"] > cfg2050.thr_day_cosz
].copy()

raw_day_regime["cosz_band"] = pd.cut(
    raw_day_regime["cosz_plus"],
    bins=cosz_bins,
    labels=cosz_labels,
    right=False
)

raw_day_regime["rd"] = np.clip(
    raw_day_regime["DHI_raw"]
    / np.maximum(raw_day_regime["GHI_raw"], 1e-6),
    0.0, 1.2
)

cosz_summary_2050 = (
    raw_day_regime
    .groupby("cosz_band", observed=True)
    .agg(
        N=("GHI_raw", "size"),
        GHI_mean=("GHI_raw", "mean"),
        DNI_mean=("DNI_raw", "mean"),
        DHI_mean=("DHI_raw", "mean"),
        kDNI_mean=("kDNI_raw", "mean"),
        kDHI_mean=("kDHI_raw", "mean"),
        rd_mean=("rd", "mean"),
    )
    .reset_index()
)

print("\n=== 2050 RAW POR RÉGIMEN SOLAR ===")
print(cosz_summary_2050.round(4).to_string(index=False))


# ------------------------------------------------------------
# 9. Evolución mensual del estado autoregresivo
#
# Una deriva progresiva aparecería, por ejemplo, como aumento
# o disminución monotónica importante de kDNI o kDHI.
# ------------------------------------------------------------

monthly_state_2050 = (
    raw_audit_2050
    .groupby("month")
    .agg(
        kDNI_mean=("kDNI_raw", "mean"),
        kDNI_p95=("kDNI_raw", lambda x: x.quantile(0.95)),
        kDHI_mean=("kDHI_raw", "mean"),
        kDHI_p95=("kDHI_raw", lambda x: x.quantile(0.95)),
        GHI_mean=("GHI_raw", "mean"),
        DNI_mean=("DNI_raw", "mean"),
        DHI_mean=("DHI_raw", "mean"),
    )
    .reset_index()
)

print("\n=== EVOLUCIÓN MENSUAL DEL ROLLING ===")
print(monthly_state_2050.round(4).to_string(index=False))


# ------------------------------------------------------------
# 10. Comparación inicio vs final del rolling
#
# Dividimos los 730 bloques en cuatro cuartiles iguales
# aproximadamente. No buscamos eliminar estacionalidad aquí;
# simplemente obtener una primera señal de comportamiento.
# ------------------------------------------------------------

raw_audit_2050["rolling_quarter"] = pd.qcut(
    raw_audit_2050["block_id"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

rolling_quarter_summary = (
    raw_audit_2050
    .groupby("rolling_quarter", observed=True)
    .agg(
        kDNI_mean=("kDNI_raw", "mean"),
        kDHI_mean=("kDHI_raw", "mean"),
        GHI_mean=("GHI_raw", "mean"),
        DNI_mean=("DNI_raw", "mean"),
        DHI_mean=("DHI_raw", "mean"),
    )
    .reset_index()
)

print("\n=== RESUMEN POR CUARTIL DEL ROLLING ===")
print(rolling_quarter_summary.round(4).to_string(index=False))


# ------------------------------------------------------------
# 11. Meses fuera del intervalo histórico P05–P95
# ------------------------------------------------------------

print("\n=== MESES FUERA DEL RANGO HISTÓRICO P05–P95 ===")

for var in ["GHI", "DNI", "DHI"]:
    outside = monthly_compare_2050.loc[
        ~monthly_compare_2050[f"{var}_inside_P05_P95"],
        "month"
    ].tolist()

    print(f"{var}: {outside if outside else 'ninguno'}")


# ------------------------------------------------------------
# 12. Guardar auditorías
# ------------------------------------------------------------

ANNUAL_COMPARE_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "annual_energy_compare_2050_RAW.csv"
)

MONTHLY_COMPARE_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "monthly_energy_compare_2050_RAW.csv"
)

RATIO_SUMMARY_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "diffuse_ratio_compare_2050_RAW.csv"
)

COSZ_SUMMARY_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "cosz_regime_2050_RAW.csv"
)

ROLLING_MONTHLY_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "rolling_monthly_diagnostics_2050_RAW.csv"
)

annual_compare_2050.to_csv(ANNUAL_COMPARE_PATH, index=False)
monthly_compare_2050.to_csv(MONTHLY_COMPARE_PATH, index=False)
ratio_summary_2050.to_csv(RATIO_SUMMARY_PATH, index=False)
cosz_summary_2050.to_csv(COSZ_SUMMARY_PATH, index=False)
monthly_state_2050.to_csv(ROLLING_MONTHLY_PATH, index=False)


# ------------------------------------------------------------
# 13. Resumen compacto de decisión
# ------------------------------------------------------------

print("\n============================================================")
print(" RESUMEN DIAGNÓSTICO RAW 2050")
print("============================================================")

for _, r in annual_compare_2050.iterrows():
    print(
        f"{r['variable']}: "
        f"2050={r['raw_2050_kWh_m2']:.2f} kWh/m² | "
        f"hist.mean={r['hist_mean_kWh_m2']:.2f} | "
        f"error={r['error_vs_hist_mean_pct']:+.2f}%"
    )

print(
    f"\nDiffuse energy ratio 2050 : {raw_rd_energy:.6f}"
)
print(
    f"Diffuse energy ratio hist.: "
    f"{hist_rd_energy_year.mean():.6f} (media)"
)

print("\n✓ Energía anual auditada.")
print("✓ Energía mensual comparada con 1998–2024.")
print("✓ Diffuse ratio puntual y energético calculados.")
print("✓ Regímenes solares auditados.")
print("✓ Evolución autoregresiva mensual auditada.")
print("✓ Ninguna corrección aplicada.")
print("✓ Celda 10 completada correctamente.")

=== ENERGÍA ANUAL: HISTÓRICO vs 2050 RAW ===
variable  hist_mean_kWh_m2  hist_median_kWh_m2  hist_p05_kWh_m2  hist_p95_kWh_m2  raw_2050_kWh_m2  error_vs_hist_mean_pct  error_vs_hist_median_pct
     GHI       2058.867037           2052.7415       1980.31135        2149.7077      2176.064183                5.692312                  6.007706
     DNI       1694.191019           1682.7255       1511.04960        1922.5579      1637.186345               -3.364713                 -2.706274
     DHI        858.395352            862.7375        808.96895         902.3799       994.570015               15.863863                 15.280721

=== COMPARACIÓN ENERGÉTICA MENSUAL ===
 month  GHI_2050  GHI_hist_mean  GHI_error_mean_pct  DNI_2050  DNI_hist_mean  DNI_error_mean_pct  DHI_2050  DHI_hist_mean  DHI_error_mean_pct
     1   181.387        170.195               6.576   128.827        138.800              -7.185    88.204         73.335              20.274
     2   168.954        151.096        

In [11]:
# ============================================================
# 11. OBJETIVOS CLIMATOLÓGICOS PARA POSTPROCESO 2050
# ============================================================
#
# OBJETIVO
# --------
# Construir, exclusivamente a partir de NSRDB 1998–2024,
# los objetivos climatológicos que podrán utilizarse en el
# postproceso del año sintético 2050.
#
# No se modifica todavía df_2050_raw.
#
# Se definen:
#
#   1. energía mensual objetivo de GHI;
#   2. diffuse energy ratio mensual histórico;
#   3. factores GHI requeridos por mes;
#   4. diagnóstico de cuánto ajuste sería necesario.
#
# PRINCIPIO METODOLÓGICO
# ----------------------
# Los objetivos provienen del histórico y NO de una búsqueda
# destinada a minimizar retrospectivamente el error de 2050.
#
# La salida PI-LSTM RAW permanecerá conservada por separado.
# ============================================================


# ------------------------------------------------------------
# 1. Diffuse energy ratio histórico por año y mes
#
# Para cada año y mes:
#
#       rd_E = E_DHI / E_GHI
#
# Después obtenemos la climatología 1998–2024 de cada mes.
# ------------------------------------------------------------

hist_month_year_ratio = hist_month_year_energy.copy()

hist_month_year_ratio["rd_energy"] = (
    hist_month_year_ratio["DHI_energy"]
    / hist_month_year_ratio["GHI_energy"]
)

hist_month_ratio_clim = (
    hist_month_year_ratio
    .groupby("month")
    .agg(
        rd_hist_mean=("rd_energy", "mean"),
        rd_hist_median=("rd_energy", "median"),
        rd_hist_p05=("rd_energy", lambda x: x.quantile(0.05)),
        rd_hist_p95=("rd_energy", lambda x: x.quantile(0.95)),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 2. Diffuse energy ratio mensual del RAW 2050
# ------------------------------------------------------------

raw_month_ratio = raw_month_energy.copy()

raw_month_ratio["rd_raw"] = (
    raw_month_ratio["DHI_2050"]
    / raw_month_ratio["GHI_2050"]
)

monthly_targets_2050 = (
    monthly_compare_2050[
        [
            "month",
            "GHI_2050",
            "GHI_hist_mean",
            "GHI_hist_p05",
            "GHI_hist_p95",
        ]
    ]
    .merge(
        raw_month_ratio[["month", "rd_raw"]],
        on="month",
        validate="one_to_one"
    )
    .merge(
        hist_month_ratio_clim,
        on="month",
        validate="one_to_one"
    )
)


# ------------------------------------------------------------
# 3. Factor mensual requerido para GHI
#
# Si quisiéramos igualar exactamente la media climatológica:
#
#       f_GHI = E_GHI,hist / E_GHI,raw
#
# Todavía NO lo aplicamos.
# ------------------------------------------------------------

monthly_targets_2050["ghi_factor_exact"] = (
    monthly_targets_2050["GHI_hist_mean"]
    / monthly_targets_2050["GHI_2050"]
)


# ------------------------------------------------------------
# 4. Magnitud del ajuste difuso requerido
#
# No lo utilizamos directamente como multiplicador de DHI.
#
# Solo cuantificamos:
#
#       Δrd = rd_hist - rd_raw
#
# porque la futura reconciliación deberá mantener:
#
#       GHI = DNI*cos(z) + DHI
# ------------------------------------------------------------

monthly_targets_2050["rd_delta"] = (
    monthly_targets_2050["rd_hist_mean"]
    - monthly_targets_2050["rd_raw"]
)


# ------------------------------------------------------------
# 5. Diagnóstico mensual
# ------------------------------------------------------------

print("=== OBJETIVOS CLIMATOLÓGICOS MENSUALES ===")

print(
    monthly_targets_2050[
        [
            "month",
            "GHI_2050",
            "GHI_hist_mean",
            "ghi_factor_exact",
            "rd_raw",
            "rd_hist_mean",
            "rd_delta",
            "rd_hist_p05",
            "rd_hist_p95",
        ]
    ]
    .round(5)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Rango de factores GHI requeridos
# ------------------------------------------------------------

ghi_factor_min = float(
    monthly_targets_2050["ghi_factor_exact"].min()
)

ghi_factor_max = float(
    monthly_targets_2050["ghi_factor_exact"].max()
)

print("\n=== MAGNITUD DEL AJUSTE GHI ===")
print("Factor mínimo:", round(ghi_factor_min, 5))
print("Factor máximo:", round(ghi_factor_max, 5))


# ------------------------------------------------------------
# 7. Evaluar si el factor requerido es razonablemente acotado
#
# Se establece una banda metodológica previa:
#
#       0.85 <= f_GHI <= 1.15
#
# No significa que todos los meses deban ajustarse.
# Permite detectar si igualar la climatología exigiría una
# modificación excesiva de la señal generada por el modelo.
# ------------------------------------------------------------

GHI_FACTOR_LOWER = 0.85
GHI_FACTOR_UPPER = 1.15

monthly_targets_2050["ghi_factor_inside_bounds"] = (
    monthly_targets_2050["ghi_factor_exact"]
    .between(GHI_FACTOR_LOWER, GHI_FACTOR_UPPER)
)

print(
    "\nMeses cuyo factor GHI exacto cae en [0.85, 1.15]:",
    int(monthly_targets_2050["ghi_factor_inside_bounds"].sum()),
    "/ 12"
)

outside_ghi_factor = monthly_targets_2050.loc[
    ~monthly_targets_2050["ghi_factor_inside_bounds"],
    ["month", "ghi_factor_exact"]
]

if len(outside_ghi_factor):
    print("\nFactores GHI fuera de la banda:")
    print(outside_ghi_factor.round(5).to_string(index=False))
else:
    print("Todos los factores GHI están dentro de la banda.")


# ------------------------------------------------------------
# 8. Diagnóstico del objetivo difuso
#
# Determinamos qué meses RAW están fuera de la distribución
# histórica P05–P95 del diffuse energy ratio.
# ------------------------------------------------------------

monthly_targets_2050["rd_raw_inside_P05_P95"] = (
    (monthly_targets_2050["rd_raw"] >= monthly_targets_2050["rd_hist_p05"])
    &
    (monthly_targets_2050["rd_raw"] <= monthly_targets_2050["rd_hist_p95"])
)

outside_rd_months = monthly_targets_2050.loc[
    ~monthly_targets_2050["rd_raw_inside_P05_P95"],
    "month"
].tolist()

print("\n=== DIFFUSE ENERGY RATIO ===")
print(
    "Meses RAW fuera del P05–P95 histórico:",
    outside_rd_months if outside_rd_months else "ninguno"
)

print(
    "Máximo |Δrd| requerido:",
    round(float(monthly_targets_2050["rd_delta"].abs().max()), 6)
)


# ------------------------------------------------------------
# 9. Comparación anual de objetivos
# ------------------------------------------------------------

target_annual_ghi = float(
    hist_annual_energy_2050["GHI_energy"].mean()
)

target_annual_rd = float(
    hist_rd_energy_year.mean()
)

print("\n=== OBJETIVOS ANUALES DE REFERENCIA ===")
print(
    f"GHI histórico medio : {target_annual_ghi:.3f} kWh/m²"
)
print(
    f"GHI RAW 2050        : {raw_annual_energy_2050['GHI']:.3f} kWh/m²"
)
print(
    f"rd_E histórico medio: {target_annual_rd:.6f}"
)
print(
    f"rd_E RAW 2050       : {raw_rd_energy:.6f}"
)


# ------------------------------------------------------------
# 10. Guardar tabla de objetivos
# ------------------------------------------------------------

POSTPROCESS_TARGETS_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "postprocess_climatological_targets_2050.csv"
)

monthly_targets_2050.to_csv(
    POSTPROCESS_TARGETS_PATH,
    index=False
)

print("\nArchivo:")
print(POSTPROCESS_TARGETS_PATH.name)


# ------------------------------------------------------------
# 11. Cierre
# ------------------------------------------------------------

print("\n✓ Objetivos derivados únicamente de NSRDB 1998–2024.")
print("✓ Objetivo mensual GHI definido.")
print("✓ Objetivo mensual de diffuse energy ratio definido.")
print("✓ Magnitud de ajustes cuantificada.")
print("✓ Ninguna modificación aplicada todavía al RAW.")
print("✓ Celda 11 completada correctamente.")

=== OBJETIVOS CLIMATOLÓGICOS MENSUALES ===
 month  GHI_2050  GHI_hist_mean  ghi_factor_exact  rd_raw  rd_hist_mean  rd_delta  rd_hist_p05  rd_hist_p95
     1 181.38736      170.19480           0.93829 0.48627       0.43451  -0.05176      0.33554      0.51798
     2 168.95365      151.09574           0.89430 0.50889       0.50540  -0.00349      0.41110      0.57908
     3 190.60170      168.27430           0.88286 0.51043       0.50470  -0.00573      0.45164      0.56225
     4 172.45865      167.09757           0.96891 0.54387       0.44647  -0.09740      0.40129      0.49730
     5 166.28490      168.69750           1.01451 0.55201       0.41359  -0.13842      0.35710      0.46004
     6 164.24998      160.60211           0.97779 0.49609       0.39926  -0.09683      0.31381      0.45178
     7 169.82553      173.81646           1.02350 0.52719       0.37226  -0.15493      0.29899      0.43850
     8 179.99438      182.71159           1.01510 0.50274       0.37145  -0.13129      0.2841

In [12]:
# ============================================================
# 12. POSTPROCESO CLIMATOLÓGICO MÍNIMO — CANDIDATO 2050
# ============================================================
#
# OBJETIVO
# --------
# Construir una versión postprocesada mediante el principio de
# mínima intervención:
#
#   1. Si la energía mensual GHI RAW está dentro del intervalo
#      histórico P05–P95 -> NO se modifica.
#
#   2. Si está fuera -> se lleva únicamente al límite histórico
#      más cercano (P05 o P95).
#
#   3. Para la partición difusa se aplica el mismo criterio al
#      diffuse energy ratio mensual:
#
#             rd,E = E_DHI / E_GHI
#
#   4. La distribución temporal intramensual de rd se conserva
#      mediante una transformación monótona en espacio logit.
#
#   5. Finalmente:
#
#          DHI = rd * GHI
#          DNI = (GHI - DHI) / cos(z)
#
#      garantizando:
#
#          GHI = DNI*cos(z) + DHI
#
# IMPORTANTE
# ----------
# - No se modifica la salida RAW almacenada.
# - No se emplea calibración isotónica.
# - No se fuerza ningún mes a la media histórica.
# - No existe un factor difuso seleccionado manualmente.
# ============================================================


# ------------------------------------------------------------
# 1. Copia independiente del RAW
# ------------------------------------------------------------

df_2050_post = df_2050_raw.copy()
df_2050_post["month"] = df_2050_post["timestamp"].dt.month

EPS_POST = 1e-8


# ------------------------------------------------------------
# 2. Objetivo GHI de mínima intervención
#
# Si E_RAW está dentro de P05-P95:
#       target = E_RAW
#
# Si E_RAW < P05:
#       target = P05
#
# Si E_RAW > P95:
#       target = P95
# ------------------------------------------------------------

monthly_targets_post = monthly_targets_2050.copy()

monthly_targets_post["GHI_target"] = np.clip(
    monthly_targets_post["GHI_2050"],
    monthly_targets_post["GHI_hist_p05"],
    monthly_targets_post["GHI_hist_p95"]
)

monthly_targets_post["GHI_factor_minimal"] = (
    monthly_targets_post["GHI_target"]
    / monthly_targets_post["GHI_2050"]
)

monthly_targets_post["GHI_adjusted"] = (
    ~np.isclose(
        monthly_targets_post["GHI_factor_minimal"],
        1.0,
        rtol=0.0,
        atol=1e-12
    )
)


# ------------------------------------------------------------
# 3. Objetivo diffuse energy ratio de mínima intervención
#
# Exactamente el mismo principio:
#
#     target_rd = clip(rd_RAW, P05, P95)
# ------------------------------------------------------------

monthly_targets_post["rd_target"] = np.clip(
    monthly_targets_post["rd_raw"],
    monthly_targets_post["rd_hist_p05"],
    monthly_targets_post["rd_hist_p95"]
)

monthly_targets_post["rd_adjusted"] = (
    ~np.isclose(
        monthly_targets_post["rd_target"],
        monthly_targets_post["rd_raw"],
        rtol=0.0,
        atol=1e-12
    )
)


print("=== OBJETIVOS DE MÍNIMA INTERVENCIÓN ===")

print(
    monthly_targets_post[
        [
            "month",
            "GHI_2050",
            "GHI_hist_p05",
            "GHI_hist_p95",
            "GHI_target",
            "GHI_factor_minimal",
            "GHI_adjusted",
            "rd_raw",
            "rd_hist_p05",
            "rd_hist_p95",
            "rd_target",
            "rd_adjusted",
        ]
    ]
    .round(5)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 4. Aplicar ajuste GHI mensual
#
# El factor es uniforme dentro del mes; así se conserva
# exactamente la estructura temporal producida por el modelo.
# ------------------------------------------------------------

ghi_factor_map = (
    monthly_targets_post
    .set_index("month")["GHI_factor_minimal"]
    .to_dict()
)

df_2050_post["GHI_factor"] = (
    df_2050_post["month"]
    .map(ghi_factor_map)
    .astype(float)
)

df_2050_post["GHI_post"] = (
    df_2050_post["GHI_raw"]
    * df_2050_post["GHI_factor"]
)

# Noche exactamente cero
night_post = df_2050_post["cosz_plus"] <= 0.0
df_2050_post.loc[night_post, "GHI_post"] = 0.0

assert (df_2050_post["GHI_post"] >= 0.0).all()
assert np.isfinite(df_2050_post["GHI_post"]).all()


# ------------------------------------------------------------
# 5. Ratio difuso puntual RAW
#
# Debido al cierre físico del RAW:
#
#       0 <= DHI/GHI <= 1
#
# salvo pequeños errores numéricos.
# ------------------------------------------------------------

rd_point_raw = np.zeros(len(df_2050_post), dtype=float)

positive_ghi = (
    df_2050_post["GHI_raw"].to_numpy(dtype=float) > EPS_POST
)

rd_point_raw[positive_ghi] = (
    df_2050_post.loc[positive_ghi, "DHI_raw"].to_numpy(dtype=float)
    /
    df_2050_post.loc[positive_ghi, "GHI_raw"].to_numpy(dtype=float)
)

rd_point_raw = np.clip(rd_point_raw, 0.0, 1.0)

df_2050_post["rd_raw_point"] = rd_point_raw


# ------------------------------------------------------------
# 6. Transformación monotónica del diffuse ratio
#
# Queremos modificar el nivel mensual de rd sin destruir su
# orden temporal.
#
# Se utiliza:
#
#       rd* = sigmoid(logit(rd) + delta)
#
# donde delta se determina automáticamente para que:
#
#       sum(rd* · GHI) / sum(GHI) = rd_target
#
# No hay factor ajustado manualmente.
# ------------------------------------------------------------

def sigmoid_post(x):
    return 1.0 / (1.0 + np.exp(-x))


def logit_post(p):
    p = np.clip(p, 1e-6, 1.0 - 1e-6)
    return np.log(p / (1.0 - p))


def solve_logit_shift(rd, weights, target, max_iter=100):
    """
    Encuentra delta mediante bisección para satisfacer el
    diffuse energy ratio objetivo conservando el orden de rd.
    """

    rd = np.asarray(rd, dtype=float)
    weights = np.asarray(weights, dtype=float)

    valid = weights > 0.0
    if not valid.any():
        return 0.0

    r = np.clip(rd[valid], 1e-6, 1.0 - 1e-6)
    w = weights[valid]
    z = logit_post(r)

    def weighted_ratio(delta):
        rr = sigmoid_post(z + delta)
        return np.sum(rr * w) / np.sum(w)

    current = weighted_ratio(0.0)

    if np.isclose(current, target, atol=1e-12):
        return 0.0

    lo, hi = -20.0, 20.0

    # El intervalo es suficientemente amplio para cubrir
    # prácticamente todo target dentro de (0,1).
    if not (weighted_ratio(lo) <= target <= weighted_ratio(hi)):
        raise RuntimeError(
            f"No fue posible encerrar target rd={target:.6f}."
        )

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)

        if weighted_ratio(mid) < target:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)


# ------------------------------------------------------------
# 7. Resolver ajuste difuso mes por mes
# ------------------------------------------------------------

df_2050_post["rd_post"] = df_2050_post["rd_raw_point"].copy()

rd_shift_rows = []

for _, target_row in monthly_targets_post.iterrows():

    month = int(target_row["month"])
    target_rd = float(target_row["rd_target"])

    mask_month = df_2050_post["month"] == month

    rd_month = df_2050_post.loc[
        mask_month, "rd_raw_point"
    ].to_numpy(dtype=float)

    ghi_month = df_2050_post.loc[
        mask_month, "GHI_post"
    ].to_numpy(dtype=float)

    # Solo irradiancia positiva aporta energía.
    delta = solve_logit_shift(
        rd_month,
        ghi_month,
        target_rd
    )

    positive = ghi_month > EPS_POST
    rd_new = np.zeros(len(rd_month), dtype=float)

    if positive.any():
        rd_new[positive] = sigmoid_post(
            logit_post(rd_month[positive]) + delta
        )

    rd_new = np.clip(rd_new, 0.0, 1.0)

    df_2050_post.loc[mask_month, "rd_post"] = rd_new

    achieved = (
        np.sum(rd_new * ghi_month)
        / np.sum(ghi_month)
        if np.sum(ghi_month) > 0 else 0.0
    )

    rd_shift_rows.append({
        "month": month,
        "rd_raw": float(target_row["rd_raw"]),
        "rd_target": target_rd,
        "logit_shift": delta,
        "rd_achieved": achieved,
        "abs_target_error": abs(achieved - target_rd),
    })


rd_shift_summary_2050 = pd.DataFrame(rd_shift_rows)

print("\n=== AJUSTE DIFUSO MONÓTONO ===")
print(rd_shift_summary_2050.round(7).to_string(index=False))

assert (
    rd_shift_summary_2050["abs_target_error"].max()
    < 1e-8
)


# ------------------------------------------------------------
# 8. Reconstrucción física DNI/DHI
# ------------------------------------------------------------

ghi_post = df_2050_post["GHI_post"].to_numpy(dtype=float)
rd_post = df_2050_post["rd_post"].to_numpy(dtype=float)
cosz_post = df_2050_post["cosz_plus"].to_numpy(dtype=float)

dhi_post = rd_post * ghi_post
dni_post = np.zeros(len(df_2050_post), dtype=float)

day_positive_cosz = cosz_post > 0.0

dni_post[day_positive_cosz] = (
    ghi_post[day_positive_cosz]
    - dhi_post[day_positive_cosz]
) / cosz_post[day_positive_cosz]

# Protección únicamente contra ruido numérico
dni_post = np.maximum(dni_post, 0.0)
dhi_post = np.maximum(dhi_post, 0.0)

dni_post[night_post] = 0.0
dhi_post[night_post] = 0.0

df_2050_post["DNI_post"] = dni_post
df_2050_post["DHI_post"] = dhi_post


# ------------------------------------------------------------
# 9. Auditoría inmediata de cierre
# ------------------------------------------------------------

closure_post = (
    df_2050_post["GHI_post"].to_numpy(dtype=float)
    -
    (
        df_2050_post["DNI_post"].to_numpy(dtype=float)
        * df_2050_post["cosz_plus"].to_numpy(dtype=float)
        +
        df_2050_post["DHI_post"].to_numpy(dtype=float)
    )
)

closure_post_mae = float(np.mean(np.abs(closure_post)))
closure_post_max = float(np.max(np.abs(closure_post)))

print("\n=== CIERRE FÍSICO POST ===")
print("MAE     :", closure_post_mae)
print("Max abs :", closure_post_max)

assert closure_post_max < 1e-6


# ------------------------------------------------------------
# 10. Auditoría de DNI respecto del dominio físico del modelo
#
# Todavía NO aplicamos cap.
#
# Queremos saber si la redistribución difusa produce valores
# DNI que excedan:
#
#       kDNI = DNI / CS_DNI > 1.20
#
# Si ocurre, se estudiará explícitamente en la Celda 13.
# ------------------------------------------------------------

csdni_post = df_2050_post[COL_CSDNI].to_numpy(dtype=float)

kDNI_post = np.zeros(len(df_2050_post), dtype=float)
valid_csdni_post = csdni_post > cfg2050.thr_cs

kDNI_post[valid_csdni_post] = (
    dni_post[valid_csdni_post]
    / csdni_post[valid_csdni_post]
)

df_2050_post["kDNI_post"] = kDNI_post

n_kdni_over_cap = int(
    (df_2050_post["kDNI_post"] > cfg2050.kdni_max).sum()
)

max_kdni_post = float(
    df_2050_post["kDNI_post"].max()
)

print("\n=== DIAGNÓSTICO DNI POST ===")
print("kDNI post máximo :", max_kdni_post)
print(
    f"Registros kDNI > {cfg2050.kdni_max}:",
    n_kdni_over_cap
)


# ------------------------------------------------------------
# 11. Comprobar objetivos energéticos alcanzados
# ------------------------------------------------------------

post_month_energy = (
    df_2050_post
    .groupby("month")
    .agg(
        GHI_post=("GHI_post", lambda x: x.sum() * DT_HOURS / 1000.0),
        DNI_post=("DNI_post", lambda x: x.sum() * DT_HOURS / 1000.0),
        DHI_post=("DHI_post", lambda x: x.sum() * DT_HOURS / 1000.0),
    )
    .reset_index()
)

post_month_energy["rd_post"] = (
    post_month_energy["DHI_post"]
    / post_month_energy["GHI_post"]
)

post_check_2050 = (
    monthly_targets_post[
        ["month", "GHI_target", "rd_target"]
    ]
    .merge(
        post_month_energy,
        on="month",
        validate="one_to_one"
    )
)

post_check_2050["GHI_target_error"] = (
    post_check_2050["GHI_post"]
    - post_check_2050["GHI_target"]
)

post_check_2050["rd_target_error"] = (
    post_check_2050["rd_post"]
    - post_check_2050["rd_target"]
)


print("\n=== OBJETIVOS ALCANZADOS ===")

print(
    post_check_2050[
        [
            "month",
            "GHI_target",
            "GHI_post",
            "GHI_target_error",
            "rd_target",
            "rd_post",
            "rd_target_error",
        ]
    ]
    .round(7)
    .to_string(index=False)
)

assert post_check_2050["GHI_target_error"].abs().max() < 1e-6
assert post_check_2050["rd_target_error"].abs().max() < 1e-8


# ------------------------------------------------------------
# 12. Energía anual preliminar POST
# ------------------------------------------------------------

post_annual_energy_2050 = {
    "GHI": df_2050_post["GHI_post"].sum() * DT_HOURS / 1000.0,
    "DNI": df_2050_post["DNI_post"].sum() * DT_HOURS / 1000.0,
    "DHI": df_2050_post["DHI_post"].sum() * DT_HOURS / 1000.0,
}

post_rd_energy_2050 = (
    post_annual_energy_2050["DHI"]
    / post_annual_energy_2050["GHI"]
)

print("\n=== ENERGÍA ANUAL PRELIMINAR ===")

for var in ["GHI", "DNI", "DHI"]:
    print(
        f"{var}: RAW={raw_annual_energy_2050[var]:.3f} "
        f"| POST={post_annual_energy_2050[var]:.3f} kWh/m²"
    )

print(
    f"\nrd_E RAW : {raw_rd_energy:.6f}"
)
print(
    f"rd_E POST: {post_rd_energy_2050:.6f}"
)
print(
    f"rd_E HIST: {hist_rd_energy_year.mean():.6f}"
)


# ------------------------------------------------------------
# 13. Guardar candidato postprocesado
#
# Se denomina CANDIDATE porque aún falta la auditoría completa
# de la Celda 13 antes de declararlo producto final.
# ------------------------------------------------------------

POST_CANDIDATE_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "PI_LSTM_V24V1_2050_POST_CANDIDATE.csv"
)

POST_TARGET_CHECK_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "postprocess_target_check_2050.csv"
)

POST_RD_SHIFT_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "postprocess_rd_logit_shifts_2050.csv"
)

df_2050_post.to_csv(
    POST_CANDIDATE_PATH,
    index=False
)

post_check_2050.to_csv(
    POST_TARGET_CHECK_PATH,
    index=False
)

rd_shift_summary_2050.to_csv(
    POST_RD_SHIFT_PATH,
    index=False
)


# ------------------------------------------------------------
# 14. Resumen
# ------------------------------------------------------------

print("\n=== ARCHIVOS GENERADOS ===")
print(POST_CANDIDATE_PATH.name)
print(POST_TARGET_CHECK_PATH.name)
print(POST_RD_SHIFT_PATH.name)

print("\n✓ GHI corregido solo cuando estaba fuera de P05–P95.")
print("✓ rd_E corregido solo cuando estaba fuera de P05–P95.")
print("✓ No se utilizó factor difuso manual.")
print("✓ Orden temporal del diffuse ratio preservado.")
print("✓ Cierre radiativo exacto.")
print("✓ RAW conservado íntegramente.")
print("✓ Candidato POST generado.")
print("✓ Celda 12 completada correctamente.")

=== OBJETIVOS DE MÍNIMA INTERVENCIÓN ===
 month  GHI_2050  GHI_hist_p05  GHI_hist_p95  GHI_target  GHI_factor_minimal  GHI_adjusted  rd_raw  rd_hist_p05  rd_hist_p95  rd_target  rd_adjusted
     1 181.38736     155.17235     189.01330   181.38736             1.00000         False 0.48627      0.33554      0.51798    0.48627        False
     2 168.95365     133.46435     169.24215   168.95365             1.00000         False 0.50889      0.41110      0.57908    0.50889        False
     3 190.60170     154.85585     180.49440   180.49440             0.94697          True 0.51043      0.45164      0.56225    0.51043        False
     4 172.45865     156.86925     178.73920   172.45865             1.00000         False 0.54387      0.40129      0.49730    0.49730         True
     5 166.28490     160.06755     180.57855   166.28490             1.00000         False 0.55201      0.35710      0.46004    0.46004         True
     6 164.24998     150.50360     174.76985   164.24998         

In [14]:
# ============================================================
# 13. AUDITORÍA FINAL DEL CANDIDATO POST
#     RAW vs POST vs CLIMATOLOGÍA HISTÓRICA
# ============================================================

# Tolerancias numéricas para comparaciones en límites P05–P95
TOL_ENERGY = 1e-6
TOL_RATIO = 1e-8

# ------------------------------------------------------------
# 1. Preparar columnas auxiliares
# ------------------------------------------------------------

df_post_audit = df_2050_post.copy()
df_post_audit["month"] = df_post_audit["timestamp"].dt.month

irr_raw_cols = ["GHI_raw", "DNI_raw", "DHI_raw"]
irr_post_cols = ["GHI_post", "DNI_post", "DHI_post"]

assert len(df_post_audit) == 17520
assert not df_post_audit[irr_post_cols].isna().any().any()

# ------------------------------------------------------------
# 2. Energía anual RAW vs POST vs histórico
# ------------------------------------------------------------

annual_final_rows = []

for var in ["GHI", "DNI", "DHI"]:
    row_hist = annual_compare_2050.loc[annual_compare_2050["variable"] == var].iloc[0]

    hist_mean = float(row_hist["hist_mean_kWh_m2"])
    hist_p05 = float(row_hist["hist_p05_kWh_m2"])
    hist_p95 = float(row_hist["hist_p95_kWh_m2"])
    raw_e = float(raw_annual_energy_2050[var])
    post_e = float(post_annual_energy_2050[var])

    annual_final_rows.append({
        "variable": var,
        "hist_mean": hist_mean,
        "hist_p05": hist_p05,
        "hist_p95": hist_p95,
        "RAW_2050": raw_e,
        "POST_2050": post_e,
        "RAW_error_pct": 100.0 * (raw_e / hist_mean - 1.0),
        "POST_error_pct": 100.0 * (post_e / hist_mean - 1.0),
        "POST_inside_P05_P95": (hist_p05 - TOL_ENERGY) <= post_e <= (hist_p95 + TOL_ENERGY),
    })

annual_final_compare = pd.DataFrame(annual_final_rows)

print("=== ENERGÍA ANUAL FINAL ===")
print(annual_final_compare.round(3).to_string(index=False))

# ------------------------------------------------------------
# 3. Energía mensual POST y comparación histórica
# ------------------------------------------------------------

monthly_final = post_month_energy.merge(hist_month_clim, on="month", validate="one_to_one")

for var in ["GHI", "DNI", "DHI"]:
    monthly_final[f"{var}_POST_error_pct"] = 100.0 * (
        monthly_final[f"{var}_post"] / monthly_final[f"{var}_hist_mean"] - 1.0
    )

    monthly_final[f"{var}_POST_inside_P05_P95"] = (
        (monthly_final[f"{var}_post"] >= monthly_final[f"{var}_hist_p05"] - TOL_ENERGY) &
        (monthly_final[f"{var}_post"] <= monthly_final[f"{var}_hist_p95"] + TOL_ENERGY)
    )

print("\n=== ENERGÍA MENSUAL POST vs HISTÓRICO ===")

show_month_cols = [
    "month",
    "GHI_post", "GHI_hist_mean", "GHI_POST_error_pct",
    "DNI_post", "DNI_hist_mean", "DNI_POST_error_pct",
    "DHI_post", "DHI_hist_mean", "DHI_POST_error_pct",
]

print(monthly_final[show_month_cols].round(3).to_string(index=False))

# ------------------------------------------------------------
# 4. Resumen mensual RAW vs POST
# ------------------------------------------------------------

monthly_final_summary_rows = []

for var in ["GHI", "DNI", "DHI"]:
    raw_err = monthly_compare_2050[f"{var}_error_mean_pct"]
    post_err = monthly_final[f"{var}_POST_error_pct"]

    monthly_final_summary_rows.append({
        "variable": var,
        "RAW_mean_bias_pct": raw_err.mean(),
        "POST_mean_bias_pct": post_err.mean(),
        "RAW_MAE_pct": raw_err.abs().mean(),
        "POST_MAE_pct": post_err.abs().mean(),
        "RAW_max_abs_pct": raw_err.abs().max(),
        "POST_max_abs_pct": post_err.abs().max(),
        "RAW_months_inside_P05_P95": int(monthly_compare_2050[f"{var}_inside_P05_P95"].sum()),
        "POST_months_inside_P05_P95": int(monthly_final[f"{var}_POST_inside_P05_P95"].sum()),
    })

monthly_final_summary = pd.DataFrame(monthly_final_summary_rows)

print("\n=== RESUMEN MENSUAL RAW vs POST ===")
print(monthly_final_summary.round(3).to_string(index=False))

# ------------------------------------------------------------
# 5. Diffuse energy ratio anual
# ------------------------------------------------------------

hist_rd_mean = float(hist_rd_energy_year.mean())
hist_rd_p05 = float(hist_rd_energy_year.quantile(0.05))
hist_rd_p95 = float(hist_rd_energy_year.quantile(0.95))

ratio_final_summary = pd.DataFrame([{
    "hist_mean": hist_rd_mean,
    "hist_p05": hist_rd_p05,
    "hist_p95": hist_rd_p95,
    "RAW_2050": raw_rd_energy,
    "POST_2050": post_rd_energy_2050,
    "RAW_error": raw_rd_energy - hist_rd_mean,
    "POST_error": post_rd_energy_2050 - hist_rd_mean,
    "POST_inside_P05_P95":
        (hist_rd_p05 - TOL_RATIO) <= post_rd_energy_2050 <= (hist_rd_p95 + TOL_RATIO),
}])

print("\n=== DIFFUSE ENERGY RATIO ANUAL ===")
print(ratio_final_summary.round(6).to_string(index=False))

# ------------------------------------------------------------
# 6. Diffuse energy ratio mensual final
# ------------------------------------------------------------

monthly_rd_final = (
    post_month_energy[["month", "rd_post"]]
    .merge(hist_month_ratio_clim, on="month", validate="one_to_one")
)

monthly_rd_final["inside_P05_P95"] = (
    (monthly_rd_final["rd_post"] >= monthly_rd_final["rd_hist_p05"] - TOL_RATIO) &
    (monthly_rd_final["rd_post"] <= monthly_rd_final["rd_hist_p95"] + TOL_RATIO)
)

print("\n=== DIFFUSE RATIO MENSUAL POST ===")
print(monthly_rd_final.round(6).to_string(index=False))

# ------------------------------------------------------------
# 7. Régimen solar RAW vs POST
# ------------------------------------------------------------

post_day_regime = df_post_audit[
    df_post_audit["cosz_plus"] > cfg2050.thr_day_cosz
].copy()

post_day_regime["cosz_band"] = pd.cut(
    post_day_regime["cosz_plus"],
    bins=cosz_bins,
    labels=cosz_labels,
    right=False
)

post_day_regime["rd_post_point"] = (
    post_day_regime["DHI_post"] /
    np.maximum(post_day_regime["GHI_post"], 1e-6)
)

cosz_post_summary = (
    post_day_regime
    .groupby("cosz_band", observed=True)
    .agg(
        N=("GHI_post", "size"),
        GHI_post_mean=("GHI_post", "mean"),
        DNI_post_mean=("DNI_post", "mean"),
        DHI_post_mean=("DHI_post", "mean"),
        kDNI_post_mean=("kDNI_post", "mean"),
        rd_post_mean=("rd_post_point", "mean"),
    )
    .reset_index()
)

cosz_raw_vs_post = cosz_summary_2050.merge(
    cosz_post_summary,
    on=["cosz_band", "N"],
    validate="one_to_one"
)

print("\n=== RÉGIMEN SOLAR RAW vs POST ===")

print(
    cosz_raw_vs_post[
        [
            "cosz_band", "N",
            "GHI_mean", "GHI_post_mean",
            "DNI_mean", "DNI_post_mean",
            "DHI_mean", "DHI_post_mean",
            "rd_mean", "rd_post_mean",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. Extremos físicos
# ------------------------------------------------------------

physical_extremes = pd.DataFrame([
    {
        "variable": "GHI",
        "RAW_max": df_post_audit["GHI_raw"].max(),
        "POST_max": df_post_audit["GHI_post"].max(),
        "CS_max": df_post_audit[COL_CSGHI].max(),
    },
    {
        "variable": "DNI",
        "RAW_max": df_post_audit["DNI_raw"].max(),
        "POST_max": df_post_audit["DNI_post"].max(),
        "CS_max": df_post_audit[COL_CSDNI].max(),
    },
    {
        "variable": "DHI",
        "RAW_max": df_post_audit["DHI_raw"].max(),
        "POST_max": df_post_audit["DHI_post"].max(),
        "CS_max": df_post_audit[COL_CSDHI].max(),
    },
])

print("\n=== EXTREMOS FÍSICOS ===")
print(physical_extremes.round(3).to_string(index=False))

# ------------------------------------------------------------
# 9. kDNI POST
# ------------------------------------------------------------

day_kdni_post = df_post_audit.loc[
    df_post_audit["cosz_plus"] > cfg2050.thr_day_cosz,
    "kDNI_post"
]

kdni_post_summary = {
    "mean": float(day_kdni_post.mean()),
    "p95": float(day_kdni_post.quantile(0.95)),
    "p99": float(day_kdni_post.quantile(0.99)),
    "max": float(day_kdni_post.max()),
    "count_over_1": int((day_kdni_post > 1.0).sum()),
    "count_over_1_2": int((day_kdni_post > cfg2050.kdni_max).sum()),
}

print("\n=== kDNI POST DAYTIME ===")
for k, v in kdni_post_summary.items():
    print(f"{k:16s}: {v}")

# ------------------------------------------------------------
# 10. Continuidad temporal
# ------------------------------------------------------------

post_duplicates = int(df_post_audit["timestamp"].duplicated().sum())

post_dt = (
    df_post_audit["timestamp"]
    .diff()
    .dropna()
    .dt.total_seconds()
    / 60.0
)

post_bad_steps = int((post_dt != 30.0).sum())

print("\n=== CONTINUIDAD TEMPORAL ===")
print("Duplicados:", post_duplicates)
print("Saltos != 30 min:", post_bad_steps)

assert post_duplicates == 0
assert post_bad_steps == 0

# ------------------------------------------------------------
# 11. Cierre radiativo final
# ------------------------------------------------------------

closure_final = (
    df_post_audit["GHI_post"].to_numpy(dtype=float)
    - (
        df_post_audit["DNI_post"].to_numpy(dtype=float)
        * df_post_audit["cosz_plus"].to_numpy(dtype=float)
        + df_post_audit["DHI_post"].to_numpy(dtype=float)
    )
)

closure_final_mae = float(np.mean(np.abs(closure_final)))
closure_final_rmse = float(np.sqrt(np.mean(closure_final**2)))
closure_final_max = float(np.max(np.abs(closure_final)))

print("\n=== CIERRE FINAL ===")
print("MAE     :", closure_final_mae)
print("RMSE    :", closure_final_rmse)
print("Max abs :", closure_final_max)

assert closure_final_max < 1e-6

# ------------------------------------------------------------
# 12. Leakage nocturno final
# ------------------------------------------------------------

night_final = df_post_audit["cosz_plus"] <= 0.0
night_values_final = df_post_audit.loc[night_final, irr_post_cols].to_numpy(dtype=float)
night_leakage_final = int((night_values_final != 0.0).sum())

print("\n=== NOCHE ===")
print("Night leakage:", night_leakage_final)

assert night_leakage_final == 0

# ------------------------------------------------------------
# 13. Negativos / NaN / Inf
# ------------------------------------------------------------

post_values = df_post_audit[irr_post_cols].to_numpy(dtype=float)

negative_count_post = int((post_values < 0.0).sum())
nan_count_post = int(np.isnan(post_values).sum())
inf_count_post = int(np.isinf(post_values).sum())

print("\n=== INTEGRIDAD NUMÉRICA ===")
print("Negativos:", negative_count_post)
print("NaN      :", nan_count_post)
print("Inf      :", inf_count_post)

assert negative_count_post == 0
assert nan_count_post == 0
assert inf_count_post == 0

# ------------------------------------------------------------
# 14. Criterios de aceptación del candidato
#
# No se exige coincidencia exacta con la media histórica.
# Se exige pertenencia a P05–P95 y consistencia física.
# Las comparaciones en frontera usan tolerancia numérica.
# ------------------------------------------------------------

acceptance_checks = {
    "GHI_annual_inside_P05_P95":
        bool(annual_final_compare.loc[
            annual_final_compare["variable"] == "GHI",
            "POST_inside_P05_P95"
        ].iloc[0]),

    "DNI_annual_inside_P05_P95":
        bool(annual_final_compare.loc[
            annual_final_compare["variable"] == "DNI",
            "POST_inside_P05_P95"
        ].iloc[0]),

    "DHI_annual_inside_P05_P95":
        bool(annual_final_compare.loc[
            annual_final_compare["variable"] == "DHI",
            "POST_inside_P05_P95"
        ].iloc[0]),

    "diffuse_ratio_annual_inside_P05_P95":
        bool(ratio_final_summary["POST_inside_P05_P95"].iloc[0]),

    "all_monthly_GHI_inside_P05_P95":
        bool(monthly_final["GHI_POST_inside_P05_P95"].all()),

    "all_monthly_rd_inside_P05_P95":
        bool(monthly_rd_final["inside_P05_P95"].all()),

    "no_kDNI_over_cap":
        kdni_post_summary["count_over_1_2"] == 0,

    "closure_ok":
        closure_final_max < 1e-6,

    "night_leakage_zero":
        night_leakage_final == 0,

    "temporal_continuity_ok":
        post_duplicates == 0 and post_bad_steps == 0,

    "numeric_integrity_ok":
        negative_count_post == 0
        and nan_count_post == 0
        and inf_count_post == 0,
}

candidate_accepted = all(acceptance_checks.values())

print("\n============================================================")
print(" CRITERIOS DE ACEPTACIÓN")
print("============================================================")

for key, value in acceptance_checks.items():
    print(f"{key:42s}: {value}")

print("\nCANDIDATO ACEPTADO:", candidate_accepted)

# ------------------------------------------------------------
# 15. Guardar resultados de auditoría
# ------------------------------------------------------------

FINAL_ANNUAL_COMPARE_PATH = OUTPUT_DIR_2050_V24V1 / "final_RAW_vs_POST_annual_energy.csv"
FINAL_MONTHLY_COMPARE_PATH = OUTPUT_DIR_2050_V24V1 / "final_POST_monthly_energy_audit.csv"
FINAL_RATIO_PATH = OUTPUT_DIR_2050_V24V1 / "final_POST_diffuse_ratio_audit.csv"
FINAL_COSZ_PATH = OUTPUT_DIR_2050_V24V1 / "final_RAW_vs_POST_cosz_audit.csv"
FINAL_ACCEPTANCE_PATH = OUTPUT_DIR_2050_V24V1 / "final_POST_acceptance.json"

annual_final_compare.to_csv(FINAL_ANNUAL_COMPARE_PATH, index=False)
monthly_final.to_csv(FINAL_MONTHLY_COMPARE_PATH, index=False)
monthly_rd_final.to_csv(FINAL_RATIO_PATH, index=False)
cosz_raw_vs_post.to_csv(FINAL_COSZ_PATH, index=False)

final_acceptance_payload = {
    "candidate_file": POST_CANDIDATE_PATH.name,
    "candidate_accepted": candidate_accepted,
    "checks": acceptance_checks,
    "tolerances": {
        "energy_kWh_m2": TOL_ENERGY,
        "diffuse_ratio": TOL_RATIO,
    },
    "annual_energy_kWh_m2": post_annual_energy_2050,
    "annual_diffuse_energy_ratio": post_rd_energy_2050,
    "closure_mae_Wm2": closure_final_mae,
    "closure_rmse_Wm2": closure_final_rmse,
    "closure_max_abs_Wm2": closure_final_max,
    "night_leakage_count": night_leakage_final,
    "kDNI_post_max": kdni_post_summary["max"],
}

with open(FINAL_ACCEPTANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(final_acceptance_payload, f, indent=2)

# ------------------------------------------------------------
# 16. Cierre
# ------------------------------------------------------------

print("\n=== ARCHIVOS GENERADOS ===")
print(FINAL_ANNUAL_COMPARE_PATH.name)
print(FINAL_MONTHLY_COMPARE_PATH.name)
print(FINAL_RATIO_PATH.name)
print(FINAL_COSZ_PATH.name)
print(FINAL_ACCEPTANCE_PATH.name)

print("\n✓ Auditoría RAW vs POST completada.")
print("✓ Energía anual y mensual verificada.")
print("✓ Diffuse ratio verificado.")
print("✓ Comparaciones P05–P95 evaluadas con tolerancia numérica.")
print("✓ Régimen solar auditado.")
print("✓ Continuidad temporal verificada.")
print("✓ Cierre radiativo verificado.")
print("✓ Integridad numérica verificada.")
print("✓ Celda 13 completada correctamente.")

=== ENERGÍA ANUAL FINAL ===
variable  hist_mean  hist_p05  hist_p95  RAW_2050  POST_2050  RAW_error_pct  POST_error_pct  POST_inside_P05_P95
     GHI   2058.867  1980.311  2149.708  2176.064   2153.537          5.692           4.598                False
     DNI   1694.191  1511.050  1922.558  1637.186   1694.174         -3.365          -0.001                 True
     DHI    858.395   808.969   902.380   994.570    934.417         15.864           8.856                False

=== ENERGÍA MENSUAL POST vs HISTÓRICO ===
 month  GHI_post  GHI_hist_mean  GHI_POST_error_pct  DNI_post  DNI_hist_mean  DNI_POST_error_pct  DHI_post  DHI_hist_mean  DHI_POST_error_pct
     1   181.387        170.195               6.576   128.827        138.800              -7.185    88.204         73.335              20.274
     2   168.954        151.096              11.819   108.636        102.081               6.421    85.978         75.874              13.316
     3   180.494        168.274               7.262

In [15]:
# ============================================================
# 14. RECONCILIACIÓN ANUAL GLOBAL DE MÍNIMA INTERVENCIÓN
# ============================================================
#
# OBJETIVO
# --------
# Resolver los dos incumplimientos anuales restantes del
# candidato POST sin modificar:
#
#   - la estructura temporal;
#   - el diffuse ratio puntual o mensual;
#   - la partición relativa DNI/DHI;
#   - el cierre radiativo.
#
# Se busca un único factor escalar global:
#
#       s
#
# tal que:
#
#       GHI_final = s * GHI_post
#       DNI_final = s * DNI_post
#       DHI_final = s * DHI_post
#
# y que satisfaga simultáneamente:
#
#   1. energía anual GHI dentro de P05-P95;
#   2. energía anual DNI dentro de P05-P95;
#   3. energía anual DHI dentro de P05-P95;
#   4. TODOS los GHI mensuales dentro de P05-P95.
#
# Entre todos los valores factibles se selecciona el más
# próximo a 1, es decir, la mínima intervención posible.
# ============================================================


# ------------------------------------------------------------
# 1. Construir restricciones anuales
#
# Para una energía E_POST:
#
#       P05 <= s*E_POST <= P95
#
# implica:
#
#       P05/E_POST <= s <= P95/E_POST
# ------------------------------------------------------------

scale_constraints = []

for var in ["GHI", "DNI", "DHI"]:

    row = annual_final_compare.loc[
        annual_final_compare["variable"] == var
    ].iloc[0]

    e_post = float(row["POST_2050"])
    p05 = float(row["hist_p05"])
    p95 = float(row["hist_p95"])

    scale_constraints.append({
        "constraint": f"{var}_annual",
        "lower": p05 / e_post,
        "upper": p95 / e_post,
    })


# ------------------------------------------------------------
# 2. Añadir restricciones mensuales GHI
# ------------------------------------------------------------

for _, row in monthly_final.iterrows():

    month = int(row["month"])
    e_post = float(row["GHI_post"])

    scale_constraints.append({
        "constraint": f"GHI_month_{month:02d}",
        "lower": float(row["GHI_hist_p05"]) / e_post,
        "upper": float(row["GHI_hist_p95"]) / e_post,
    })


constraints_df = pd.DataFrame(scale_constraints)

print("=== RESTRICCIONES PARA EL FACTOR GLOBAL ===")
print(
    constraints_df
    .round(8)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3. Intersección factible
# ------------------------------------------------------------

scale_lower = float(
    constraints_df["lower"].max()
)

scale_upper = float(
    constraints_df["upper"].min()
)

print("\n=== INTERVALO FACTIBLE GLOBAL ===")
print("Límite inferior:", scale_lower)
print("Límite superior:", scale_upper)

if scale_lower > scale_upper:
    raise RuntimeError(
        "No existe un factor global que satisfaga "
        "simultáneamente todas las restricciones."
    )


# ------------------------------------------------------------
# 4. Elegir mínima intervención
#
# Queremos el valor factible más próximo a 1.
# ------------------------------------------------------------

if scale_lower <= 1.0 <= scale_upper:
    GLOBAL_SCALE_FINAL = 1.0

elif scale_upper < 1.0:
    GLOBAL_SCALE_FINAL = scale_upper

else:
    GLOBAL_SCALE_FINAL = scale_lower


print("\nFactor global seleccionado:", GLOBAL_SCALE_FINAL)
print(
    "Corrección porcentual:",
    100.0 * (GLOBAL_SCALE_FINAL - 1.0),
    "%"
)


# ------------------------------------------------------------
# 5. Aplicar corrección global
# ------------------------------------------------------------

df_2050_final = df_2050_post.copy()

df_2050_final["GHI_final"] = (
    df_2050_final["GHI_post"]
    * GLOBAL_SCALE_FINAL
)

df_2050_final["DNI_final"] = (
    df_2050_final["DNI_post"]
    * GLOBAL_SCALE_FINAL
)

df_2050_final["DHI_final"] = (
    df_2050_final["DHI_post"]
    * GLOBAL_SCALE_FINAL
)


# Noche exactamente cero
night_final_mask = df_2050_final["cosz_plus"] <= 0.0

df_2050_final.loc[
    night_final_mask,
    ["GHI_final", "DNI_final", "DHI_final"]
] = 0.0


# ------------------------------------------------------------
# 6. Cierre radiativo
# ------------------------------------------------------------

closure_final2 = (
    df_2050_final["GHI_final"].to_numpy(dtype=float)
    -
    (
        df_2050_final["DNI_final"].to_numpy(dtype=float)
        * df_2050_final["cosz_plus"].to_numpy(dtype=float)
        +
        df_2050_final["DHI_final"].to_numpy(dtype=float)
    )
)

closure_final2_mae = float(
    np.mean(np.abs(closure_final2))
)

closure_final2_max = float(
    np.max(np.abs(closure_final2))
)

assert closure_final2_max < 1e-6


# ------------------------------------------------------------
# 7. Energía anual final
# ------------------------------------------------------------

final_annual_energy_2050 = {
    "GHI":
        df_2050_final["GHI_final"].sum()
        * DT_HOURS / 1000.0,

    "DNI":
        df_2050_final["DNI_final"].sum()
        * DT_HOURS / 1000.0,

    "DHI":
        df_2050_final["DHI_final"].sum()
        * DT_HOURS / 1000.0,
}


print("\n=== ENERGÍA ANUAL FINAL ===")

for var in ["GHI", "DNI", "DHI"]:

    row = annual_final_compare.loc[
        annual_final_compare["variable"] == var
    ].iloc[0]

    e = final_annual_energy_2050[var]

    inside = (
        float(row["hist_p05"])
        - 1e-8
        <= e <=
        float(row["hist_p95"])
        + 1e-8
    )

    print(
        f"{var}: {e:.6f} kWh/m² "
        f"| P05={row['hist_p05']:.6f} "
        f"| P95={row['hist_p95']:.6f} "
        f"| inside={inside}"
    )

    assert inside


# ------------------------------------------------------------
# 8. GHI mensual final
# ------------------------------------------------------------

df_2050_final["month"] = (
    df_2050_final["timestamp"].dt.month
)

final_month_energy = (
    df_2050_final
    .groupby("month")
    .agg(
        GHI_final=(
            "GHI_final",
            lambda x: x.sum() * DT_HOURS / 1000.0
        ),
        DNI_final=(
            "DNI_final",
            lambda x: x.sum() * DT_HOURS / 1000.0
        ),
        DHI_final=(
            "DHI_final",
            lambda x: x.sum() * DT_HOURS / 1000.0
        ),
    )
    .reset_index()
)

final_month_check = final_month_energy.merge(
    hist_month_clim[
        [
            "month",
            "GHI_hist_p05",
            "GHI_hist_p95",
        ]
    ],
    on="month",
    validate="one_to_one"
)

TOL_ENERGY = 1e-8

final_month_check["GHI_inside"] = (
    (
        final_month_check["GHI_final"]
        >= final_month_check["GHI_hist_p05"] - TOL_ENERGY
    )
    &
    (
        final_month_check["GHI_final"]
        <= final_month_check["GHI_hist_p95"] + TOL_ENERGY
    )
)

print("\n=== GHI MENSUAL FINAL ===")
print(
    final_month_check
    .round(6)
    .to_string(index=False)
)

assert final_month_check["GHI_inside"].all()


# ------------------------------------------------------------
# 9. Diffuse ratio — invariancia
#
# Como las tres irradiancias se multiplican por el mismo
# factor:
#
#       rd_final = rd_post
#
# tanto puntual como energéticamente.
# ------------------------------------------------------------

final_rd_energy = (
    final_annual_energy_2050["DHI"]
    /
    final_annual_energy_2050["GHI"]
)

print("\n=== DIFFUSE ENERGY RATIO ===")
print("POST :", post_rd_energy_2050)
print("FINAL:", final_rd_energy)

assert np.isclose(
    final_rd_energy,
    post_rd_energy_2050,
    rtol=0.0,
    atol=1e-12
)


# ------------------------------------------------------------
# 10. Verificar rd mensual con tolerancia numérica
#
# Corrige los False artificiales de la Celda 13 cuando el
# resultado coincide exactamente con P05 o P95.
# ------------------------------------------------------------

RD_TOL = 1e-8

monthly_rd_final_tol = monthly_rd_final.copy()

monthly_rd_final_tol["inside_P05_P95_tol"] = (
    (
        monthly_rd_final_tol["rd_post"]
        >= monthly_rd_final_tol["rd_hist_p05"] - RD_TOL
    )
    &
    (
        monthly_rd_final_tol["rd_post"]
        <= monthly_rd_final_tol["rd_hist_p95"] + RD_TOL
    )
)

print("\n=== rd MENSUAL CON TOLERANCIA ===")
print(
    monthly_rd_final_tol[
        [
            "month",
            "rd_post",
            "rd_hist_p05",
            "rd_hist_p95",
            "inside_P05_P95_tol",
        ]
    ]
    .round(8)
    .to_string(index=False)
)

assert monthly_rd_final_tol[
    "inside_P05_P95_tol"
].all()


# ------------------------------------------------------------
# 11. kDNI final
# ------------------------------------------------------------

csdni_final = (
    df_2050_final[COL_CSDNI]
    .to_numpy(dtype=float)
)

dni_final_arr = (
    df_2050_final["DNI_final"]
    .to_numpy(dtype=float)
)

kDNI_final = np.zeros(
    len(df_2050_final),
    dtype=float
)

valid_cs_final = (
    csdni_final > cfg2050.thr_cs
)

kDNI_final[valid_cs_final] = (
    dni_final_arr[valid_cs_final]
    / csdni_final[valid_cs_final]
)

df_2050_final["kDNI_final"] = kDNI_final

print("\n=== kDNI FINAL ===")
print("Máximo:", kDNI_final.max())
print(
    "Registros > 1.20:",
    int((kDNI_final > cfg2050.kdni_max).sum())
)

assert (
    kDNI_final <= cfg2050.kdni_max + 1e-8
).all()


# ------------------------------------------------------------
# 12. Integridad final
# ------------------------------------------------------------

final_irr = df_2050_final[
    ["GHI_final", "DNI_final", "DHI_final"]
].to_numpy(dtype=float)

final_negative_count = int(
    (final_irr < 0.0).sum()
)

final_nan_count = int(
    np.isnan(final_irr).sum()
)

final_inf_count = int(
    np.isinf(final_irr).sum()
)

final_night_leakage = int(
    (
        df_2050_final.loc[
            night_final_mask,
            ["GHI_final", "DNI_final", "DHI_final"]
        ].to_numpy(dtype=float)
        != 0.0
    ).sum()
)

assert final_negative_count == 0
assert final_nan_count == 0
assert final_inf_count == 0
assert final_night_leakage == 0


# ------------------------------------------------------------
# 13. Resumen de aceptación
# ------------------------------------------------------------

print("\n============================================================")
print(" RECONCILIACIÓN ANUAL FINAL")
print("============================================================")

print("Factor global :", GLOBAL_SCALE_FINAL)
print("Closure MAE   :", closure_final2_mae)
print("Closure max   :", closure_final2_max)
print("Night leakage :", final_night_leakage)
print("Negativos     :", final_negative_count)
print("NaN           :", final_nan_count)
print("Inf           :", final_inf_count)

print("\n✓ Restricciones anuales satisfechas.")
print("✓ GHI mensual permanece dentro de P05–P95.")
print("✓ Diffuse ratio permanece inalterado.")
print("✓ rd mensual validado con tolerancia numérica.")
print("✓ kDNI permanece dentro del dominio físico.")
print("✓ Cierre radiativo preservado.")
print("✓ Celda 14 completada correctamente.")

=== RESTRICCIONES PARA EL FACTOR GLOBAL ===
  constraint    lower    upper
  GHI_annual 0.919562 0.998222
  DNI_annual 0.891910 1.134806
  DHI_annual 0.865747 0.965714
GHI_month_01 0.855475 1.042042
GHI_month_02 0.789947 1.001708
GHI_month_03 0.857954 1.000000
GHI_month_04 0.909605 1.036418
GHI_month_05 0.962610 1.085959
GHI_month_06 0.916308 1.064048
GHI_month_07 0.964351 1.086752
GHI_month_08 0.942561 1.112177
GHI_month_09 0.892455 1.013620
GHI_month_10 0.864374 1.000000
GHI_month_11 0.839414 1.000000
GHI_month_12 0.817302 1.000000

=== INTERVALO FACTIBLE GLOBAL ===
Límite inferior: 0.9643514862563343
Límite superior: 0.965714130144508

Factor global seleccionado: 0.965714130144508
Corrección porcentual: -3.428586985549198 %

=== ENERGÍA ANUAL FINAL ===
GHI: 2079.700657 kWh/m² | P05=1980.311350 | P95=2149.707700 | inside=True
DNI: 1636.087440 kWh/m² | P05=1511.049600 | P95=1922.557900 | inside=True
DHI: 902.379900 kWh/m² | P05=808.968950 | P95=902.379900 | inside=True

=== GHI MENSUA

In [16]:
# ============================================================
# 15. CONGELACIÓN DEL RECURSO SINTÉTICO FINAL 2050
#     MANIFEST + SHA-256 + TRAZABILIDAD
# ============================================================
#
# OBJETIVO
# --------
# Congelar formalmente el producto aceptado antes de realizar
# la exportación específica para SAM.
#
# Se conservarán inequívocamente tres niveles:
#
#   RAW
#       salida directa del PI-LSTM V2.4 V1;
#
#   POST_CANDIDATE
#       postproceso climatológico mensual de mínima
#       intervención;
#
#   FINAL
#       POST_CANDIDATE + reconciliación global restringida.
#
# Esta celda NO modifica nuevamente GHI, DNI ni DHI.
# ============================================================

import hashlib
from pathlib import Path


# ------------------------------------------------------------
# 1. Construir tabla FINAL canónica
#
# Se conservan las variables necesarias para trazabilidad y
# posterior exportación SAM.
# ------------------------------------------------------------

FINAL_RESOURCE_COLS = [
    "timestamp",

    "Solar Zenith Angle",
    "cosz_plus",

    "Clearsky GHI",
    "Clearsky DNI",
    "Clearsky DHI",

    "GHI_raw",
    "DNI_raw",
    "DHI_raw",

    "GHI_post",
    "DNI_post",
    "DHI_post",

    "GHI_final",
    "DNI_final",
    "DHI_final",
]

missing_final_cols = [
    c for c in FINAL_RESOURCE_COLS
    if c not in df_2050_final.columns
]

assert not missing_final_cols, (
    f"Faltan columnas para congelar FINAL: "
    f"{missing_final_cols}"
)

final_resource_2050 = (
    df_2050_final[FINAL_RESOURCE_COLS]
    .copy()
)

assert len(final_resource_2050) == 17520

assert final_resource_2050["timestamp"].iloc[0] == pd.Timestamp(
    "2050-01-01 00:00:00"
)

assert final_resource_2050["timestamp"].iloc[-1] == pd.Timestamp(
    "2050-12-31 23:30:00"
)


# ------------------------------------------------------------
# 2. Integridad temporal final
# ------------------------------------------------------------

assert (
    final_resource_2050["timestamp"]
    .duplicated()
    .sum()
    == 0
)

dt_final = (
    final_resource_2050["timestamp"]
    .diff()
    .dropna()
    .dt.total_seconds()
    / 60.0
)

assert (dt_final == 30.0).all()


# ------------------------------------------------------------
# 3. Integridad física FINAL
# ------------------------------------------------------------

final_numeric = (
    final_resource_2050[
        ["GHI_final", "DNI_final", "DHI_final"]
    ]
    .to_numpy(dtype=float)
)

assert np.isfinite(final_numeric).all()
assert (final_numeric >= 0.0).all()

final_closure = (
    final_resource_2050["GHI_final"]
    .to_numpy(dtype=float)
    -
    (
        final_resource_2050["DNI_final"]
        .to_numpy(dtype=float)
        *
        final_resource_2050["cosz_plus"]
        .to_numpy(dtype=float)
        +
        final_resource_2050["DHI_final"]
        .to_numpy(dtype=float)
    )
)

FINAL_CLOSURE_MAE = float(
    np.mean(np.abs(final_closure))
)

FINAL_CLOSURE_MAX = float(
    np.max(np.abs(final_closure))
)

assert FINAL_CLOSURE_MAX < 1e-6


# ------------------------------------------------------------
# 4. Energía FINAL congelada
# ------------------------------------------------------------

FINAL_ENERGY = {
    "GHI_kWh_m2":
        float(
            final_resource_2050["GHI_final"].sum()
            * DT_HOURS
            / 1000.0
        ),

    "DNI_kWh_m2":
        float(
            final_resource_2050["DNI_final"].sum()
            * DT_HOURS
            / 1000.0
        ),

    "DHI_kWh_m2":
        float(
            final_resource_2050["DHI_final"].sum()
            * DT_HOURS
            / 1000.0
        ),
}

FINAL_RD_ENERGY = (
    FINAL_ENERGY["DHI_kWh_m2"]
    /
    FINAL_ENERGY["GHI_kWh_m2"]
)


# ------------------------------------------------------------
# 5. Archivo canónico FINAL
# ------------------------------------------------------------

FINAL_RESOURCE_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "PI_LSTM_V24V1_2050_FINAL_RESOURCE.csv"
)

final_resource_2050.to_csv(
    FINAL_RESOURCE_PATH,
    index=False,
    float_format="%.10f"
)


# ------------------------------------------------------------
# 6. Función SHA-256
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo para SHA-256:\n{path}"
        )

    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


FINAL_RESOURCE_SHA256 = sha256_file(
    FINAL_RESOURCE_PATH
)

RAW_RESOURCE_SHA256 = sha256_file(
    RAW_2050_PATH
)

POST_RESOURCE_SHA256 = sha256_file(
    POST_CANDIDATE_PATH
)


# ------------------------------------------------------------
# 7. Artefactos fuente críticos
#
# Se fijan explícitamente las rutas para evitar dependencia
# de aliases empleados en celdas anteriores.
# ------------------------------------------------------------

MODEL_FILE_FINAL = Path(
    "/content/PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras"
)

PHYSICAL_FILE_FINAL = Path(
    "/content/physical_2050_base.csv"
)

HISTORICAL_FILE_FINAL = Path(
    "/content/consol_tumbabiro_2026_total.csv"
)

SOURCE_ARTIFACTS = {
    "Modelo V2.4 V1": MODEL_FILE_FINAL,
    "Base física 2050": PHYSICAL_FILE_FINAL,
    "Histórico NSRDB": HISTORICAL_FILE_FINAL,
}

for label, path in SOURCE_ARTIFACTS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{label}: no se encontró el archivo:\n{path}"
        )

print("=== ARTEFACTOS FUENTE ===")
print("Modelo     :", MODEL_FILE_FINAL.name)
print("Base física:", PHYSICAL_FILE_FINAL.name)
print("Histórico  :", HISTORICAL_FILE_FINAL.name)

MODEL_SHA256 = sha256_file(
    MODEL_FILE_FINAL
)

PHYSICAL_BASE_SHA256_CHECK = sha256_file(
    PHYSICAL_FILE_FINAL
)

HISTORICAL_SHA256 = sha256_file(
    HISTORICAL_FILE_FINAL
)


# ------------------------------------------------------------
# 8. Datos canónicos del sitio
#
# Se utiliza la configuración cuando está disponible.
# Para elevación se admite fallback explícito a 2050 m.
# ------------------------------------------------------------

SITE_LATITUDE_FINAL = float(
    cfg2050.latitude
)

SITE_LONGITUDE_FINAL = float(
    cfg2050.longitude
)

SITE_ELEVATION_M_FINAL = float(
    getattr(
        cfg2050,
        "elevation_m",
        2050.0
    )
)

SITE_TIMEZONE_FINAL = str(
    cfg2050.timezone
)


# ------------------------------------------------------------
# 9. Decisión metodológica final
# ------------------------------------------------------------

FINAL_STATUS = (
    "ACCEPTED_SYNTHETIC_2050_RESOURCE"
)

FINAL_DECISION = {
    "status":
        FINAL_STATUS,

    "interpretation":
        (
            "Synthetic 2050 solar resource year conditioned "
            "on the historical NSRDB climatological regime "
            "and 2050 solar geometry; not a CMIP6/SSP "
            "climate projection."
        ),

    "generator":
        "PI-LSTM V2.4 V1 RAW",

    "model_post_calibration":
        "NONE",

    "isotonic_calibration":
        "REJECTED",

    "climatological_postprocess":
        (
            "Monthly minimum-intervention constraint using "
            "historical P05-P95 bounds for GHI energy and "
            "diffuse energy ratio."
        ),

    "global_reconciliation":
        (
            "Minimum global scalar satisfying annual "
            "GHI/DNI/DHI and monthly GHI climatological "
            "constraints."
        ),

    "global_scale":
        float(
            GLOBAL_SCALE_FINAL
        ),

    "global_scale_change_pct":
        float(
            100.0
            * (GLOBAL_SCALE_FINAL - 1.0)
        ),

    "rows":
        int(
            len(final_resource_2050)
        ),

    "frequency_minutes":
        30,

    "start":
        str(
            final_resource_2050["timestamp"].min()
        ),

    "end":
        str(
            final_resource_2050["timestamp"].max()
        ),

    "annual_energy":
        FINAL_ENERGY,

    "annual_diffuse_energy_ratio":
        float(
            FINAL_RD_ENERGY
        ),

    "closure_mae_Wm2":
        FINAL_CLOSURE_MAE,

    "closure_max_abs_Wm2":
        FINAL_CLOSURE_MAX,

    "night_leakage_count":
        0,

    "negative_count":
        0,

    "nan_count":
        0,

    "inf_count":
        0,

    "kDNI_final_max":
        float(
            kDNI_final.max()
        ),

    "kDNI_cap":
        float(
            cfg2050.kdni_max
        ),
}


# ------------------------------------------------------------
# 10. Manifest de artefactos
# ------------------------------------------------------------

FINAL_MANIFEST = {
    "project":
        "Tumbabiro synthetic solar resource 2050",

    "site": {
        "latitude":
            SITE_LATITUDE_FINAL,

        "longitude":
            SITE_LONGITUDE_FINAL,

        "elevation_m":
            SITE_ELEVATION_M_FINAL,

        "timezone":
            SITE_TIMEZONE_FINAL,
    },

    "historical_period":
        "1998-2024",

    "synthetic_year":
        2050,

    "model": {
        "name":
            "PI-LSTM V2.4 V1",

        "L":
            int(
                cfg2050.L
            ),

        "H":
            int(
                cfg2050.H
            ),

        "historical_features":
            int(
                len(hist_cols_meta_2050)
            ),

        "future_features":
            int(
                len(future_cols_meta_2050)
            ),

        "kDNI_max":
            float(
                cfg2050.kdni_max
            ),

        "kDHI_max":
            float(
                cfg2050.kdhi_max
            ),

        "qDHI_max":
            float(
                cfg2050.qdhi_max
            ),
    },

    "files": {
        "historical_NSRDB": {
            "name":
                HISTORICAL_FILE_FINAL.name,

            "sha256":
                HISTORICAL_SHA256,
        },

        "physical_2050_base": {
            "name":
                PHYSICAL_FILE_FINAL.name,

            "sha256":
                PHYSICAL_BASE_SHA256_CHECK,
        },

        "model": {
            "name":
                MODEL_FILE_FINAL.name,

            "sha256":
                MODEL_SHA256,
        },

        "raw_2050": {
            "name":
                RAW_2050_PATH.name,

            "sha256":
                RAW_RESOURCE_SHA256,
        },

        "post_candidate": {
            "name":
                POST_CANDIDATE_PATH.name,

            "sha256":
                POST_RESOURCE_SHA256,
        },

        "final_resource": {
            "name":
                FINAL_RESOURCE_PATH.name,

            "sha256":
                FINAL_RESOURCE_SHA256,
        },
    },

    "decision":
        FINAL_DECISION,
}


# ------------------------------------------------------------
# 11. Guardar decisión y manifest
# ------------------------------------------------------------

FINAL_DECISION_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "FINAL_RESOURCE_DECISION_V24V1.json"
)

FINAL_MANIFEST_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "FINAL_RESOURCE_MANIFEST_V24V1.json"
)

with open(
    FINAL_DECISION_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_DECISION,
        f,
        indent=2
    )

with open(
    FINAL_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FINAL_MANIFEST,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 12. Tabla compacta de trazabilidad
# ------------------------------------------------------------

traceability_2050 = pd.DataFrame([
    {
        "stage":
            "RAW",

        "file":
            RAW_2050_PATH.name,

        "sha256":
            RAW_RESOURCE_SHA256,
    },

    {
        "stage":
            "POST_CANDIDATE",

        "file":
            POST_CANDIDATE_PATH.name,

        "sha256":
            POST_RESOURCE_SHA256,
    },

    {
        "stage":
            "FINAL",

        "file":
            FINAL_RESOURCE_PATH.name,

        "sha256":
            FINAL_RESOURCE_SHA256,
    },
])

TRACEABILITY_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "resource_traceability_v24_v1.csv"
)

traceability_2050.to_csv(
    TRACEABILITY_PATH,
    index=False
)


# ------------------------------------------------------------
# 13. Auditoría final del manifest
# ------------------------------------------------------------

assert FINAL_RESOURCE_PATH.exists()
assert FINAL_DECISION_PATH.exists()
assert FINAL_MANIFEST_PATH.exists()
assert TRACEABILITY_PATH.exists()

assert (
    FINAL_MANIFEST[
        "files"
    ][
        "final_resource"
    ][
        "sha256"
    ]
    ==
    FINAL_RESOURCE_SHA256
)

assert (
    FINAL_DECISION["status"]
    ==
    "ACCEPTED_SYNTHETIC_2050_RESOURCE"
)


# ------------------------------------------------------------
# 14. Resumen final
# ------------------------------------------------------------

print("\n============================================================")
print(" RECURSO SINTÉTICO 2050 — CONGELADO")
print("============================================================")

print("\nEstado:")
print(FINAL_STATUS)

print("\n=== SITIO ===")
print(
    f"Latitud   : {SITE_LATITUDE_FINAL}"
)
print(
    f"Longitud  : {SITE_LONGITUDE_FINAL}"
)
print(
    f"Elevación : {SITE_ELEVATION_M_FINAL} m"
)
print(
    f"Timezone  : {SITE_TIMEZONE_FINAL}"
)

print("\nArchivo FINAL:")
print(
    FINAL_RESOURCE_PATH.name
)

print("\nSHA-256 FINAL:")
print(
    FINAL_RESOURCE_SHA256
)

print("\n=== ENERGÍA FINAL ===")
print(
    f"GHI = "
    f"{FINAL_ENERGY['GHI_kWh_m2']:.6f} "
    f"kWh/m²"
)

print(
    f"DNI = "
    f"{FINAL_ENERGY['DNI_kWh_m2']:.6f} "
    f"kWh/m²"
)

print(
    f"DHI = "
    f"{FINAL_ENERGY['DHI_kWh_m2']:.6f} "
    f"kWh/m²"
)

print(
    f"rd,E = "
    f"{FINAL_RD_ENERGY:.9f}"
)

print("\n=== RECONCILIACIÓN ===")
print(
    f"Factor global = "
    f"{GLOBAL_SCALE_FINAL:.12f}"
)

print(
    f"Cambio        = "
    f"{100.0 * (GLOBAL_SCALE_FINAL - 1.0):+.6f}%"
)

print("\n=== INTEGRIDAD ===")
print(
    f"Closure MAE = "
    f"{FINAL_CLOSURE_MAE:.3e} W/m²"
)

print(
    f"Closure max = "
    f"{FINAL_CLOSURE_MAX:.3e} W/m²"
)

print(
    f"kDNI max    = "
    f"{kDNI_final.max():.9f}"
)

print("\n=== SHA-256 ARTEFACTOS FUENTE ===")
print(
    "Modelo     :",
    MODEL_SHA256
)

print(
    "Base física:",
    PHYSICAL_BASE_SHA256_CHECK
)

print(
    "Histórico  :",
    HISTORICAL_SHA256
)

print("\n=== ARCHIVOS DE CONTROL ===")
print(
    FINAL_DECISION_PATH.name
)

print(
    FINAL_MANIFEST_PATH.name
)

print(
    TRACEABILITY_PATH.name
)

print("\n✓ RAW preservado.")
print("✓ POST_CANDIDATE preservado.")
print("✓ FINAL congelado.")
print("✓ SHA-256 calculados.")
print("✓ Manifest de reproducibilidad generado.")
print("✓ Trazabilidad RAW → POST → FINAL registrada.")
print("✓ Recurso listo para etapa de exportación SAM.")
print("✓ Celda 15 completada correctamente.")

=== ARTEFACTOS FUENTE ===
Modelo     : PI_LSTM_V2_4_V1_logDHI_partition_stride2.keras
Base física: physical_2050_base.csv
Histórico  : consol_tumbabiro_2026_total.csv

 RECURSO SINTÉTICO 2050 — CONGELADO

Estado:
ACCEPTED_SYNTHETIC_2050_RESOURCE

=== SITIO ===
Latitud   : 0.45
Longitud  : -78.18
Elevación : 2050.0 m
Timezone  : Etc/GMT+5

Archivo FINAL:
PI_LSTM_V24V1_2050_FINAL_RESOURCE.csv

SHA-256 FINAL:
6a2d688bd510f82279af8ff9f89beedfd5d8de36e39cff8c237a7f64fd055872

=== ENERGÍA FINAL ===
GHI = 2079.700657 kWh/m²
DNI = 1636.087440 kWh/m²
DHI = 902.379900 kWh/m²
rd,E = 0.433898935

=== RECONCILIACIÓN ===
Factor global = 0.965714130145
Cambio        = -3.428587%

=== INTEGRIDAD ===
Closure MAE = 1.222e-14 W/m²
Closure max = 2.274e-13 W/m²
kDNI max    = 0.900345470

=== SHA-256 ARTEFACTOS FUENTE ===
Modelo     : 5aef4783a3dc0b37b98711ebb489d20458f6c5efd2308ca579909e4cc8724a4e
Base física: 230cf4a767e52917b803f4c1a45add56c48a89aa88962fe3962188e265b9fd1f
Histórico  : 6221292bd639401d7cb

## Interpretación para el artículo

Este notebook mantiene deliberadamente separados los distintos niveles del pipeline:

1. **salida PI-LSTM RAW**: corresponde a la salida directa del modelo V2.4 V1, antes de cualquier ajuste climatológico posterior;

2. **reconstrucción física**: las salidas normalizadas del modelo se transforman a DNI y DHI, y GHI se obtiene imponiendo la relación física

$$GHI = DNI\cos(z) + DHI;$$

3. **POST_CANDIDATE**: corresponde al recurso sometido a un postproceso climatológico mensual de mínima intervención, aplicado únicamente cuando la energía mensual de GHI o la fracción difusa energética quedan fuera del intervalo histórico P05–P95;

4. **FINAL RESOURCE**: corresponde al recurso sintético aceptado después de la reconciliación global restringida, manteniendo la consistencia radiativa, la continuidad temporal y la plausibilidad climatológica.

La diferencia entre la salida **RAW** y el **FINAL RESOURCE** debe formar parte explícita de la discusión metodológica. En particular, debe analizarse la partición directa–difusa mediante

$$r_d = \frac{DHI}{GHI},$$

distinguiendo entre el comportamiento predictivo del PI-LSTM y las restricciones climatológicas utilizadas para construir el recurso sintético final.

Las métricas predictivas del modelo corresponden exclusivamente a la evaluación sobre el período de prueba 2023–2024 y a la salida RAW. Las propiedades climatológicas del recurso FINAL no deben presentarse como métricas de desempeño predictivo de la red.

El recurso 2050 se interpreta como un **año sintético condicionado al régimen climatológico histórico de NSRDB y a la geometría solar del calendario 2050**. No constituye una proyección climática futura ni representa un escenario CMIP6/SSP.

In [17]:
# ============================================================
# 16. EXPORTACIÓN CSV COMPATIBLE CON NREL SAM
#     RECURSO SINTÉTICO 2050 CONGELADO
# ============================================================
#
# OBJETIVO
# --------
# Exportar PI_LSTM_V24V1_2050_FINAL_RESOURCE.csv a un archivo
# meteorológico CSV directamente utilizable en NREL SAM.
#
# IMPORTANTE
# ----------
# Esta celda:
#
#   - NO modifica GHI_final;
#   - NO modifica DNI_final;
#   - NO modifica DHI_final;
#   - NO realiza calibración;
#   - NO realiza reconciliación adicional.
#
# Únicamente:
#
#   1. incorpora la meteorología exógena climatológica 2050;
#   2. construye Year/Month/Day/Hour/Minute;
#   3. genera el encabezado SAM;
#   4. valida el archivo exportado;
#   5. calcula SHA-256.
#
# Formato:
#
# línea 1 -> nombres de metadatos
# línea 2 -> valores de metadatos
# línea 3 -> nombres de variables
# línea 4+ -> 17,520 registros a 30 min
# ============================================================

import csv
import hashlib
from pathlib import Path


# ------------------------------------------------------------
# 1. Archivo de salida
# ------------------------------------------------------------

SAM_OUTPUT_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "Tumbabiro_PI_LSTM_V24V1_2050_SAM.csv"
)


# ------------------------------------------------------------
# 2. Meteorología que acompañará el recurso final
#
# Estas variables provienen de la climatología histórica
# construida previamente para el calendario 2050.
# ------------------------------------------------------------

SAM_METEO_COLS = [
    "Temperature",
    "Dew Point",
    "Relative Humidity",
    "Pressure",
    "Wind Speed",
    "Wind Direction",
]

missing_sam_meteo = [
    c for c in SAM_METEO_COLS
    if c not in df_2050_base.columns
]

assert not missing_sam_meteo, (
    f"Faltan variables meteorológicas para SAM: "
    f"{missing_sam_meteo}"
)


# ------------------------------------------------------------
# 3. Preparar irradiancia FINAL congelada
# ------------------------------------------------------------

sam_resource = final_resource_2050[
    [
        "timestamp",
        "GHI_final",
        "DNI_final",
        "DHI_final",
    ]
].copy()

sam_resource = sam_resource.merge(
    df_2050_base[
        ["timestamp"] + SAM_METEO_COLS
    ],
    on="timestamp",
    how="left",
    validate="one_to_one"
)

assert len(sam_resource) == 17520
assert not sam_resource.isna().any().any()


# ------------------------------------------------------------
# 4. Crear columnas temporales SAM
#
# La zona horaria es tiempo local estándar UTC-5.
# No se aplica horario de verano.
# ------------------------------------------------------------

sam_resource["Year"] = (
    sam_resource["timestamp"].dt.year.astype(int)
)

sam_resource["Month"] = (
    sam_resource["timestamp"].dt.month.astype(int)
)

sam_resource["Day"] = (
    sam_resource["timestamp"].dt.day.astype(int)
)

sam_resource["Hour"] = (
    sam_resource["timestamp"].dt.hour.astype(int)
)

sam_resource["Minute"] = (
    sam_resource["timestamp"].dt.minute.astype(int)
)


# ------------------------------------------------------------
# 5. Renombrar variables según convención SAM CSV
#
# Tdry  = temperatura de bulbo seco [°C]
# Tdew  = punto de rocío [°C]
# RH    = humedad relativa [%]
# Pres  = presión [mbar]
# Wspd  = velocidad del viento [m/s]
# Wdir  = dirección del viento [grados]
# ------------------------------------------------------------

sam_resource["GHI"] = sam_resource["GHI_final"].astype(float)
sam_resource["DNI"] = sam_resource["DNI_final"].astype(float)
sam_resource["DHI"] = sam_resource["DHI_final"].astype(float)

sam_resource["Tdry"] = (
    sam_resource["Temperature"].astype(float)
)

sam_resource["Tdew"] = (
    sam_resource["Dew Point"].astype(float)
)

sam_resource["RH"] = (
    sam_resource["Relative Humidity"].astype(float)
)

sam_resource["Pres"] = (
    sam_resource["Pressure"].astype(float)
)

sam_resource["Wspd"] = (
    sam_resource["Wind Speed"].astype(float)
)

sam_resource["Wdir"] = (
    sam_resource["Wind Direction"].astype(float)
)


# ------------------------------------------------------------
# 6. Orden final de columnas
# ------------------------------------------------------------

SAM_DATA_COLS = [
    "Year",
    "Month",
    "Day",
    "Hour",
    "Minute",
    "GHI",
    "DNI",
    "DHI",
    "Tdry",
    "Tdew",
    "RH",
    "Pres",
    "Wspd",
    "Wdir",
]

sam_data = sam_resource[SAM_DATA_COLS].copy()

assert sam_data.shape == (17520, 14)


# ------------------------------------------------------------
# 7. Metadatos SAM
#
# IMPORTANTE:
# Source NO se denomina NSRDB porque el recurso radiativo
# final es sintético.
#
# Time Zone = -5 corresponde a Ecuador continental.
# ------------------------------------------------------------

SAM_METADATA_HEADER = [
    "Source",
    "Location ID",
    "City",
    "State",
    "Country",
    "Latitude",
    "Longitude",
    "Time Zone",
    "Elevation",
]

SAM_METADATA_VALUES = [
    "PI-LSTM V2.4 V1 Synthetic",
    "TUMBABIRO_2050_V24V1",
    "Tumbabiro",
    "Imbabura",
    "Ecuador",
    f"{SITE_LATITUDE_FINAL:.6f}",
    f"{SITE_LONGITUDE_FINAL:.6f}",
    "-5",
    f"{SITE_ELEVATION_M_FINAL:.1f}",
]

assert len(SAM_METADATA_HEADER) == len(SAM_METADATA_VALUES)


# ------------------------------------------------------------
# 8. Auditoría temporal previa a exportación
# ------------------------------------------------------------

assert sam_data["Year"].nunique() == 1
assert sam_data["Year"].iloc[0] == 2050

assert sam_data.iloc[0][
    ["Month", "Day", "Hour", "Minute"]
].tolist() == [1, 1, 0, 0]

assert sam_data.iloc[-1][
    ["Month", "Day", "Hour", "Minute"]
].tolist() == [12, 31, 23, 30]

assert set(sam_data["Minute"].unique()) == {0, 30}


# ------------------------------------------------------------
# 9. Auditoría de irradiancia
#
# Verificar que los valores que se exportarán son EXACTAMENTE
# los valores congelados.
# ------------------------------------------------------------

np.testing.assert_allclose(
    sam_data["GHI"].to_numpy(dtype=float),
    final_resource_2050["GHI_final"].to_numpy(dtype=float),
    rtol=0.0,
    atol=0.0
)

np.testing.assert_allclose(
    sam_data["DNI"].to_numpy(dtype=float),
    final_resource_2050["DNI_final"].to_numpy(dtype=float),
    rtol=0.0,
    atol=0.0
)

np.testing.assert_allclose(
    sam_data["DHI"].to_numpy(dtype=float),
    final_resource_2050["DHI_final"].to_numpy(dtype=float),
    rtol=0.0,
    atol=0.0
)

print("Irradiancia FINAL preservada exactamente: OK")


# ------------------------------------------------------------
# 10. Integridad numérica
# ------------------------------------------------------------

sam_numeric_cols = [
    "GHI", "DNI", "DHI",
    "Tdry", "Tdew", "RH",
    "Pres", "Wspd", "Wdir",
]

sam_numeric = sam_data[
    sam_numeric_cols
].to_numpy(dtype=float)

assert np.isfinite(sam_numeric).all()

assert (
    sam_data[
        ["GHI", "DNI", "DHI", "Wspd"]
    ] >= 0.0
).all().all()


# ------------------------------------------------------------
# 11. Cierre radiativo antes de escribir
# ------------------------------------------------------------

sam_check = sam_resource.merge(
    df_2050_base[
        ["timestamp", "cosz_plus"]
    ],
    on="timestamp",
    validate="one_to_one"
)

sam_closure = (
    sam_check["GHI"].to_numpy(dtype=float)
    -
    (
        sam_check["DNI"].to_numpy(dtype=float)
        *
        sam_check["cosz_plus"].to_numpy(dtype=float)
        +
        sam_check["DHI"].to_numpy(dtype=float)
    )
)

SAM_CLOSURE_MAE = float(
    np.mean(np.abs(sam_closure))
)

SAM_CLOSURE_MAX = float(
    np.max(np.abs(sam_closure))
)

assert SAM_CLOSURE_MAX < 1e-6


# ------------------------------------------------------------
# 12. Escribir archivo SAM
#
# newline="" permite que csv controle correctamente los
# finales de línea.
#
# Se usa lineterminator="\n", aceptado por SAM.
# ------------------------------------------------------------

with open(
    SAM_OUTPUT_PATH,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f,
        lineterminator="\n"
    )

    # Línea 1: nombres de metadatos
    writer.writerow(
        SAM_METADATA_HEADER
    )

    # Línea 2: valores de metadatos
    writer.writerow(
        SAM_METADATA_VALUES
    )

    # Línea 3: encabezados de datos
    writer.writerow(
        SAM_DATA_COLS
    )

    # Líneas siguientes: datos
    for row in sam_data.itertuples(
        index=False,
        name=None
    ):

        formatted_row = [
            int(row[0]),       # Year
            int(row[1]),       # Month
            int(row[2]),       # Day
            int(row[3]),       # Hour
            int(row[4]),       # Minute

            f"{row[5]:.6f}",   # GHI
            f"{row[6]:.6f}",   # DNI
            f"{row[7]:.6f}",   # DHI

            f"{row[8]:.3f}",   # Tdry
            f"{row[9]:.3f}",   # Tdew
            f"{row[10]:.3f}",  # RH
            f"{row[11]:.3f}",  # Pres
            f"{row[12]:.3f}",  # Wspd
            f"{row[13]:.3f}",  # Wdir
        ]

        writer.writerow(
            formatted_row
        )


assert SAM_OUTPUT_PATH.exists()


# ------------------------------------------------------------
# 13. Leer nuevamente el archivo para validarlo
#
# Las dos primeras líneas son metadatos.
# La tercera línea contiene los encabezados.
# ------------------------------------------------------------

sam_reload = pd.read_csv(
    SAM_OUTPUT_PATH,
    skiprows=2
)

assert len(sam_reload) == 17520

assert list(sam_reload.columns) == SAM_DATA_COLS

assert not sam_reload.isna().any().any()


# ------------------------------------------------------------
# 14. Comprobar temporalidad después de escritura
# ------------------------------------------------------------

sam_reload_timestamp = pd.to_datetime(
    dict(
        year=sam_reload["Year"],
        month=sam_reload["Month"],
        day=sam_reload["Day"],
        hour=sam_reload["Hour"],
        minute=sam_reload["Minute"],
    )
)

assert sam_reload_timestamp.iloc[0] == pd.Timestamp(
    "2050-01-01 00:00:00"
)

assert sam_reload_timestamp.iloc[-1] == pd.Timestamp(
    "2050-12-31 23:30:00"
)

sam_reload_dt = (
    sam_reload_timestamp
    .diff()
    .dropna()
    .dt.total_seconds()
    / 60.0
)

assert (sam_reload_dt == 30.0).all()


# ------------------------------------------------------------
# 15. Error introducido exclusivamente por escritura decimal
#
# El archivo usa 6 decimales para irradiancia.
# Comprobamos que la diferencia respecto del recurso congelado
# sea únicamente la esperada por redondeo de texto.
# ------------------------------------------------------------

SAM_ROUNDING_ERRORS = {}

for var, final_col in [
    ("GHI", "GHI_final"),
    ("DNI", "DNI_final"),
    ("DHI", "DHI_final"),
]:

    diff = (
        sam_reload[var].to_numpy(dtype=float)
        -
        final_resource_2050[
            final_col
        ].to_numpy(dtype=float)
    )

    SAM_ROUNDING_ERRORS[var] = {
        "mae_Wm2":
            float(
                np.mean(np.abs(diff))
            ),

        "max_abs_Wm2":
            float(
                np.max(np.abs(diff))
            ),
    }


# ------------------------------------------------------------
# 16. Energía del archivo SAM recargado
# ------------------------------------------------------------

SAM_ENERGY = {
    "GHI_kWh_m2":
        float(
            sam_reload["GHI"].sum()
            * DT_HOURS
            / 1000.0
        ),

    "DNI_kWh_m2":
        float(
            sam_reload["DNI"].sum()
            * DT_HOURS
            / 1000.0
        ),

    "DHI_kWh_m2":
        float(
            sam_reload["DHI"].sum()
            * DT_HOURS
            / 1000.0
        ),
}

SAM_RD_ENERGY = (
    SAM_ENERGY["DHI_kWh_m2"]
    /
    SAM_ENERGY["GHI_kWh_m2"]
)


# ------------------------------------------------------------
# 17. SHA-256 del archivo SAM
# ------------------------------------------------------------

SAM_SHA256 = sha256_file(
    SAM_OUTPUT_PATH
)


# ------------------------------------------------------------
# 18. Manifest específico de exportación
# ------------------------------------------------------------

SAM_EXPORT_MANIFEST = {
    "status":
        "SAM_EXPORT_READY",

    "format":
        "SAM CSV",

    "synthetic_year":
        2050,

    "resolution_minutes":
        30,

    "records":
        int(
            len(sam_reload)
        ),

    "timezone":
        -5,

    "source_resource": {
        "file":
            FINAL_RESOURCE_PATH.name,

        "sha256":
            FINAL_RESOURCE_SHA256,
    },

    "sam_file": {
        "file":
            SAM_OUTPUT_PATH.name,

        "sha256":
            SAM_SHA256,
    },

    "site": {
        "city":
            "Tumbabiro",

        "state":
            "Imbabura",

        "country":
            "Ecuador",

        "latitude":
            SITE_LATITUDE_FINAL,

        "longitude":
            SITE_LONGITUDE_FINAL,

        "elevation_m":
            SITE_ELEVATION_M_FINAL,
    },

    "irradiance_energy_kWh_m2":
        SAM_ENERGY,

    "diffuse_energy_ratio":
        SAM_RD_ENERGY,

    "closure_mae_Wm2":
        SAM_CLOSURE_MAE,

    "closure_max_abs_Wm2":
        SAM_CLOSURE_MAX,

    "rounding_error_Wm2":
        SAM_ROUNDING_ERRORS,

    "radiative_values_modified":
        False,

    "note":
        (
            "Format-only export from the frozen synthetic "
            "2050 final resource. Irradiance values are "
            "unchanged except for decimal text serialization."
        ),
}


SAM_MANIFEST_PATH = (
    OUTPUT_DIR_2050_V24V1 /
    "SAM_EXPORT_MANIFEST_V24V1.json"
)

with open(
    SAM_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SAM_EXPORT_MANIFEST,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 19. Mostrar primeras líneas reales del archivo
# ------------------------------------------------------------

print("\n============================================================")
print(" CSV COMPATIBLE CON NREL SAM — GENERADO")
print("============================================================")

print("\nPrimeras 6 líneas del archivo:\n")

with open(
    SAM_OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    for _ in range(6):
        print(
            f.readline().rstrip()
        )


# ------------------------------------------------------------
# 20. Resumen final
# ------------------------------------------------------------

print("\n=== ARCHIVO SAM ===")
print(
    SAM_OUTPUT_PATH.name
)

print("\nSHA-256:")
print(
    SAM_SHA256
)

print("\n=== DIMENSIONES ===")
print(
    "Registros:",
    len(sam_reload)
)
print(
    "Resolución:",
    "30 min"
)
print(
    "Inicio:",
    sam_reload_timestamp.iloc[0]
)
print(
    "Fin:",
    sam_reload_timestamp.iloc[-1]
)

print("\n=== ENERGÍA VERIFICADA DESDE CSV SAM ===")
print(
    f"GHI = "
    f"{SAM_ENERGY['GHI_kWh_m2']:.6f} kWh/m²"
)
print(
    f"DNI = "
    f"{SAM_ENERGY['DNI_kWh_m2']:.6f} kWh/m²"
)
print(
    f"DHI = "
    f"{SAM_ENERGY['DHI_kWh_m2']:.6f} kWh/m²"
)
print(
    f"rd,E = "
    f"{SAM_RD_ENERGY:.9f}"
)

print("\n=== ERROR MÁXIMO DE SERIALIZACIÓN ===")
for var in ["GHI", "DNI", "DHI"]:
    print(
        f"{var}: "
        f"{SAM_ROUNDING_ERRORS[var]['max_abs_Wm2']:.9e} "
        f"W/m²"
    )

print("\n=== INTEGRIDAD ===")
print(
    f"Closure MAE = {SAM_CLOSURE_MAE:.3e} W/m²"
)
print(
    f"Closure max = {SAM_CLOSURE_MAX:.3e} W/m²"
)

print("\nManifest:")
print(
    SAM_MANIFEST_PATH.name
)

print("\n✓ Recurso FINAL preservado.")
print("✓ Meteorología climatológica 2050 incorporada.")
print("✓ Formato temporal SAM construido.")
print("✓ 17,520 registros a 30 minutos.")
print("✓ Archivo releído y validado.")
print("✓ Energía verificada después de serialización.")
print("✓ SHA-256 de exportación calculado.")
print("✓ Exportación CSV para SAM completada.")
print("✓ Celda 16 completada correctamente.")

Irradiancia FINAL preservada exactamente: OK

 CSV COMPATIBLE CON NREL SAM — GENERADO

Primeras 6 líneas del archivo:

Source,Location ID,City,State,Country,Latitude,Longitude,Time Zone,Elevation
PI-LSTM V2.4 V1 Synthetic,TUMBABIRO_2050_V24V1,Tumbabiro,Imbabura,Ecuador,0.450000,-78.180000,-5,2050.0
Year,Month,Day,Hour,Minute,GHI,DNI,DHI,Tdry,Tdew,RH,Pres,Wspd,Wdir
2050,1,1,0,0,0.000000,0.000000,0.000000,14.600,11.900,83.630,803.000,0.700,119.809
2050,1,1,0,30,0.000000,0.000000,0.000000,14.400,11.900,84.620,803.000,0.700,121.294
2050,1,1,1,0,0.000000,0.000000,0.000000,14.300,11.700,83.930,802.000,0.600,116.634

=== ARCHIVO SAM ===
Tumbabiro_PI_LSTM_V24V1_2050_SAM.csv

SHA-256:
a54dd7f659330495a4f1ca8949dd730dd1030ad2a0b05738d8743253ac0ca693

=== DIMENSIONES ===
Registros: 17520
Resolución: 30 min
Inicio: 2050-01-01 00:00:00
Fin: 2050-12-31 23:30:00

=== ENERGÍA VERIFICADA DESDE CSV SAM ===
GHI = 2079.700657 kWh/m²
DNI = 1636.087440 kWh/m²
DHI = 902.379900 kWh/m²
rd,E = 0.433898935

=== 

In [18]:
import shutil
from google.colab import files

folder_path = "/content/model_v24_v1_2050_outputs"

zip_base = "/content/Tumbabiro_V24V1_SAM_Exportado_Rectificado"

shutil.make_archive(
    zip_base,
    "zip",
    folder_path
)

files.download(
    zip_base + ".zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>